# ResNet-50 ImageNet-1K V2 — Full Fine-Tuning Hyperparameter Tuning

**Standalone Google Colab notebook.** This notebook mounts Google Drive, loads the DPD-Small dataset, and starts each experiment from the **same ImageNet-1K V2 pretrained weights**. No earlier notebook runtime is required.

**Evaluation protocol:** 6,905 development-training images; 695 development-seen and 400 development-unseen validation images; 9,625 crop–disease candidate pairs. **Official test files and labels are never read.**

**Automated selection:** three primary configurations → five-epoch screening → two best configurations continue to epoch 15 → one best continues to epoch 30. Two optional configurations are supplied in subsequent cells. Each Crop and Disease CNN trains independently; checkpoint selection per task uses its individual development harmonic mean; configuration selection uses **development composition harmonic mean**.

**Training display:** per-batch progress bars, per-epoch loss/accuracy metrics, updated plots, resume checkpoints, and phase-specific validation reports. A five-epoch screen is a budget-conscious approximation—not proof of the globally best hyperparameters. The full automated primary workflow may still take more than one night on a Colab T4.

**Recommended execution:** run Cells 1–8 once, then run Primary Screening and Automatic Continuation. Optional experiments and Final Summary are independent subsequent cells. Re-running an interrupted training cell resumes from the last **completed epoch**, not the beginning.

## 1. Mount Google Drive and configure the experiment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, json, random, time, gc, hashlib, subprocess, shutil, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, clear_output

DRIVE_PROJECT = Path('/content/drive/MyDrive/DPD-Small')
LOCAL_PROJECT = Path('/content/DPD-Small')
RESULTS_ROOT = Path('/content/drive/MyDrive/AML_B1_ResNet50_V2_Full_Tuning')
SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 2
SCREEN_EPOCHS = 5
SHORTLIST_EPOCHS = 15
MAX_EPOCHS = 30
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'

assert DEVICE.type == 'cuda', 'Enable GPU: Runtime > Change runtime type > T4 GPU.'
assert DRIVE_PROJECT.is_dir(), f'Dataset not found: {DRIVE_PROJECT}'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', DRIVE_PROJECT)
print('Results:', RESULTS_ROOT)
print('Evaluation: DEVELOPMENT VALIDATION ONLY; no official test access')

## 2. Prepare the dataset (first run copies to Colab local storage)

In [ ]:
LOCAL_PROJECT.mkdir(parents=True, exist_ok=True)

def find_case_insensitive(parent, target):
    matches = [p for p in parent.iterdir() if p.name.lower() == target.lower()]
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected {target} in {parent}; matches={matches}')
    return matches[0]

for name in ('images', 'splits_v2'):
    source = find_case_insensitive(DRIVE_PROJECT, name)
    target = LOCAL_PROJECT / name
    target.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-rt', '--size-only', '--info=progress2',
                    f'{source}/', f'{target}/'], check=True)

print('Local dataset copy ready:', LOCAL_PROJECT)

In [ ]:
train_df = pd.read_csv(LOCAL_PROJECT / 'splits_v2' / 'train_fit.csv')
val_seen_df = pd.read_csv(LOCAL_PROJECT / 'splits_v2' / 'val_seen.csv')
val_unseen_df = pd.read_csv(LOCAL_PROJECT / 'splits_v2' / 'val_unseen.csv')

assert (len(train_df), len(val_seen_df), len(val_unseen_df)) == (6905, 695, 400)
required = {'filename', 'crop_id', 'disease_id'}
for name, df in [('train', train_df), ('seen', val_seen_df), ('unseen', val_unseen_df)]:
    assert required.issubset(df.columns), f'Missing CSV fields in {name}'
    assert df.filename.is_unique, f'Duplicate filenames: {name}'

filename_sets = [set(df.filename) for df in (train_df, val_seen_df, val_unseen_df)]
assert all(not (filename_sets[i] & filename_sets[j])
           for i in range(3) for j in range(i + 1, 3))
def class_pairs(df):
    return set(map(tuple, df[['crop_id','disease_id']].astype(int).to_numpy()))
assert class_pairs(val_seen_df).issubset(class_pairs(train_df))
assert class_pairs(val_unseen_df).isdisjoint(class_pairs(train_df) | class_pairs(val_seen_df))

crop_ids = sorted(set(train_df.crop_id.astype(int)) | set(val_seen_df.crop_id.astype(int)) |
                  set(val_unseen_df.crop_id.astype(int)))
disease_ids = sorted(set(train_df.disease_id.astype(int)) | set(val_seen_df.disease_id.astype(int)) |
                     set(val_unseen_df.disease_id.astype(int)))
assert len(crop_ids) == 55 and len(disease_ids) == 175
assert set(val_unseen_df.crop_id).issubset(set(train_df.crop_id))
assert set(val_unseen_df.disease_id).issubset(set(train_df.disease_id))
crop_index = {label: i for i, label in enumerate(crop_ids)}
disease_index = {label: i for i, label in enumerate(disease_ids)}

image_lookup = {}
for path in tqdm((LOCAL_PROJECT / 'images').rglob('*'), desc='Indexing images'):
    if path.is_file() and path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
        assert path.name not in image_lookup, f'Duplicate image name: {path.name}'
        image_lookup[path.name] = path
missing = set.union(*filename_sets) - image_lookup.keys()
assert not missing, f'Missing image files: {sorted(missing)[:5]}'
print('Train:', len(train_df), 'Development seen:', len(val_seen_df),
      'Development unseen:', len(val_unseen_df))
print('Class mappings:', len(crop_ids), 'crops;', len(disease_ids), 'diseases')
print('Split integrity: PASS')

## 3. Image preprocessing and development DataLoaders

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
validation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class PlantDataset(Dataset):
    def __init__(self, table, transform):
        self.rows = table.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        with Image.open(image_lookup[str(row.filename)]) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, crop_index[int(row.crop_id)], disease_index[int(row.disease_id)]

def make_loader(table, training, batch_size=BATCH_SIZE):
    return DataLoader(PlantDataset(table, train_transform if training else validation_transform),
                      batch_size=batch_size, shuffle=training, num_workers=NUM_WORKERS,
                      pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

seen_loader = make_loader(val_seen_df, False, 32)
unseen_loader = make_loader(val_unseen_df, False, 32)
print('Augmentation:', train_transform)
print('Validation transforms:', validation_transform)

## 4. Hyperparameter configurations and output organization

In [ ]:
# All configurations use full-backbone fine-tuning and identical dataset splits.
# The first three are the primary experiments. Two further experiments are optional.
EXPERIMENTS = {
    'lower_lr': dict(label='Lower Learning Rate', backbone_lr=5e-6, head_lr=5e-4,
                     weight_decay=1e-4, crop_dropout=0.3, disease_dropout=0.2),
    'higher_decay': dict(label='Higher Weight Decay', backbone_lr=1e-5, head_lr=1e-3,
                         weight_decay=1e-3, crop_dropout=0.3, disease_dropout=0.2),
    'higher_dropout': dict(label='Higher Dropout', backbone_lr=1e-5, head_lr=1e-3,
                           weight_decay=1e-4, crop_dropout=0.4, disease_dropout=0.3),
    'very_low_lr': dict(label='Very Low Learning Rate', backbone_lr=1e-6, head_lr=1e-4,
                        weight_decay=1e-4, crop_dropout=0.3, disease_dropout=0.2),
    'lower_dropout': dict(label='Lower Dropout', backbone_lr=1e-5, head_lr=1e-3,
                          weight_decay=1e-4, crop_dropout=0.2, disease_dropout=0.1),
}
PRIMARY = ('lower_lr', 'higher_decay', 'higher_dropout')
OPTIONAL = ('very_low_lr', 'lower_dropout')

def config_for(name, task):
    spec = EXPERIMENTS[name]
    return dict(backbone='resnet50', weights='IMAGENET1K_V2', strategy='full',
                task=task, backbone_lr=spec['backbone_lr'], head_lr=spec['head_lr'],
                weight_decay=spec['weight_decay'], dropout=spec[f'{task}_dropout'],
                batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, seed=SEED,
                scheduler='CosineAnnealingLR(T_max=30)',
                evaluation_split='DEVELOPMENT_VALIDATION_ONLY')

def run_dir(name, task):
    return RESULTS_ROOT / 'experiments' / name / task

def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    try:
        tmp.write_text(json.dumps(data, indent=2), encoding='utf-8')
        os.replace(tmp, path)
    finally:
        if tmp.exists(): tmp.unlink()

def atomic_torch_save(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    try:
        torch.save(data, tmp)
        os.replace(tmp, path)
    finally:
        if tmp.exists(): tmp.unlink()

parameter_table = pd.DataFrame([{'Experiment': v['label'], 'Backbone LR': v['backbone_lr'],
    'Head LR': v['head_lr'], 'Weight Decay': v['weight_decay'],
    'Crop Dropout': v['crop_dropout'], 'Disease Dropout': v['disease_dropout']}
    for v in EXPERIMENTS.values()])
display(parameter_table)
save_json({'experiments': EXPERIMENTS, 'primary': PRIMARY, 'optional': OPTIONAL,
           'training_images': 6905, 'seen_validation_images': 695,
           'unseen_validation_images': 400, 'candidate_pairs': 9625,
           'maximum_epochs': MAX_EPOCHS}, RESULTS_ROOT / 'experiment_plan.json')

## 5. Model architecture, optimisation and reproducibility helpers

In [ ]:
def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build_model(task, dropout):
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    n_classes = 55 if task == 'crop' else 175
    model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, n_classes))
    assert all(p.requires_grad for p in model.parameters()), 'Full fine-tuning must unfreeze all layers.'
    return model.to(DEVICE)

def build_optimizer(model, cfg):
    backbone = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
    head = list(model.fc.parameters())
    return torch.optim.AdamW([{'params': backbone, 'lr': cfg['backbone_lr']},
                              {'params': head, 'lr': cfg['head_lr']}],
                             weight_decay=cfg['weight_decay'])

def capture_rng():
    result = {'python':random.getstate(), 'numpy':np.random.get_state(),
              'torch':torch.get_rng_state()}
    if DEVICE.type == 'cuda': result['cuda'] = torch.cuda.get_rng_state_all()
    return result

def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if DEVICE.type == 'cuda' and 'cuda' in state:
        torch.cuda.set_rng_state_all(state['cuda'])

print('ResNet-50 V2 architecture and optimizer functions ready.')

## 6. One-epoch training and Development Validation functions

In [ ]:
def train_epoch(model, loader, task, optimizer, scaler, description):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total, loss_sum, correct = 0, 0.0, 0
    progress = tqdm(loader, desc=description, leave=False)
    for images, crops, diseases in progress:
        images = images.to(DEVICE, non_blocking=True)
        labels = (crops if task == 'crop' else diseases).to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=AMP):
            logits = model(images)
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        n = labels.numel()
        total += n
        loss_sum += loss.item() * n
        correct += (logits.argmax(1) == labels).sum().item()
        progress.set_postfix(loss=f'{loss_sum/total:.3f}', acc=f'{correct/total:.1%}')
    return {'loss':loss_sum/total, 'top1':correct/total}

@torch.no_grad()
def evaluate_task(model, loader, task):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total, loss_sum, correct = 0, 0.0, 0
    for images, crops, diseases in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = (crops if task == 'crop' else diseases).to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=AMP):
            logits = model(images)
            loss = loss_fn(logits, labels)
        total += labels.numel()
        loss_sum += loss.item() * labels.numel()
        correct += (logits.argmax(1) == labels).sum().item()
    return {'loss':loss_sum/total, 'top1':correct/total}

@torch.no_grad()
def predict_log_probabilities(model, loader):
    model.eval()
    output = []
    for images, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=AMP):
            logits = model(images)
        output.append(F.log_softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(output, axis=0)

def harmonic(a, b):
    return 0.0 if a + b == 0 else 2.0*a*b/(a+b)

print('Training and Development Validation functions ready.')

## 7. Per-epoch training plots and recovery functions

In [ ]:
def plot_history(history, name, task, folder):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    x = df.epoch.to_numpy()
    for field, label in [('train_loss','Training'), ('seen_loss','Development Seen'),
                         ('unseen_loss','Development Unseen')]:
        axes[0].plot(x, df[field], marker='o', markersize=3, label=label)
    for field, label in [('train_acc','Training'), ('seen_acc','Development Seen'),
                         ('unseen_acc','Development Unseen')]:
        axes[1].plot(x, df[field]*100, marker='o', markersize=3, label=label)
    axes[2].plot(x, df.individual_h*100, marker='o', markersize=3, label='Current HM')
    axes[2].plot(x, df.best_h*100, linestyle='--', label='Best HM')
    for ax, title, ylabel in zip(axes, ('Cross-Entropy Loss','Top-1 Accuracy','Individual Validation HM'),
                                 ('Loss','Accuracy (%)','HM (%)')):
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(f'{EXPERIMENTS[name]["label"]} | {task.title()} | DEVELOPMENT VALIDATION')
    fig.tight_layout()
    fig.savefig(folder/'training_curves.png', dpi=170, bbox_inches='tight')
    clear_output(wait=True)
    display(fig)
    plt.close(fig)

def save_validation_predictions(model, folder):
    destination = folder/'best_validation.npz'
    temporary = folder/'best_validation.npz.tmp'
    with open(temporary, 'wb') as stream:
        np.savez_compressed(stream, seen=predict_log_probabilities(model, seen_loader),
                            unseen=predict_log_probabilities(model, unseen_loader))
    os.replace(temporary, destination)

print('Plotting and recovery utilities ready.')

## 8. Resumable Full Fine-Tuning function

In [ ]:
def train_to_epoch(name, task, target_epoch):
    assert name in EXPERIMENTS and task in ('crop', 'disease')
    assert target_epoch in (SCREEN_EPOCHS, SHORTLIST_EPOCHS, MAX_EPOCHS)
    folder = run_dir(name, task)
    folder.mkdir(parents=True, exist_ok=True)
    cfg = config_for(name, task)
    config_path = folder/'config.json'
    resume_path = folder/'resume_state.pt'
    history_path = folder/'history.csv'
    best_path = folder/'best.pt'

    if config_path.exists():
        assert json.loads(config_path.read_text()) == cfg, 'Stored configuration mismatch.'
    else:
        assert not any(folder.iterdir()), f'Unrecognized existing files: {folder}'
        save_json(cfg, config_path)

    if resume_path.exists():
        state = torch.load(resume_path, map_location='cpu', weights_only=False)
        assert state['config'] == cfg and state['task'] == task and state['name'] == name
        assert best_path.exists() and (folder/'best_validation.npz').exists()
        if state['epoch'] >= target_epoch:
            print(f'SKIP {name}/{task}: already completed epoch {state["epoch"]}.')
            return
    else:
        assert not history_path.exists() and not best_path.exists(), 'Partial run without resume state.'

    seed_all()
    model = build_model(task, cfg['dropout'])
    optimizer = build_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    history, best_h, best_epoch, first_epoch = [], -1.0, 0, 1

    if resume_path.exists():
        model.load_state_dict(state['model_state'])
        optimizer.load_state_dict(state['optimizer_state'])
        scheduler.load_state_dict(state['scheduler_state'])
        scaler.load_state_dict(state['scaler_state'])
        history = state['history']
        best_h, best_epoch = state['best_h'], state['best_epoch']
        first_epoch = state['epoch'] + 1
        restore_rng(state['rng'])

    train_loader = make_loader(train_df, True, BATCH_SIZE)
    print(f'{EXPERIMENTS[name]["label"]} / {task.title()}: epochs {first_epoch}–{target_epoch}')
    try:
        for epoch in range(first_epoch, target_epoch+1):
            started = time.time()
            train = train_epoch(model, train_loader, task, optimizer, scaler,
                                f'{name} {task} epoch {epoch}/{MAX_EPOCHS}')
            seen = evaluate_task(model, seen_loader, task)
            unseen = evaluate_task(model, unseen_loader, task)
            current_h = harmonic(seen['top1'], unseen['top1'])
            if current_h > best_h + 1e-9:
                best_h, best_epoch = current_h, epoch
                atomic_torch_save({'state_dict': {k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                    'task':task, 'backbone':'resnet50', 'pretrained_weights':'IMAGENET1K_V2',
                    'fine_tuning':'full', 'epoch':epoch, 'config':cfg,
                    'crop_ids':crop_ids, 'disease_ids':disease_ids,
                    'selection_metric':'individual_development_hm', 'score':best_h}, best_path)
                save_validation_predictions(model, folder)
            history.append({'epoch':epoch, 'train_loss':train['loss'], 'train_acc':train['top1'],
                'seen_loss':seen['loss'], 'seen_acc':seen['top1'],
                'unseen_loss':unseen['loss'], 'unseen_acc':unseen['top1'],
                'individual_h':current_h, 'best_h':best_h, 'seconds':time.time()-started})
            scheduler.step()
            atomic_torch_save({'name':name, 'task':task, 'config':cfg, 'epoch':epoch,
                'history':copy.deepcopy(history), 'best_h':best_h, 'best_epoch':best_epoch,
                'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer_state':optimizer.state_dict(), 'scheduler_state':scheduler.state_dict(),
                'scaler_state':scaler.state_dict(), 'rng':capture_rng()}, resume_path)
            pd.DataFrame(history).to_csv(history_path, index=False)
            plot_history(history, name, task, folder)
            print(f'{EXPERIMENTS[name]["label"]} | {task.upper()} | epoch {epoch}/{target_epoch}')
            print(f'Train top-1: {train["top1"]:.2%} | loss: {train["loss"]:.4f}')
            print(f'DEVELOPMENT seen top-1: {seen["top1"]:.2%} | loss: {seen["loss"]:.4f}')
            print(f'DEVELOPMENT unseen top-1: {unseen["top1"]:.2%} | loss: {unseen["loss"]:.4f}')
            print(f'Individual development HM: {current_h:.2%} | best: {best_h:.2%} (epoch {best_epoch})')
            print(f'Saved checkpoint and history: {folder}')
    finally:
        del model, optimizer, scheduler, scaler, train_loader
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## 9. Composition evaluation — 9,625 candidate pairs

In [ ]:
def composition_metrics_from_saved(name):
    crop_file = run_dir(name, 'crop')/'best_validation.npz'
    disease_file = run_dir(name, 'disease')/'best_validation.npz'
    assert crop_file.exists() and disease_file.exists(), f'Missing task predictions: {name}'
    output = {}
    with np.load(crop_file) as crop, np.load(disease_file) as disease:
        for split, table in [('seen', val_seen_df), ('unseen', val_unseen_df)]:
            crop_log, disease_log = crop[split], disease[split]
            assert crop_log.shape == (len(table), 55)
            assert disease_log.shape == (len(table), 175)
            score = crop_log[:, :, None] + disease_log[:, None, :]
            prediction = score.reshape(len(table), 9625).argmax(axis=1)
            predicted_crop = np.array(crop_ids)[prediction // 175]
            predicted_disease = np.array(disease_ids)[prediction % 175]
            output[split] = {
                'composition_top1': float(np.mean((predicted_crop == table.crop_id.to_numpy()) &
                                                  (predicted_disease == table.disease_id.to_numpy()))),
                'crop_top1': float(np.mean(crop_log.argmax(1) ==
                           np.array([crop_index[int(x)] for x in table.crop_id]))),
                'disease_top1': float(np.mean(disease_log.argmax(1) ==
                              np.array([disease_index[int(x)] for x in table.disease_id]))),
            }
    output['composition_hm'] = harmonic(output['seen']['composition_top1'],
                                        output['unseen']['composition_top1'])
    return output

def phase_report(name, phase, expected_epoch):
    folder = RESULTS_ROOT/'development_validation'/phase
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder/f'{name}.json'
    if destination.exists():
        report = json.loads(destination.read_text())
        assert report['phase'] == phase and report['screen_or_target_epoch'] == expected_epoch
        return report
    epochs = []
    for task in ('crop','disease'):
        resume = run_dir(name, task)/'resume_state.pt'
        assert resume.exists(), f'Missing training result: {name}/{task}'
        state = torch.load(resume, map_location='cpu', weights_only=False)
        epochs.append(state['epoch'])
    assert epochs == [expected_epoch, expected_epoch], (
        f'Cannot recreate {phase} snapshot after further training: {name}; epochs={epochs}')
    metrics = composition_metrics_from_saved(name)
    report = {'experiment':name, 'label':EXPERIMENTS[name]['label'],
              'phase':phase, 'screen_or_target_epoch':expected_epoch,
              'evaluation':'DEVELOPMENT VALIDATION — NOT OFFICIAL TEST',
              'candidate_pairs':9625, **metrics}
    save_json(report, destination)
    return report

def compare_phase(names, phase, target_epoch):
    reports = [phase_report(name, phase, target_epoch) for name in names]
    df = pd.DataFrame([{'Experiment':x['label'], 'ID':x['experiment'],
        'Seen Dev Top-1 (%)':100*x['seen']['composition_top1'],
        'Unseen Dev Top-1 (%)':100*x['unseen']['composition_top1'],
        'Development Composition HM (%)':100*x['composition_hm']} for x in reports])
    df = df.sort_values('Development Composition HM (%)', ascending=False)
    display(df.round(2))
    destination = RESULTS_ROOT/'development_validation'/f'{phase}_comparison.csv'
    df.to_csv(destination, index=False)
    return df['ID'].tolist()

print('Composition evaluator ready: 55 × 175 = 9,625 candidates.')

## 10. Primary experiments — five-epoch screening

Runs **Lower Learning Rate**, **Higher Weight Decay**, and **Higher Dropout** independently for Crop and Disease. After all six short training runs complete, it saves the **immutable five-epoch Development Validation comparison**. If the runtime disconnects, re-run this cell after re-running the notebook setup.

In [ ]:
for experiment in PRIMARY:
    for task in ('crop', 'disease'):
        train_to_epoch(experiment, task, SCREEN_EPOCHS)

primary_screen_order = compare_phase(PRIMARY, 'primary_screen5', SCREEN_EPOCHS)
print('Five-epoch primary screening completed.')
print('Selected for continuation:', [EXPERIMENTS[x]['label'] for x in primary_screen_order[:2]])

## 11. Automatic continuation — best two to 15 epochs, best one to 30

This cell **automatically selects the two highest five-epoch Development Composition HM configurations**, resumes their Crop and Disease training to epoch 15, then selects the best of those two to continue to epoch 30. It uses saved phase snapshots even if run again after completion. It does **not** restart from ImageNet on continuation.

In [ ]:
screen_order = compare_phase(PRIMARY, 'primary_screen5', SCREEN_EPOCHS)
shortlist = screen_order[:2]

for experiment in shortlist:
    for task in ('crop', 'disease'):
        train_to_epoch(experiment, task, SHORTLIST_EPOCHS)

shortlist_order = compare_phase(shortlist, 'primary_shortlist15', SHORTLIST_EPOCHS)
selected = shortlist_order[0]

for task in ('crop', 'disease'):
    train_to_epoch(selected, task, MAX_EPOCHS)

compare_phase([selected], 'primary_final30', MAX_EPOCHS)
save_json({'shortlist':shortlist, 'selected':selected, 'selection_metric':
           'development_composition_hm'}, RESULTS_ROOT/'primary_selection.json')
print('Selected primary configuration:', EXPERIMENTS[selected]['label'])
print('Final training: 30 epochs completed for Crop and Disease.')

## 12. Optional experiments — five-epoch screening

Runs **Very Low Learning Rate** and **Lower Dropout**. These are separate from the primary experiments. Optional results are compared at the same **five-epoch budget** before deciding whether to continue.

In [ ]:
for experiment in OPTIONAL:
    for task in ('crop', 'disease'):
        train_to_epoch(experiment, task, SCREEN_EPOCHS)

optional_screen_order = compare_phase(OPTIONAL, 'optional_screen5', SCREEN_EPOCHS)
print('Optional five-epoch screening complete.')

## 13. Optional automatic continuation

The highest-performing optional five-epoch configuration is continued **only if it exceeds the best primary five-epoch configuration**. It then continues to epoch 15, and to epoch 30 if it remains promising. All decisions and final results are recorded as Development Validation results. This rule is a time-budget heuristic, not a guarantee of the globally strongest configuration.

In [ ]:
primary_screen = compare_phase(PRIMARY, 'primary_screen5', SCREEN_EPOCHS)
optional_screen = compare_phase(OPTIONAL, 'optional_screen5', SCREEN_EPOCHS)
primary_report = phase_report(primary_screen[0], 'primary_screen5', SCREEN_EPOCHS)
optional_report = phase_report(optional_screen[0], 'optional_screen5', SCREEN_EPOCHS)

if optional_report['composition_hm'] > primary_report['composition_hm']:
    candidate = optional_screen[0]
    for task in ('crop','disease'):
        train_to_epoch(candidate, task, SHORTLIST_EPOCHS)
    compare_phase([candidate], 'optional_shortlist15', SHORTLIST_EPOCHS)
    for task in ('crop','disease'):
        train_to_epoch(candidate, task, MAX_EPOCHS)
    compare_phase([candidate], 'optional_final30', MAX_EPOCHS)
    print('Optional candidate continued:', EXPERIMENTS[candidate]['label'])
else:
    print('No optional configuration exceeded the best primary five-epoch HM.')
    print('Optional screening results retained; no additional training required.')

In [ ]:

# ============================================================
# ALL PRIMARY EXPERIMENTS: EPOCH 5 / 15 / 30 COMPARISON
# Includes:
#   1. Lower Learning Rate
#   2. Higher Weight Decay
#   3. Higher Dropout
#
# Shows:
#   - Seen Composition Accuracy
#   - Unseen Composition Accuracy
#   - Development Composition HM
#   - Improvement between stages
#
# No training is performed.
# ============================================================

import json
import pandas as pd
import numpy as np
from IPython.display import display

experiments = [
    "lower_lr",
    "higher_decay",
    "higher_dropout"
]

phases = {
    5: "primary_screen5",
    15: "primary_shortlist15",
    30: "primary_final30"
}

rows = []

for name in experiments:
    for epoch, phase in phases.items():

        report_path = (
            RESULTS_ROOT
            / "development_validation"
            / phase
            / f"{name}.json"
        )

        if report_path.exists():
            with open(report_path, "r") as f:
                report = json.load(f)

            seen = report["seen"]["composition_top1"] * 100
            unseen = report["unseen"]["composition_top1"] * 100
            hm = report["composition_hm"] * 100
            status = "Available"

        else:
            seen = np.nan
            unseen = np.nan
            hm = np.nan
            status = "Not trained / No saved report"

        rows.append({
            "Experiment": EXPERIMENTS[name]["label"],
            "Epoch": epoch,
            "Seen Composition (%)": seen,
            "Unseen Composition (%)": unseen,
            "Composition HM (%)": hm,
            "Status": status
        })

comparison = pd.DataFrame(rows)

print("ALL EXPERIMENTS — DEVELOPMENT COMPOSITION RESULTS")
display(comparison.round(2))


# ============================================================
# COMPARE IMPROVEMENT BETWEEN TRAINING STAGES
# ============================================================

improvements = []

for experiment in comparison["Experiment"].unique():

    data = comparison[
        comparison["Experiment"] == experiment
    ].set_index("Epoch")

    for start_epoch, end_epoch in [(5, 15), (15, 30), (5, 30)]:

        before = data.loc[start_epoch]
        after = data.loc[end_epoch]

        valid = (
            pd.notna(before["Composition HM (%)"])
            and pd.notna(after["Composition HM (%)"])
        )

        improvements.append({
            "Experiment": experiment,
            "Comparison": f"{start_epoch} → {end_epoch}",
            "Seen Change (pp)": (
                after["Seen Composition (%)"]
                - before["Seen Composition (%)"]
            ) if valid else np.nan,
            "Unseen Change (pp)": (
                after["Unseen Composition (%)"]
                - before["Unseen Composition (%)"]
            ) if valid else np.nan,
            "HM Change (pp)": (
                after["Composition HM (%)"]
                - before["Composition HM (%)"]
            ) if valid else np.nan
        })

improvement_df = pd.DataFrame(improvements)

print("\nIMPROVEMENT BETWEEN TRAINING STAGES")
display(improvement_df.round(2))


# ============================================================
# SAVE RESULTS TO GOOGLE DRIVE
# ============================================================

output_dir = RESULTS_ROOT / "development_validation"
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "all_experiments_epoch_comparison.csv",
    index=False
)

improvement_df.to_csv(
    output_dir / "all_experiments_epoch_improvements.csv",
    index=False
)

print("\nComparison completed and saved to Google Drive.")
print("Missing results are shown as NaN, not 0%.")


## 14. Final Development Validation summary and evidence

In [ ]:
rows = []
for experiment, spec in EXPERIMENTS.items():
    histories = [run_dir(experiment, task)/'history.csv' for task in ('crop','disease')]
    if not all(path.exists() for path in histories):
        continue
    metrics = composition_metrics_from_saved(experiment)
    rows.append({'Experiment':spec['label'],
                 'Crop Epochs':len(pd.read_csv(histories[0])),
                 'Disease Epochs':len(pd.read_csv(histories[1])),
                 'Backbone LR':spec['backbone_lr'], 'Head LR':spec['head_lr'],
                 'Weight Decay':spec['weight_decay'],
                 'Crop Dropout':spec['crop_dropout'], 'Disease Dropout':spec['disease_dropout'],
                 'Seen Development Top-1 (%)':100*metrics['seen']['composition_top1'],
                 'Unseen Development Top-1 (%)':100*metrics['unseen']['composition_top1'],
                 'Development Composition HM (%)':100*metrics['composition_hm']})

assert rows, 'No completed Crop/Disease experiment pairs.'
summary = pd.DataFrame(rows).sort_values('Development Composition HM (%)', ascending=False)
display(summary.round(2))
summary.to_csv(RESULTS_ROOT/'development_validation'/'latest_summary.csv', index=False)
print('Reference: EfficientNet-B0 C4/D1 Stage 1 development HM = 10.61%')
print('Reference: ResNet-50 V2 Full original Stage 1 development HM = 8.93%')
print('WARNING: latest rows may use different epoch budgets.')
print('Evaluation split: DEVELOPMENT VALIDATION ONLY; OFFICIAL TEST NOT USED.')

In [ ]:

# ============================================================
# NEW EXPERIMENT: HIGHER WEIGHT DECAY + HIGHER DROPOUT
# ResNet-50 ImageNet-1K V2 | Full Fine-Tuning
# ============================================================

NEW_EXP = "higher_decay_dropout"

EXPERIMENTS[NEW_EXP] = dict(
    label="Higher Weight Decay + Higher Dropout",

    backbone_lr=1e-5,
    head_lr=1e-3,

    weight_decay=1e-3,

    crop_dropout=0.4,
    disease_dropout=0.3
)

print("New experiment registered:")
print(EXPERIMENTS[NEW_EXP])

print("\nCrop checkpoint folder:")
print(run_dir(NEW_EXP, "crop"))

print("\nDisease checkpoint folder:")
print(run_dir(NEW_EXP, "disease"))


In [ ]:

# ============================================================
# NEW COMBINATION: EPOCH 5 -> EPOCH 15
# Resumable | Development Validation Only
# Saves exact epoch weights for future transfer learning
# ============================================================

import torch
import pandas as pd
from pathlib import Path
from IPython.display import display

NEW_EXP = "higher_decay_dropout"

assert NEW_EXP in EXPERIMENTS, "Run Cell 1 first."

def save_transfer_snapshot(name, task, target_epoch):
    folder = run_dir(name, task)
    source = folder / "resume_state.pt"

    if not source.exists():
        raise FileNotFoundError(source)

    state = torch.load(
        source,
        map_location="cpu",
        weights_only=False
    )

    actual_epoch = int(state["epoch"])
    destination = folder / f"epoch_{target_epoch}_weights.pt"

    if destination.exists():
        print(f"Already saved: {destination.name}")
        return

    if actual_epoch != target_epoch:
        print(
            f"Snapshot unavailable for {name}/{task}: "
            f"current epoch is {actual_epoch}, "
            f"requested epoch is {target_epoch}."
        )
        return

    # Save model weights without optimizer state.
    atomic_torch_save(
        {
            "epoch": actual_epoch,
            "task": task,
            "experiment": name,
            "config": state["config"],
            "state_dict": state["model_state"],
            "crop_ids": crop_ids,
            "disease_ids": disease_ids
        },
        destination
    )

    print("Saved transfer weights:", destination)


def get_saved_phase(name, phase):
    path = (
        RESULTS_ROOT
        / "development_validation"
        / phase
        / f"{name}.json"
    )

    if not path.exists():
        return None

    import json
    with open(path, "r") as f:
        return json.load(f)


# ============================================================
# STAGE 1: TRAIN TO EPOCH 5
# ============================================================

print("\n" + "=" * 65)
print("STAGE 1: TRAINING TO EPOCH 5")
print("=" * 65)

for task in ("crop", "disease"):
    train_to_epoch(
        NEW_EXP,
        task,
        SCREEN_EPOCHS
    )

    save_transfer_snapshot(
        NEW_EXP,
        task,
        SCREEN_EPOCHS
    )

# Save phase report before continuing training.
phase5 = "combined_regularization5"

report5 = get_saved_phase(NEW_EXP, phase5)

if report5 is None:
    report5 = phase_report(
        NEW_EXP,
        phase5,
        SCREEN_EPOCHS
    )

print("\nEPOCH 5 COMPOSITION RESULTS")
print(
    "Seen:",
    f"{report5['seen']['composition_top1'] * 100:.2f}%"
)
print(
    "Unseen:",
    f"{report5['unseen']['composition_top1'] * 100:.2f}%"
)
print(
    "Composition HM:",
    f"{report5['composition_hm'] * 100:.2f}%"
)


# ============================================================
# STAGE 2: CONTINUE FROM EPOCH 6 TO EPOCH 15
# ============================================================

print("\n" + "=" * 65)
print("STAGE 2: CONTINUING TO EPOCH 15")
print("=" * 65)

for task in ("crop", "disease"):
    train_to_epoch(
        NEW_EXP,
        task,
        SHORTLIST_EPOCHS
    )

    save_transfer_snapshot(
        NEW_EXP,
        task,
        SHORTLIST_EPOCHS
    )

phase15 = "combined_regularization15"

report15 = get_saved_phase(NEW_EXP, phase15)

if report15 is None:
    report15 = phase_report(
        NEW_EXP,
        phase15,
        SHORTLIST_EPOCHS
    )


# ============================================================
# COMPARE WITH PREVIOUS EXPERIMENTS
# ============================================================

rows = []

comparison_specs = [
    ("lower_lr", 5, "primary_screen5"),
    ("higher_decay", 5, "primary_screen5"),
    ("higher_dropout", 5, "primary_screen5"),
    (NEW_EXP, 5, phase5),

    ("higher_decay", 15, "primary_shortlist15"),
    ("higher_dropout", 15, "primary_shortlist15"),
    (NEW_EXP, 15, phase15)
]

for name, epoch, phase in comparison_specs:
    result = get_saved_phase(name, phase)

    if result is None:
        print(f"Report missing: {name}, epoch {epoch}")
        continue

    rows.append({
        "Experiment": EXPERIMENTS[name]["label"],
        "Stage Epoch": epoch,
        "Seen Composition (%)":
            result["seen"]["composition_top1"] * 100,
        "Unseen Composition (%)":
            result["unseen"]["composition_top1"] * 100,
        "Composition HM (%)":
            result["composition_hm"] * 100
    })

comparison = pd.DataFrame(rows)

comparison = comparison.sort_values(
    ["Stage Epoch", "Composition HM (%)"],
    ascending=[True, False]
)

print("\nALL EXPERIMENTS — EPOCH 5 AND 15")
display(comparison.round(2))


# ============================================================
# NEW EXPERIMENT IMPROVEMENT
# ============================================================

print("\nNEW EXPERIMENT: EPOCH 5 -> EPOCH 15")

for metric, label in [
    ("composition_top1", "Seen"),
]:
    pass

seen5 = report5["seen"]["composition_top1"] * 100
seen15 = report15["seen"]["composition_top1"] * 100

unseen5 = report5["unseen"]["composition_top1"] * 100
unseen15 = report15["unseen"]["composition_top1"] * 100

hm5 = report5["composition_hm"] * 100
hm15 = report15["composition_hm"] * 100

print(f"Seen improvement:   {seen15 - seen5:+.2f} pp")
print(f"Unseen improvement: {unseen15 - unseen5:+.2f} pp")
print(f"HM improvement:     {hm15 - hm5:+.2f} pp")


# ============================================================
# SAVE COMPARISON
# ============================================================

output_dir = RESULTS_ROOT / "development_validation"
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "combined_regularization_comparison.csv",
    index=False
)

print("\nTraining completed through Epoch 15.")
print("Results saved to:", output_dir)
print("Original experiments remain unchanged.")
print("Epoch 30 has NOT been started.")


In [ ]:

# ============================================================
# 15. DISEASE GENERALISATION — CONFIGURATION
# ============================================================

DISEASE_EXPERIMENTS = {
    'disease_resolution_320': {
        'label': 'Disease Resolution 320',
        'image_size': 320,
        'weighted_ce': False,
    },
    'disease_weighted_ce': {
        'label': 'Disease Weighted Cross-Entropy',
        'image_size': 224,
        'weighted_ce': True,
    },
}

DISEASE_EXP_ROOT = (
    RESULTS_ROOT / 'disease_generalisation_experiments'
)

DISEASE_EXP_ROOT.mkdir(parents=True, exist_ok=True)

DISEASE_SCHEDULE_EPOCHS = 30
DISEASE_INITIAL_EPOCHS = 5

# Fixed best Crop CNN from original higher_decay experiment
BASELINE_CROP_DIR = run_dir('higher_decay', 'crop')
BASELINE_CROP_CHECKPOINT = BASELINE_CROP_DIR / 'best.pt'
BASELINE_CROP_PREDICTIONS = (
    BASELINE_CROP_DIR / 'best_validation.npz'
)

assert BASELINE_CROP_CHECKPOINT.exists(), (
    f'Missing Crop checkpoint: {BASELINE_CROP_CHECKPOINT}'
)
assert BASELINE_CROP_PREDICTIONS.exists(), (
    f'Missing Crop predictions: {BASELINE_CROP_PREDICTIONS}'
)

def disease_exp_dir(name):
    return DISEASE_EXP_ROOT / name

print('DISEASE GENERALISATION EXPERIMENTS')
print('Training images:', len(train_df))
print('Development Seen:', len(val_seen_df))
print('Development Unseen:', len(val_unseen_df))
print('Crop checkpoint:', BASELINE_CROP_CHECKPOINT)
print('Initial epochs:', DISEASE_INITIAL_EPOCHS)
print('Scheduler T_max:', DISEASE_SCHEDULE_EPOCHS)
print('Output:', DISEASE_EXP_ROOT)

display(pd.DataFrame([
    {
        'Experiment': name,
        'Resolution': spec['image_size'],
        'Weighted CE': spec['weighted_ce'],
        'Backbone LR': 1e-5,
        'Head LR': 1e-3,
        'Weight Decay': 1e-3,
    }
    for name, spec in DISEASE_EXPERIMENTS.items()
]))


In [ ]:

# ============================================================
# 16. DISEASE DATA PROCESSING AND CLASS WEIGHTS
# ============================================================

def disease_transforms(image_size):

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(
            image_size,
            scale=(0.85, 1.0)
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        ),
    ])

    resize_size = round(image_size * 256 / 224)

    val_tf = transforms.Compose([
        transforms.Resize(resize_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD
        ),
    ])

    return train_tf, val_tf


def disease_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def disease_loader(
    table, transform, training=False, epoch=None
):
    generator = torch.Generator()

    if training:
        assert epoch is not None
        generator.manual_seed(SEED + int(epoch))
    else:
        generator.manual_seed(SEED)

    return DataLoader(
        PlantDataset(table, transform),
        batch_size=BATCH_SIZE if training else 32,
        shuffle=training,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        worker_init_fn=disease_seed_worker,
        generator=generator,
    )


# Class frequency calculated from train_fit.csv ONLY
disease_indices = np.array([
    disease_index[int(label)]
    for label in train_df['disease_id']
])

disease_counts = np.bincount(
    disease_indices,
    minlength=len(disease_ids)
)

assert np.all(disease_counts > 0), (
    'Some disease classes have zero training samples.'
)

# Inverse square-root frequency weighting
weights = 1.0 / np.sqrt(
    disease_counts.astype(np.float64)
)

# Normalize and cap at 4.0
weights /= np.average(
    weights,
    weights=disease_counts
)
weights = np.minimum(weights, 4.0)

DISEASE_CLASS_WEIGHTS = torch.tensor(
    weights,
    dtype=torch.float32,
    device=DEVICE
)

print('DISEASE WEIGHT STATISTICS')
print('Min:', round(weights.min(), 4))
print('Median:', round(np.median(weights), 4))
print('Max:', round(weights.max(), 4))
print('Training loss: Weighted CE for E2 only')
print('Validation loss: Standard CE for both experiments')


In [ ]:

# ============================================================
# 17. DISEASE METRICS, GRAPHS AND CHECKPOINT HELPERS
# ============================================================

# Verify that the fixed Crop model uses the same label order
crop_ckpt = torch.load(
    BASELINE_CROP_CHECKPOINT,
    map_location='cpu',
    weights_only=False
)

assert crop_ckpt['crop_ids'] == crop_ids
assert crop_ckpt['disease_ids'] == disease_ids
del crop_ckpt

with np.load(BASELINE_CROP_PREDICTIONS) as data:
    BASELINE_CROP_LOG = {
        'seen': data['seen'].copy(),
        'unseen': data['unseen'].copy()
    }

for split, table in [
    ('seen', val_seen_df),
    ('unseen', val_unseen_df)
]:
    assert BASELINE_CROP_LOG[split].shape == (
        len(table), len(crop_ids)
    )


def disease_composition_metrics(
    disease_seen_log, disease_unseen_log
):
    results = {}

    disease_logs = {
        'seen': disease_seen_log,
        'unseen': disease_unseen_log
    }

    for split, table in [
        ('seen', val_seen_df),
        ('unseen', val_unseen_df)
    ]:

        crop_log = BASELINE_CROP_LOG[split]
        disease_log = disease_logs[split]

        assert disease_log.shape == (
            len(table), len(disease_ids)
        )

        # All 55 × 175 candidate combinations
        scores = (
            crop_log[:, :, None]
            + disease_log[:, None, :]
        )

        best_pair_index = scores.reshape(
            len(table), -1
        ).argmax(axis=1)

        predicted_crop = np.asarray(crop_ids)[
            best_pair_index // len(disease_ids)
        ]

        predicted_disease = np.asarray(disease_ids)[
            best_pair_index % len(disease_ids)
        ]

        true_crop = table['crop_id'].to_numpy()
        true_disease = table['disease_id'].to_numpy()

        results[split] = {
            'composition_top1': float(np.mean(
                (predicted_crop == true_crop)
                & (predicted_disease == true_disease)
            )),
            'disease_top1': float(np.mean(
                disease_log.argmax(axis=1)
                == np.array([
                    disease_index[int(x)]
                    for x in true_disease
                ])
            )),
        }

    results['composition_hm'] = harmonic(
        results['seen']['composition_top1'],
        results['unseen']['composition_top1']
    )

    results['disease_hm'] = harmonic(
        results['seen']['disease_top1'],
        results['unseen']['disease_top1']
    )

    return results


def make_disease_checkpoint(
    model, cfg, epoch, metric, score
):
    return {
        'state_dict': {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        },
        'task': 'disease',
        'backbone': 'resnet50',
        'pretrained_weights': 'IMAGENET1K_V2',
        'fine_tuning': 'full',
        'epoch': int(epoch),
        'config': cfg,
        'crop_ids': crop_ids,
        'disease_ids': disease_ids,
        'selection_metric': metric,
        'score': float(score),
    }


def save_disease_epoch_predictions(
    folder, epoch, seen_log, unseen_log
):
    destination = (
        folder / f'validation_epoch_{epoch:02d}.npz'
    )
    temporary = folder / f'validation_epoch_{epoch:02d}.npz.tmp'

    with open(temporary, 'wb') as file:
        np.savez_compressed(
            file,
            seen=seen_log,
            unseen=unseen_log
        )

    os.replace(temporary, destination)


def plot_disease_history(history, name, folder):

    df = pd.DataFrame(history)
    epoch = int(df['epoch'].iloc[-1])

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # 1. Cross-Entropy Loss
    ax = axes[0, 0]

    for col, label in [
        ('train_loss', 'Training'),
        ('seen_loss', 'Dev Seen'),
        ('unseen_loss', 'Dev Unseen'),
    ]:
        ax.plot(df['epoch'], df[col],
                marker='o', label=label)

    ax.set_title('Loss')
    ax.set_ylabel('Cross-Entropy Loss')

    # 2. Disease Accuracy
    ax = axes[0, 1]

    for col, label in [
        ('train_acc', 'Training'),
        ('seen_acc', 'Dev Seen'),
        ('unseen_acc', 'Dev Unseen'),
    ]:
        ax.plot(
            df['epoch'],
            df[col] * 100,
            marker='o',
            label=label
        )

    ax.set_title('Disease Top-1 Accuracy')
    ax.set_ylabel('Accuracy (%)')

    # 3. Individual Disease HM
    ax = axes[1, 0]

    ax.plot(
        df['epoch'],
        df['disease_hm'] * 100,
        marker='o',
        label='Disease HM'
    )

    ax.plot(
        df['epoch'],
        df['best_disease_hm'] * 100,
        linestyle='--',
        label='Best Disease HM'
    )

    ax.set_title('Disease Development HM')
    ax.set_ylabel('HM (%)')

    # 4. Combined Composition HM
    ax = axes[1, 1]

    ax.plot(
        df['epoch'],
        df['composition_hm'] * 100,
        marker='o',
        label='Composition HM'
    )

    ax.plot(
        df['epoch'],
        df['best_composition_hm'] * 100,
        linestyle='--',
        label='Best Composition HM'
    )

    ax.set_title('Crop + Disease Composition HM')
    ax.set_ylabel('HM (%)')

    for ax in axes.flat:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(
        f'{DISEASE_EXPERIMENTS[name]["label"]} | '
        f'Epoch {epoch}/{DISEASE_SCHEDULE_EPOCHS}'
    )

    fig.tight_layout()

    # Save graph for this exact epoch
    fig.savefig(
        folder / f'training_curves_epoch_{epoch:02d}.png',
        dpi=160,
        bbox_inches='tight'
    )

    # Overwrite only the latest combined graph
    fig.savefig(
        folder / 'training_curves.png',
        dpi=160,
        bbox_inches='tight'
    )

    display(fig)
    plt.close(fig)


print('Composition metrics and graph helpers ready.')
print('Candidate pairs:', len(crop_ids) * len(disease_ids))


In [ ]:

# ============================================================
# 18. DISEASE TRAINING — SAVE EVERY EPOCH
# ============================================================

def train_disease_one_epoch(
    model, loader, optimizer, scaler, loss_fn, description
):
    model.train()

    total = 0
    loss_sum = 0.0
    correct = 0

    progress = tqdm(loader, desc=description, leave=False)

    for images, crops, diseases in progress:

        images = images.to(
            DEVICE, non_blocking=True
        )
        labels = diseases.to(
            DEVICE, non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast('cuda', enabled=AMP):
            logits = model(images)
            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        n = labels.numel()

        total += n
        loss_sum += float(loss.item()) * n
        correct += int(
            (logits.argmax(1) == labels).sum().item()
        )

        progress.set_postfix(
            loss=f'{loss_sum / total:.4f}',
            acc=f'{correct / total:.2%}'
        )

    return {
        'loss': loss_sum / total,
        'top1': correct / total
    }


def run_disease_experiment(name, target_epoch=5):

    assert name in DISEASE_EXPERIMENTS
    assert 1 <= target_epoch <= DISEASE_SCHEDULE_EPOCHS

    spec = DISEASE_EXPERIMENTS[name]

    folder = disease_exp_dir(name)
    folder.mkdir(parents=True, exist_ok=True)

    cfg = config_for('higher_decay', 'disease')

    cfg.update({
        'experiment': name,
        'image_size': spec['image_size'],
        'weighted_ce': spec['weighted_ce'],
        'scheduler': 'CosineAnnealingLR(T_max=30)',
        'loader_seed_strategy': 'seed_plus_epoch',
    })

    config_path = folder / 'config.json'
    resume_path = folder / 'resume_state.pt'
    history_path = folder / 'history.csv'

    if config_path.exists():
        stored_cfg = json.loads(
            config_path.read_text(encoding='utf-8')
        )
        assert stored_cfg == cfg, (
            'Experiment configuration mismatch.'
        )
    else:
        assert not any(folder.iterdir()), (
            f'Existing unrecognized files: {folder}'
        )
        save_json(cfg, config_path)

    # --------------------------------------------------------
    # Initialize from ImageNet V2 pretrained weights
    # --------------------------------------------------------

    seed_all(SEED)

    model = build_model(
        'disease',
        cfg['dropout']
    )

    optimizer = build_optimizer(model, cfg)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=DISEASE_SCHEDULE_EPOCHS
    )

    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=AMP
    )

    history = []

    best_disease_hm = -1.0
    best_composition_hm = -1.0

    first_epoch = 1

    # --------------------------------------------------------
    # Resume existing experiment
    # --------------------------------------------------------

    if resume_path.exists():

        state = torch.load(
            resume_path,
            map_location='cpu',
            weights_only=False
        )

        assert state['config'] == cfg
        assert state['name'] == name

        if state['epoch'] >= target_epoch:
            print(
                f'{name} already reached Epoch '
                f'{state["epoch"]}. Skipping.'
            )
            del model, optimizer, scheduler, scaler
            return

        model.load_state_dict(
            state['model_state']
        )

        optimizer.load_state_dict(
            state['optimizer_state']
        )

        scheduler.load_state_dict(
            state['scheduler_state']
        )

        scaler.load_state_dict(
            state['scaler_state']
        )

        history = state['history']

        best_disease_hm = state['best_disease_hm']
        best_composition_hm = state['best_composition_hm']

        first_epoch = int(state['epoch']) + 1

        restore_rng(state['rng'])

        print(
            f'Resuming {name} from Epoch {first_epoch}'
        )

    else:
        assert not history_path.exists(), (
            'Found history without a resume state.'
        )

    # --------------------------------------------------------
    # Model-specific validation DataLoaders
    # --------------------------------------------------------

    train_tf, val_tf = disease_transforms(
        spec['image_size']
    )

    seen_loader_new = disease_loader(
        val_seen_df,
        val_tf,
        training=False
    )

    unseen_loader_new = disease_loader(
        val_unseen_df,
        val_tf,
        training=False
    )

    # E1: Standard CE
    # E2: Weighted CE
    loss_fn = nn.CrossEntropyLoss(
        weight=(
            DISEASE_CLASS_WEIGHTS
            if spec['weighted_ce']
            else None
        )
    )

    print('\n' + '=' * 60)
    print('EXPERIMENT:', spec['label'])
    print('Input Resolution:', spec['image_size'])
    print('Weighted CE:', spec['weighted_ce'])
    print('Epoch Range:', first_epoch, 'to', target_epoch)
    print('Source Weights: ImageNet-1K V2')
    print('Output:', folder)
    print('=' * 60)

    # --------------------------------------------------------
    # Training Loop
    # --------------------------------------------------------

    try:

        for epoch in range(
            first_epoch,
            target_epoch + 1
        ):

            started = time.time()

            # Fixed per-epoch shuffle/augmentation seed
            train_loader_new = disease_loader(
                train_df,
                train_tf,
                training=True,
                epoch=epoch
            )

            train_result = train_disease_one_epoch(
                model,
                train_loader_new,
                optimizer,
                scaler,
                loss_fn,
                f'{name} | Epoch {epoch}/{DISEASE_SCHEDULE_EPOCHS}'
            )

            del train_loader_new

            # Validation loss remains STANDARD CE
            seen_result = evaluate_task(
                model,
                seen_loader_new,
                'disease'
            )

            unseen_result = evaluate_task(
                model,
                unseen_loader_new,
                'disease'
            )

            # Obtain predictions for composition scoring
            seen_log = predict_log_probabilities(
                model,
                seen_loader_new
            )

            unseen_log = predict_log_probabilities(
                model,
                unseen_loader_new
            )

            metrics = disease_composition_metrics(
                seen_log,
                unseen_log
            )

            disease_hm = metrics['disease_hm']
            composition_hm = metrics['composition_hm']

            # Save predictions for EVERY epoch
            save_disease_epoch_predictions(
                folder, epoch, seen_log, unseen_log
            )

            # Save complete FP32 model for EVERY epoch
            checkpoint = make_disease_checkpoint(
                model,
                cfg,
                epoch,
                'fixed_epoch',
                composition_hm
            )

            epoch_path = (
                folder / f'epoch_{epoch:02d}.pt'
            )

            atomic_torch_save(
                checkpoint,
                epoch_path
            )

            # Best Disease Development HM
            if disease_hm > best_disease_hm + 1e-9:

                best_disease_hm = disease_hm

                shutil.copy2(
                    epoch_path,
                    folder / 'best.pt'
                )

                with open(
                    folder / 'best_validation.npz.tmp',
                    'wb'
                ) as file:
                    np.savez_compressed(
                        file,
                        seen=seen_log,
                        unseen=unseen_log
                    )

                os.replace(
                    folder / 'best_validation.npz.tmp',
                    folder / 'best_validation.npz'
                )

            # Best Combined Composition Development HM
            if composition_hm > best_composition_hm + 1e-9:

                best_composition_hm = composition_hm

                shutil.copy2(
                    epoch_path,
                    folder / 'best_composition.pt'
                )

            history.append({
                'epoch': epoch,
                'train_loss': train_result['loss'],
                'train_acc': train_result['top1'],
                'seen_loss': seen_result['loss'],
                'seen_acc': seen_result['top1'],
                'unseen_loss': unseen_result['loss'],
                'unseen_acc': unseen_result['top1'],
                'disease_hm': disease_hm,
                'best_disease_hm': best_disease_hm,
                'composition_seen':
                    metrics['seen']['composition_top1'],
                'composition_unseen':
                    metrics['unseen']['composition_top1'],
                'composition_hm': composition_hm,
                'best_composition_hm': best_composition_hm,
                'seconds': time.time() - started,
            })

            scheduler.step()

            # Save full resume state after each epoch
            atomic_torch_save({
                'name': name,
                'config': cfg,
                'epoch': epoch,
                'history': copy.deepcopy(history),
                'best_disease_hm': best_disease_hm,
                'best_composition_hm': best_composition_hm,
                'model_state': {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
                'rng': capture_rng(),
            }, resume_path)

            pd.DataFrame(history).to_csv(
                history_path,
                index=False
            )

            print('\n' + '-' * 60)
            print(f'{name} | EPOCH {epoch}/{target_epoch}')
            print('-' * 60)

            print(
                f'Train Accuracy: '
                f'{train_result["top1"]:.2%}'
            )

            print(
                f'Dev Seen Disease: '
                f'{seen_result["top1"]:.2%}'
            )

            print(
                f'Dev Unseen Disease: '
                f'{unseen_result["top1"]:.2%}'
            )

            print(
                f'Disease HM: {disease_hm:.2%}'
            )

            print(
                f'Composition Seen: '
                f'{metrics["seen"]["composition_top1"]:.2%}'
            )

            print(
                f'Composition Unseen: '
                f'{metrics["unseen"]["composition_top1"]:.2%}'
            )

            print(
                f'Composition HM: {composition_hm:.2%}'
            )

            print(
                f'Best Composition HM: '
                f'{best_composition_hm:.2%}'
            )

            print(
                'Saved checkpoint:',
                epoch_path.name
            )

            # Display and save updated graphs
            plot_disease_history(
                history,
                name,
                folder
            )

    finally:

        del model, optimizer, scheduler, scaler
        del seen_loader_new, unseen_loader_new

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print('\nEXPERIMENT COMPLETED:', name)
    print('History:', history_path)
    print('Checkpoints:', folder)


In [ ]:

# ============================================================
# 19. INITIAL DISEASE SCREENING — 5 EPOCHS
# ============================================================

SCREEN_TARGET = 5

for experiment in [
    'disease_resolution_320',
    'disease_weighted_ce'
]:

    run_disease_experiment(
        name=experiment,
        target_epoch=SCREEN_TARGET
    )

print('\nBOTH EXPERIMENTS COMPLETED.')
print('Next: Development Composition Comparison.')


In [ ]:

# ============================================================
# 20. DISEASE EXPERIMENT COMPARISON — FIRST 5 EPOCHS
# Resolution 320 vs Weighted Cross-Entropy
# SAME FIXED CROP CNN — NO ORIGINAL BASELINE
# DEVELOPMENT VALIDATION ONLY
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Experiment configuration
# ------------------------------------------------------------

COMPARE_EXPERIMENTS = {
    "Disease Resolution 320": "disease_resolution_320",
    "Disease Weighted CE": "disease_weighted_ce",
}

FIXED_CROP_FILE = (
    run_dir("higher_decay", "crop")
    / "best_validation.npz"
)

assert FIXED_CROP_FILE.exists(), FIXED_CROP_FILE

with np.load(FIXED_CROP_FILE) as data:
    fixed_crop_logs = {
        "seen": data["seen"].copy(),
        "unseen": data["unseen"].copy()
    }

# ------------------------------------------------------------
# 2. Calculate metrics from saved predictions
# ------------------------------------------------------------

def evaluate_saved_disease(prediction_file):

    results = {}

    with np.load(prediction_file) as data:

        for split, table in [
            ("seen", val_seen_df),
            ("unseen", val_unseen_df),
        ]:

            crop_log = fixed_crop_logs[split]
            disease_log = data[split]

            assert crop_log.shape == (
                len(table), len(crop_ids)
            )

            assert disease_log.shape == (
                len(table), len(disease_ids)
            )

            crop_pred = np.asarray(crop_ids)[
                crop_log.argmax(axis=1)
            ]

            disease_pred = np.asarray(disease_ids)[
                disease_log.argmax(axis=1)
            ]

            actual_crop = table["crop_id"].to_numpy()
            actual_disease = table["disease_id"].to_numpy()

            crop_correct = crop_pred == actual_crop
            disease_correct = disease_pred == actual_disease

            results[split] = {
                "disease": disease_correct.mean(),
                "composition": (
                    crop_correct & disease_correct
                ).mean()
            }

    results["disease_hm"] = harmonic(
        results["seen"]["disease"],
        results["unseen"]["disease"]
    )

    results["composition_hm"] = harmonic(
        results["seen"]["composition"],
        results["unseen"]["composition"]
    )

    return results


# ------------------------------------------------------------
# 3. Evaluate all 5 saved epochs
# ------------------------------------------------------------

rows = []

for label, key in COMPARE_EXPERIMENTS.items():

    folder = disease_exp_dir(key)

    assert folder.exists(), folder

    for epoch in range(1, 6):

        prediction_file = (
            folder
            / f"validation_epoch_{epoch:02d}.npz"
        )

        assert prediction_file.exists(), prediction_file

        metrics = evaluate_saved_disease(
            prediction_file
        )

        rows.append({
            "Experiment": label,
            "Epoch": epoch,

            "Disease Seen (%)":
                metrics["seen"]["disease"] * 100,

            "Disease Unseen (%)":
                metrics["unseen"]["disease"] * 100,

            "Disease HM (%)":
                metrics["disease_hm"] * 100,

            "Composition Seen (%)":
                metrics["seen"]["composition"] * 100,

            "Composition Unseen (%)":
                metrics["unseen"]["composition"] * 100,

            "Composition HM (%)":
                metrics["composition_hm"] * 100,
        })


all_results = pd.DataFrame(rows)

# ------------------------------------------------------------
# 4. Best checkpoint within 5 epochs
# Selection: highest individual Disease HM
# ------------------------------------------------------------

best_rows = []

for label in COMPARE_EXPERIMENTS:

    subset = all_results[
        all_results["Experiment"] == label
    ]

    best = subset.sort_values(
        ["Disease HM (%)", "Epoch"],
        ascending=[False, True]
    ).iloc[0]

    best_rows.append(best)


best_within_5 = pd.DataFrame(best_rows)

# ------------------------------------------------------------
# 5. Exact Epoch 5 comparison
# ------------------------------------------------------------

exact_epoch_5 = all_results[
    all_results["Epoch"] == 5
].copy()

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("=" * 70)
print("DISEASE EXPERIMENT COMPARISON")
print("RESOLUTION 320 vs WEIGHTED CROSS-ENTROPY")
print("=" * 70)

print("Crop CNN: Higher WD Stage 1 Best (Fixed)")
print("Disease CNN: Independently trained")
print("Development split: 695 Seen / 400 Unseen")
print("Candidate pairs: 9,625")
print("No Original Baseline included")

print("\nA. BEST WITHIN 5 EPOCHS")
display(
    best_within_5.sort_values(
        "Composition HM (%)",
        ascending=False
    ).round(2)
)

print("\nB. EXACT EPOCH 5")
display(
    exact_epoch_5.sort_values(
        "Composition HM (%)",
        ascending=False
    ).round(2)
)

# ------------------------------------------------------------
# 7. Save comparison
# ------------------------------------------------------------

output_dir = (
    DISEASE_EXP_ROOT / "comparison_5epoch"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

all_results.to_csv(
    output_dir / "all_epoch_results.csv",
    index=False
)

best_within_5.to_csv(
    output_dir / "best_within_5.csv",
    index=False
)

exact_epoch_5.to_csv(
    output_dir / "exact_epoch_5.csv",
    index=False
)

print("\nSaved comparison files to:")
print(output_dir)


In [ ]:

# ============================================================
# 24. STAGE 2 — CONFIGURATION AND DATASET VERIFICATION
# ============================================================

STAGE2_EPOCHS = 10
STAGE2_BACKBONE_LR = 1e-6
STAGE2_HEAD_LR = 1e-4

STAGE2_ROOT = (
    RESULTS_ROOT / 'stage2_from_higher_decay_best_full8000'
)

STAGE2_SOURCE = {
    task: run_dir('higher_decay', task) / 'best.pt'
    for task in ('crop', 'disease')
}

# Read the ORIGINAL training CSV, not train_fit.csv
original_csv = find_case_insensitive(
    DRIVE_PROJECT,
    'DPD_Small_train.csv'
)

STAGE2_COLUMNS = [
    'filename', 'crop_id', 'disease_id', 'composition_id',
    'crop_name', 'disease_name', 'composition_name'
]

with open(original_csv, encoding='utf-8-sig') as file:
    first_column = file.readline().split(',')[0].strip().lower()

full_train_df = pd.read_csv(
    original_csv,
    names=STAGE2_COLUMNS,
    header=0 if first_column == 'filename' else None,
    encoding='utf-8-sig'
)

for col in ('crop_id', 'disease_id', 'composition_id'):
    full_train_df[col] = pd.to_numeric(
        full_train_df[col],
        errors='raise'
    ).astype(int)

full_train_df['filename'] = (
    full_train_df['filename'].astype(str)
)

# Dataset integrity
assert len(full_train_df) == 8000
assert full_train_df['filename'].is_unique

expected_train_images = (
    set(train_df['filename'])
    | set(val_seen_df['filename'])
    | set(val_unseen_df['filename'])
)

assert set(full_train_df['filename']) == expected_train_images, (
    'Full training data does not match the original '
    '6905 + 695 + 400 development split.'
)

assert set(full_train_df['filename']).issubset(
    set(image_lookup)
), 'Some training images are missing from local storage.'

assert set(full_train_df['crop_id']) == set(crop_ids)
assert set(full_train_df['disease_id']) == set(disease_ids)

# Verify original best checkpoints
STAGE2_SOURCE_METADATA = {}

for task in ('crop', 'disease'):

    path = STAGE2_SOURCE[task]

    assert path.exists(), f'Missing checkpoint: {path}'

    ckpt = torch.load(
        path,
        map_location='cpu',
        weights_only=False
    )

    assert ckpt['task'] == task
    assert ckpt['backbone'] == 'resnet50'
    assert ckpt['crop_ids'] == crop_ids
    assert ckpt['disease_ids'] == disease_ids
    assert ckpt['config'] == config_for('higher_decay', task)

    STAGE2_SOURCE_METADATA[task] = {
        'path': str(path),
        'epoch': int(ckpt['epoch']),
        'development_hm': float(ckpt['score']),
        'dropout': float(ckpt['config']['dropout'])
    }

    print(f'\n{task.upper()} SOURCE CHECKPOINT')
    print('Path:', path)
    print('Stage 1 Best Epoch:', ckpt['epoch'])
    print('Individual Development HM:', f'{ckpt["score"]:.2%}')

    del ckpt

# Confirm original selected pair
baseline_metrics = composition_metrics_from_saved(
    'higher_decay'
)

print('\n' + '=' * 60)
print('STAGE 2 PREFLIGHT')
print('=' * 60)
print('Full Training Images:', len(full_train_df))
print('Crop Classes:', len(crop_ids))
print('Disease Classes:', len(disease_ids))
print('Source Development HM:',
      f'{baseline_metrics["composition_hm"]:.2%}')
print('Additional Epochs:', STAGE2_EPOCHS)
print('Backbone LR:', STAGE2_BACKBONE_LR)
print('Head LR:', STAGE2_HEAD_LR)
print('Output:', STAGE2_ROOT)
print('Official Test: NOT USED')

assert abs(
    baseline_metrics['composition_hm'] - 0.1193
) < 0.001, (
    'Source composition HM does not match '
    'the expected 11.93% baseline. Check source files.'
)

print('\nPREFLIGHT PASSED.')


In [ ]:

# ============================================================
# 25. STAGE 2 — TRAINING, CHECKPOINTS AND GRAPHS
# ============================================================

def stage2_plot_history(history, task, folder):

    df = pd.DataFrame(history)
    epoch = int(df['epoch'].iloc[-1])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Training Loss
    axes[0].plot(
        df['epoch'],
        df['train_loss'],
        marker='o',
        label='Training Loss'
    )
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Stage 2 Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')

    # Training Accuracy
    axes[1].plot(
        df['epoch'],
        df['train_acc'] * 100,
        marker='o',
        label='Training Accuracy'
    )
    axes[1].set_title('Training Accuracy')
    axes[1].set_xlabel('Stage 2 Epoch')
    axes[1].set_ylabel('Top-1 Accuracy (%)')
    axes[1].set_ylim(0, 100)

    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()

    fig.suptitle(
        f'Stage 2 — {task.title()} | '
        f'Epoch {epoch}/{STAGE2_EPOCHS}'
    )
    fig.tight_layout()

    fig.savefig(
        folder / f'training_curves_epoch_{epoch:02d}.png',
        dpi=160,
        bbox_inches='tight'
    )

    fig.savefig(
        folder / 'training_curves.png',
        dpi=160,
        bbox_inches='tight'
    )

    display(fig)
    plt.close(fig)


def stage2_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def stage2_train_loader(epoch):

    # Separate deterministic shuffle / augmentation seed
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)

    return DataLoader(
        PlantDataset(full_train_df, train_transform),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        worker_init_fn=stage2_seed_worker,
        generator=generator
    )


def stage2_model_checkpoint(
    model, task, cfg, epoch, accuracy, loss
):

    return {
        'state_dict': {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        },
        'task': task,
        'backbone': 'resnet50',
        'pretrained_weights': 'IMAGENET1K_V2',
        'fine_tuning': 'full',
        'stage': 2,
        'epoch': epoch,
        'config': cfg,
        'crop_ids': crop_ids,
        'disease_ids': disease_ids,
        'selection_metric': 'stage2_training_accuracy',
        'score': float(accuracy),
        'training_loss': float(loss)
    }


def run_stage2(task):

    assert task in ('crop', 'disease')

    folder = STAGE2_ROOT / task
    folder.mkdir(parents=True, exist_ok=True)

    source = STAGE2_SOURCE[task]

    cfg = {
        'stage': 2,
        'task': task,
        'backbone': 'resnet50',
        'source_checkpoint': str(source),
        'source_epoch': STAGE2_SOURCE_METADATA[task]['epoch'],
        'image_size': 224,
        'epochs': STAGE2_EPOCHS,
        'training_images': 8000,
        'backbone_lr': STAGE2_BACKBONE_LR,
        'head_lr': STAGE2_HEAD_LR,
        'weight_decay': EXPERIMENTS['higher_decay']['weight_decay'],
        'dropout': STAGE2_SOURCE_METADATA[task]['dropout'],
        'batch_size': BATCH_SIZE,
        'seed': SEED,
        'scheduler': 'CosineAnnealingLR(T_max=10)',
        'checkpoint_selection': 'highest_training_accuracy',
        'validation': 'none_full_training_data',
        'official_test': 'not_used_for_selection',
    }

    config_path = folder / 'config.json'
    history_path = folder / 'history.csv'
    resume_path = folder / 'resume_state.pt'
    complete_path = folder / 'COMPLETE.json'

    if config_path.exists():
        saved_cfg = json.loads(
            config_path.read_text(encoding='utf-8')
        )
        assert saved_cfg == cfg, (
            'Stage 2 configuration mismatch.'
        )
    else:
        assert not any(folder.iterdir()), (
            f'Existing unrecognized files in: {folder}'
        )
        save_json(cfg, config_path)

    if complete_path.exists():
        print(f'{task.upper()} STAGE 2 already completed.')
        print('Output:', folder)
        return

    # --------------------------------------------------------
    # Build original ResNet-50 architecture
    # --------------------------------------------------------

    seed_all(SEED)

    # No ImageNet download required: load saved Stage 1 weights
    model = models.resnet50(weights=None)

    model.fc = nn.Sequential(
        nn.Dropout(cfg['dropout']),
        nn.Linear(
            model.fc.in_features,
            len(crop_ids) if task == 'crop' else len(disease_ids)
        )
    )

    model = model.to(DEVICE)

    # Same optimizer parameter grouping as Standalone
    optimizer = build_optimizer(model, cfg)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=STAGE2_EPOCHS
    )

    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=AMP
    )

    history = []
    best_acc = -1.0
    best_epoch = 0
    start_epoch = 1

    # --------------------------------------------------------
    # Resume, or load original Stage 1 best checkpoint
    # --------------------------------------------------------

    if resume_path.exists():

        state = torch.load(
            resume_path,
            map_location='cpu',
            weights_only=False
        )

        assert state['config'] == cfg
        assert state['task'] == task

        model.load_state_dict(state['model_state'])
        optimizer.load_state_dict(state['optimizer_state'])
        scheduler.load_state_dict(state['scheduler_state'])
        scaler.load_state_dict(state['scaler_state'])

        history = state['history']
        best_acc = float(state['best_acc'])
        best_epoch = int(state['best_epoch'])
        start_epoch = int(state['epoch']) + 1

        restore_rng(state['rng'])

        print(
            f'Resuming {task.upper()} from '
            f'Stage 2 Epoch {start_epoch}'
        )

    else:

        assert not history_path.exists(), (
            'History exists without resume state.'
        )

        source_ckpt = torch.load(
            source,
            map_location='cpu',
            weights_only=False
        )

        model.load_state_dict(source_ckpt['state_dict'])

        print(
            f'Starting {task.upper()} from original '
            f'Stage 1 Epoch {source_ckpt["epoch"]}'
        )

        del source_ckpt

    # --------------------------------------------------------
    # Fixed 10-Epoch Stage 2 training
    # --------------------------------------------------------

    print('\n' + '=' * 65)
    print('STAGE 2 —', task.upper())
    print('=' * 65)
    print('Training Images: 8000')
    print('Epoch Range:', start_epoch, 'to', STAGE2_EPOCHS)
    print('Backbone LR:', STAGE2_BACKBONE_LR)
    print('Head LR:', STAGE2_HEAD_LR)
    print('Weight Decay:', cfg['weight_decay'])
    print('Source:', source)
    print('Output:', folder)
    print('Official Test: NOT USED')
    print('=' * 65)

    try:

        for epoch in range(start_epoch, STAGE2_EPOCHS + 1):

            started = time.time()

            loader = stage2_train_loader(epoch)

            metrics = train_epoch(
                model,
                loader,
                task,
                optimizer,
                scaler,
                f'STAGE 2 {task.upper()} | '
                f'Epoch {epoch}/{STAGE2_EPOCHS}'
            )

            del loader

            scheduler.step()

            # Save EVERY epoch checkpoint
            epoch_path = folder / f'epoch_{epoch:02d}.pt'

            atomic_torch_save(
                stage2_model_checkpoint(
                    model,
                    task,
                    cfg,
                    epoch,
                    metrics['top1'],
                    metrics['loss']
                ),
                epoch_path
            )

            # Save highest Training Accuracy checkpoint
            # Strict improvement: earliest epoch wins a tie
            if metrics['top1'] > best_acc + 1e-12:

                best_acc = float(metrics['top1'])
                best_epoch = epoch

                shutil.copy2(
                    epoch_path,
                    folder / 'best_train_acc.pt'
                )

            # Record epoch results
            history.append({
                'epoch': epoch,
                'train_loss': float(metrics['loss']),
                'train_acc': float(metrics['top1']),
                'best_train_acc': best_acc,
                'best_train_epoch': best_epoch,
                'seconds': time.time() - started,
            })

            # Full training state for Resume
            atomic_torch_save({
                'task': task,
                'config': cfg,
                'epoch': epoch,
                'history': copy.deepcopy(history),
                'best_acc': best_acc,
                'best_epoch': best_epoch,
                'model_state': {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
                'rng': capture_rng(),
            }, resume_path)

            pd.DataFrame(history).to_csv(
                history_path,
                index=False
            )

            # Print progress
            print('\n' + '-' * 60)
            print(
                f'STAGE 2 {task.upper()} | '
                f'Epoch {epoch}/{STAGE2_EPOCHS}'
            )
            print(
                'Training Loss:',
                f'{metrics["loss"]:.4f}'
            )
            print(
                'Training Accuracy:',
                f'{metrics["top1"]:.2%}'
            )
            print(
                'Best Training Accuracy:',
                f'{best_acc:.2%} (Epoch {best_epoch})'
            )
            print('Saved:', epoch_path.name)

            # Updated graph after every epoch
            stage2_plot_history(history, task, folder)

        # ----------------------------------------------------
        # Save final Epoch 10 separately
        # ----------------------------------------------------

        final_path = (
            folder / f'final_epoch{STAGE2_EPOCHS}.pt'
        )

        shutil.copy2(
            folder / f'epoch_{STAGE2_EPOCHS:02d}.pt',
            final_path
        )

        save_json({
            'task': task,
            'completed_epochs': STAGE2_EPOCHS,
            'best_training_epoch': best_epoch,
            'best_training_accuracy': best_acc,
            'selected_checkpoint': 'best_train_acc.pt',
            'final_checkpoint': final_path.name,
        }, complete_path)

        print('\n' + '=' * 65)
        print(f'{task.upper()} STAGE 2 COMPLETED')
        print('Best Training Epoch:', best_epoch)
        print('Best Training Accuracy:', f'{best_acc:.2%}')
        print('Selected Checkpoint:',
              folder / 'best_train_acc.pt')
        print('Final Epoch Checkpoint:', final_path)
        print('=' * 65)

    finally:

        del model, optimizer, scheduler, scaler

        gc.collect()

        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()


print('Stage 2 training functions ready.')
print('No training has started.')


In [ ]:

# ============================================================
# 26. STAGE 2 — CROP FULL-DATA FINE-TUNING
# ============================================================

run_stage2('crop')


In [ ]:

# ============================================================
# 27. STAGE 2 — DISEASE FULL-DATA FINE-TUNING
# ============================================================

run_stage2('disease')


In [ ]:

# ============================================================
# 28. STAGE 2 — FINAL RESULTS AND CHECKPOINT VERIFICATION
# ============================================================

stage2_summary_rows = []

for task in ('crop', 'disease'):

    folder = STAGE2_ROOT / task

    history_path = folder / 'history.csv'
    best_path = folder / 'best_train_acc.pt'
    final_path = folder / f'final_epoch{STAGE2_EPOCHS}.pt'
    complete_path = folder / 'COMPLETE.json'

    assert complete_path.exists(), (
        f'{task.upper()} Stage 2 is not complete.'
    )

    assert history_path.exists()
    assert best_path.exists()
    assert final_path.exists()

    history = pd.read_csv(history_path)

    assert len(history) == STAGE2_EPOCHS
    assert history['epoch'].tolist() == list(
        range(1, STAGE2_EPOCHS + 1)
    )

    best_row = history.loc[
        history['train_acc'].idxmax()
    ]

    best_epoch = int(best_row['epoch'])

    selected = torch.load(
        best_path,
        map_location='cpu',
        weights_only=False
    )

    assert selected['task'] == task
    assert int(selected['epoch']) == best_epoch
    assert selected['crop_ids'] == crop_ids
    assert selected['disease_ids'] == disease_ids

    stage2_summary_rows.append({
        'Model': task.title(),
        'Stage 1 Source Epoch':
            STAGE2_SOURCE_METADATA[task]['epoch'],
        'Stage 2 Best Epoch': best_epoch,
        'Highest Training Accuracy (%)':
            float(best_row['train_acc']) * 100,
        'Training Loss at Best Epoch':
            float(best_row['train_loss']),
        'Selected Checkpoint': str(best_path)
    })

    print('\n' + '=' * 60)
    print(task.upper(), 'STAGE 2 RESULTS')
    print('=' * 60)

    display(history.round(4))

    stage2_plot_history(
        history.to_dict('records'),
        task,
        folder
    )

    del selected

stage2_summary = pd.DataFrame(stage2_summary_rows)

print('\nSTAGE 2 — SELECTED CHECKPOINTS')
display(stage2_summary.round(3))

stage2_summary.to_csv(
    STAGE2_ROOT / 'stage2_summary.csv',
    index=False
)

print('\nSTAGE 2 COMPLETE.')
print('Next step: Official Test using best_train_acc.pt')


In [ ]:

# ============================================================
# 29. STAGE 3 — OFFICIAL TEST SETUP
# ============================================================

import hashlib
from sklearn.metrics import confusion_matrix

OFFICIAL_ROOT = STAGE2_ROOT / 'stage3_official_test'

OFFICIAL_CROP_CKPT = (
    STAGE2_ROOT / 'crop' / 'best_train_acc.pt'
)

OFFICIAL_DISEASE_CKPT = (
    STAGE2_ROOT / 'disease' / 'best_train_acc.pt'
)

OFFICIAL_BATCH_SIZE = 32
GALLERY_IMAGES = 12

# ------------------------------------------------------------
# Verify Stage 2 is complete
# ------------------------------------------------------------

SELECTED_OFFICIAL_CKPTS = {}

for task, path in [
    ('crop', OFFICIAL_CROP_CKPT),
    ('disease', OFFICIAL_DISEASE_CKPT),
]:

    assert (STAGE2_ROOT / task / 'COMPLETE.json').exists(), (
        f'{task}: Stage 2 training is not completed.'
    )

    assert path.exists(), f'Missing checkpoint: {path}'

    ckpt = torch.load(
        path,
        map_location='cpu',
        weights_only=False
    )

    history = pd.read_csv(
        STAGE2_ROOT / task / 'history.csv'
    )

    best_row = history.loc[
        history['train_acc'].idxmax()
    ]

    assert ckpt['task'] == task
    assert ckpt['backbone'] == 'resnet50'
    assert ckpt['crop_ids'] == crop_ids
    assert ckpt['disease_ids'] == disease_ids
    assert int(ckpt['epoch']) == int(best_row['epoch'])
    assert ckpt['stage'] == 2

    SELECTED_OFFICIAL_CKPTS[task] = {
        'path': str(path),
        'epoch': int(ckpt['epoch']),
        'training_accuracy': float(best_row['train_acc']),
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    }

    print(f'\n{task.upper()} SELECTED MODEL')
    print('Stage 2 Best Epoch:', ckpt['epoch'])
    print(
        'Training Accuracy:',
        f'{best_row["train_acc"]:.2%}'
    )
    print('Checkpoint:', path)

    del ckpt


# ------------------------------------------------------------
# Read official CSVs
# ------------------------------------------------------------

OFFICIAL_COLUMNS = [
    'filename', 'crop_id', 'disease_id', 'composition_id',
    'crop_name', 'disease_name', 'composition_name'
]


def read_official_csv(path):

    with open(path, encoding='utf-8-sig') as file:
        first_column = (
            file.readline().split(',')[0].strip().lower()
        )

    df = pd.read_csv(
        path,
        names=OFFICIAL_COLUMNS,
        header=0 if first_column == 'filename' else None,
        encoding='utf-8-sig'
    )

    for col in ('crop_id', 'disease_id', 'composition_id'):
        df[col] = pd.to_numeric(
            df[col],
            errors='raise'
        ).astype(int)

    df['filename'] = df['filename'].astype(str)

    return df


official_seen_csv = find_case_insensitive(
    DRIVE_PROJECT,
    'DPD_Small_test_seen.csv'
)

official_unseen_csv = find_case_insensitive(
    DRIVE_PROJECT,
    'DPD_Small_test_unseen.csv'
)

official_dfs = {
    'seen': read_official_csv(official_seen_csv),
    'unseen': read_official_csv(official_unseen_csv),
}

assert len(official_dfs['seen']) == 2000
assert len(official_dfs['unseen']) == 1448

train_filenames = set(full_train_df['filename'])
seen_filenames = set(official_dfs['seen']['filename'])
unseen_filenames = set(official_dfs['unseen']['filename'])

assert not (train_filenames & seen_filenames)
assert not (train_filenames & unseen_filenames)
assert not (seen_filenames & unseen_filenames)


# ------------------------------------------------------------
# Locate official images
# ------------------------------------------------------------

missing_names = (
    seen_filenames | unseen_filenames
) - set(image_lookup)

if missing_names:

    print(
        '\nLocating official test images on Google Drive:',
        len(missing_names)
    )

    source_image_dir = find_case_insensitive(
        DRIVE_PROJECT, 'images'
    )

    found = {}

    for path in source_image_dir.rglob('*'):

        if path.is_file() and path.name in missing_names:
            assert path.name not in found, (
                f'Duplicate official filename: {path.name}'
            )
            found[path.name] = path

    assert set(found) == missing_names, (
        'Missing official images: '
        f'{sorted(missing_names - set(found))[:10]}'
    )

    image_lookup.update(found)


# ------------------------------------------------------------
# Verify labels and image paths
# ------------------------------------------------------------

for split, df in official_dfs.items():

    assert df['filename'].is_unique

    assert set(df['crop_id']).issubset(set(crop_ids))
    assert set(df['disease_id']).issubset(set(disease_ids))

    assert all(
        str(name) in image_lookup
        for name in df['filename']
    )

    print(
        f'Official {split.title()}: '
        f'{len(df)} images — PASS'
    )


# ------------------------------------------------------------
# Freeze evaluation protocol
# ------------------------------------------------------------

OFFICIAL_PROTOCOL = {
    'dataset': 'DPD-Small',
    'stage': 'Official Test',
    'crop': SELECTED_OFFICIAL_CKPTS['crop'],
    'disease': SELECTED_OFFICIAL_CKPTS['disease'],
    'selection_rule':
        'highest_stage2_training_accuracy_earliest_tie',
    'seen_images': 2000,
    'unseen_images': 1448,
    'candidate_compositions': 9625,
    'preprocessing':
        'Resize256_CenterCrop224_ImageNetNormalize',
    'composition_score':
        'crop_log_probability_plus_disease_log_probability',
    'checkpoint_selection_uses_official_test': False,
}

protocol_path = (
    OFFICIAL_ROOT / 'frozen_evaluation_protocol.json'
)

if protocol_path.exists():

    existing = json.loads(
        protocol_path.read_text(encoding='utf-8')
    )

    assert existing == OFFICIAL_PROTOCOL, (
        'Official evaluation protocol has changed. '
        'Do not overwrite or mix official test runs.'
    )

else:

    OFFICIAL_ROOT.mkdir(parents=True, exist_ok=True)

    save_json(
        OFFICIAL_PROTOCOL,
        protocol_path
    )


print('\n' + '=' * 60)
print('OFFICIAL TEST PREFLIGHT PASSED')
print('=' * 60)
print('Evaluation:', OFFICIAL_ROOT)
print('No training performed.')


In [ ]:

# ============================================================
# 30. STAGE 3 — OFFICIAL TEST INFERENCE
# ============================================================

class OfficialPlantDataset(Dataset):

    def __init__(self, table):

        self.rows = table.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):

        row = self.rows.iloc[idx]

        with Image.open(
            image_lookup[str(row.filename)]
        ) as image:

            tensor = validation_transform(
                image.convert('RGB')
            )

        return (
            tensor,
            crop_index[int(row.crop_id)],
            disease_index[int(row.disease_id)]
        )


def load_stage2_official_model(task):

    path = (
        OFFICIAL_CROP_CKPT
        if task == 'crop'
        else OFFICIAL_DISEASE_CKPT
    )

    ckpt = torch.load(
        path,
        map_location='cpu',
        weights_only=False
    )

    dropout = ckpt['config']['dropout']

    # Reconstruct the SAME Stage 2 architecture
    model = models.resnet50(weights=None)

    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(
            model.fc.in_features,
            len(crop_ids) if task == 'crop'
            else len(disease_ids)
        )
    )

    model.load_state_dict(
        ckpt['state_dict'],
        strict=True
    )

    model = model.to(DEVICE)
    model.eval()

    del ckpt

    return model


def calculate_topk(scores, targets, k=5):

    indices = np.argpartition(
        scores,
        -k,
        axis=1
    )[:, -k:]

    return np.any(
        indices == targets[:, None],
        axis=1
    )


@torch.inference_mode()
def evaluate_official_split(
    split, crop_model, disease_model
):

    table = official_dfs[split].reset_index(drop=True)

    loader = DataLoader(
        OfficialPlantDataset(table),
        batch_size=OFFICIAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0)
    )

    crop_logs = []
    disease_logs = []

    for images, _, _ in tqdm(
        loader,
        desc=f'OFFICIAL {split.upper()}'
    ):

        images = images.to(
            DEVICE, non_blocking=True
        )

        with torch.autocast('cuda', enabled=AMP):

            crop_logits = crop_model(images)
            disease_logits = disease_model(images)

        crop_logs.append(
            F.log_softmax(
                crop_logits.float(), dim=1
            ).cpu().numpy()
        )

        disease_logs.append(
            F.log_softmax(
                disease_logits.float(), dim=1
            ).cpu().numpy()
        )

    crop_log = np.concatenate(crop_logs)
    disease_log = np.concatenate(disease_logs)

    assert crop_log.shape == (
        len(table), len(crop_ids)
    )

    assert disease_log.shape == (
        len(table), len(disease_ids)
    )

    true_crop = table['crop_id'].map(
        crop_index
    ).to_numpy(dtype=int)

    true_disease = table['disease_id'].map(
        disease_index
    ).to_numpy(dtype=int)

    pred_crop = crop_log.argmax(axis=1)
    pred_disease = disease_log.argmax(axis=1)

    # Combine all crop-disease log probabilities
    composition_scores = (
        crop_log[:, :, None]
        + disease_log[:, None, :]
    ).reshape(len(table), -1)

    true_pair = (
        true_crop * len(disease_ids)
        + true_disease
    )

    pred_pair = composition_scores.argmax(axis=1)

    # Top-1
    crop_correct = pred_crop == true_crop
    disease_correct = pred_disease == true_disease
    composition_correct = pred_pair == true_pair

    assert np.array_equal(
        composition_correct,
        crop_correct & disease_correct
    )

    # Top-5
    crop_top5 = calculate_topk(
        crop_log, true_crop
    )

    disease_top5 = calculate_topk(
        disease_log, true_disease
    )

    composition_top5 = calculate_topk(
        composition_scores, true_pair
    )

    # Image-level predictions
    result = table.copy()

    result['split'] = split

    result['pred_crop_id'] = np.asarray(
        crop_ids
    )[pred_crop]

    result['pred_disease_id'] = np.asarray(
        disease_ids
    )[pred_disease]

    result['pred_composition_crop_id'] = np.asarray(
        crop_ids
    )[pred_pair // len(disease_ids)]

    result['pred_composition_disease_id'] = np.asarray(
        disease_ids
    )[pred_pair % len(disease_ids)]

    result['crop_confidence'] = np.exp(
        crop_log.max(axis=1)
    )

    result['disease_confidence'] = np.exp(
        disease_log.max(axis=1)
    )

    result['crop_correct'] = crop_correct
    result['disease_correct'] = disease_correct
    result['composition_correct'] = composition_correct

    result['crop_top5_correct'] = crop_top5
    result['disease_top5_correct'] = disease_top5
    result['composition_top5_correct'] = composition_top5

    result['error_category'] = np.select(
        [
            crop_correct & disease_correct,
            (~crop_correct) & disease_correct,
            crop_correct & (~disease_correct),
            (~crop_correct) & (~disease_correct),
        ],
        [
            'both_correct',
            'crop_wrong_only',
            'disease_wrong_only',
            'both_wrong'
        ],
        default='unknown'
    )

    stats = {
        'crop_top1': float(crop_correct.mean()),
        'crop_top5': float(crop_top5.mean()),
        'disease_top1': float(disease_correct.mean()),
        'disease_top5': float(disease_top5.mean()),
        'composition_top1': float(
            composition_correct.mean()
        ),
        'composition_top5': float(
            composition_top5.mean()
        ),
    }

    return stats, result


# ------------------------------------------------------------
# Execute official inference
# ------------------------------------------------------------

OFFICIAL_PRED_DIR = OFFICIAL_ROOT / 'predictions'
OFFICIAL_PRED_DIR.mkdir(parents=True, exist_ok=True)

official_stats = {}
official_predictions = {}

# Prevent rerunning after completed evaluation
#assert not (
#    OFFICIAL_ROOT / 'OFFICIAL_EVALUATION_COMPLETE.json'
#).exists(), 'Official evaluation already completed.'

crop_model = load_stage2_official_model('crop')
disease_model = load_stage2_official_model('disease')

try:

    for split in ('seen', 'unseen'):

        stats, predictions = evaluate_official_split(
            split,
            crop_model,
            disease_model
        )

        official_stats[split] = stats
        official_predictions[split] = predictions

        output = (
            OFFICIAL_PRED_DIR
            / f'official_{split}_predictions.csv'
        )

        predictions.to_csv(
            output,
            index=False
        )

        print(f'\nOFFICIAL {split.upper()}')
        print('Crop Top-1:', f'{stats["crop_top1"]:.2%}')
        print('Disease Top-1:', f'{stats["disease_top1"]:.2%}')
        print(
            'Composition Top-1:',
            f'{stats["composition_top1"]:.2%}'
        )
        print('Saved:', output)

finally:

    del crop_model, disease_model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


save_json(
    official_stats,
    OFFICIAL_ROOT / 'official_raw_metrics.json'
)

print('\nOFFICIAL INFERENCE COMPLETED.')
print('Seen / Unseen predictions saved.')


In [ ]:

# ============================================================
# 31. STAGE 3 — OFFICIAL TEST METRICS
# ============================================================

# Can also run this cell after reconnecting Colab
if 'official_stats' not in globals():

    with open(
        OFFICIAL_ROOT / 'official_raw_metrics.json',
        encoding='utf-8'
    ) as file:
        official_stats = json.load(file)


metric_rows = []

for task in ('crop', 'disease', 'composition'):

    for topk in (1, 5):

        key = f'{task}_top{topk}'

        seen = official_stats['seen'][key]
        unseen = official_stats['unseen'][key]

        metric_rows.append({
            'Task': task.title(),
            'Metric': f'Top-{topk}',
            'Official Seen (%)': seen * 100,
            'Official Unseen (%)': unseen * 100,
            'Harmonic Mean (%)':
                harmonic(seen, unseen) * 100
        })


official_metrics_df = pd.DataFrame(metric_rows)

print('=' * 65)
print('RESNET-50 — FINAL OFFICIAL TEST RESULTS')
print('=' * 65)

display(official_metrics_df.round(2))

official_metrics_df.to_csv(
    OFFICIAL_ROOT / 'official_test_metrics.csv',
    index=False
)

primary_hm = harmonic(
    official_stats['seen']['composition_top1'],
    official_stats['unseen']['composition_top1']
)

print(
    '\nOFFICIAL COMPOSITION TOP-1 HM:',
    f'{primary_hm:.2%}'
)

print(
    '\nSaved:',
    OFFICIAL_ROOT / 'official_test_metrics.csv'
)


In [ ]:

# ============================================================
# 35. OFFICIAL TEST — TOP 10 BEST / WORST CONFUSION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CM_ANALYSIS_ROOT = (
    OFFICIAL_ROOT / "confusion_matrices" / "best_worst"
)
CM_ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

TOP_N = 10
MIN_SUPPORT = 5


def prepare_cm_data(df, task):

    df = df.copy()

    if task == "crop":
        df["actual_key"] = df["crop_id"].astype(str)
        df["pred_key"] = df["pred_crop_id"].astype(str)

    elif task == "disease":
        df["actual_key"] = df["disease_id"].astype(str)
        df["pred_key"] = df["pred_disease_id"].astype(str)

    elif task == "composition":
        df["actual_key"] = (
            df["crop_id"].astype(str)
            + "|"
            + df["disease_id"].astype(str)
        )

        df["pred_key"] = (
            df["pred_composition_crop_id"].astype(str)
            + "|"
            + df["pred_composition_disease_id"].astype(str)
        )

    else:
        raise ValueError(task)

    return df


def make_class_names(df, task):

    crop_names = (
        df[["crop_id", "crop_name"]]
        .drop_duplicates("crop_id")
        .set_index("crop_id")["crop_name"]
        .to_dict()
    )

    disease_names = (
        df[["disease_id", "disease_name"]]
        .drop_duplicates("disease_id")
        .set_index("disease_id")["disease_name"]
        .to_dict()
    )

    def label(key):

        if task == "crop":
            c = int(key)
            return f"{c}: {crop_names.get(c, c)}"

        if task == "disease":
            d = int(key)
            return f"{d}: {disease_names.get(d, d)}"

        c, d = map(int, key.split("|"))

        return (
            f"{crop_names.get(c, c)} / "
            f"{disease_names.get(d, d)} [{c},{d}]"
        )

    return label


def plot_best_worst_cm(df, split, task, ranking):

    data = prepare_cm_data(df, task)
    label_name = make_class_names(data, task)

    # Per-class support and true positives
    actual_count = data["actual_key"].value_counts()

    correct = data[
        data["actual_key"] == data["pred_key"]
    ]

    true_positive_count = (
        correct["actual_key"].value_counts()
    )

    report = pd.DataFrame({
        "support": actual_count
    })

    report["TP"] = (
        true_positive_count
        .reindex(report.index, fill_value=0)
    )

    report["FN"] = report["support"] - report["TP"]

    report["recall"] = (
        report["TP"] / report["support"]
    )

    report["class_name"] = [
        label_name(key) for key in report.index
    ]

    # Save ALL classes' recall results
    report.to_csv(
        CM_ANALYSIS_ROOT /
        f"{split}_{task}_per_class_recall.csv",
        index_label="class_key"
    )

    # Exclude classes with too few test images
    eligible = report[
        report["support"] >= MIN_SUPPORT
    ].copy()

    if eligible.empty:
        print(
            f"Skipped {split} {task}: "
            "no classes meet minimum support."
        )
        return

    if ranking == "best":

        selected_report = eligible.sort_values(
            ["recall", "support", "class_name"],
            ascending=[False, False, True]
        ).head(TOP_N)

    else:

        selected_report = eligible.sort_values(
            ["recall", "support", "class_name"],
            ascending=[True, False, True]
        ).head(TOP_N)

    selected = selected_report.index.tolist()

    # Use SAME selected classes on both axes
    # to preserve correct diagonal interpretation.
    subset = data[
        data["actual_key"].isin(selected)
    ]

    matrix = pd.crosstab(
        subset["actual_key"],
        subset["pred_key"]
    )

    # Selected actual classes vs selected predictions
    compact = matrix.reindex(
        index=selected,
        columns=selected,
        fill_value=0
    )

    # OTHER = predicted outside the selected classes
    other = (
        subset.groupby("actual_key").size()
        .reindex(selected, fill_value=0)
        - compact.sum(axis=1)
    )

    compact["OTHER"] = other

    # Verify no errors were silently excluded
    assert compact.to_numpy().sum() == len(subset)

    x_names = [
        label_name(x) for x in selected
    ] + ["OTHER"]

    y_names = [
        label_name(x) for x in selected
    ]

    compact.index = y_names
    compact.columns = x_names

    compact.to_csv(
        CM_ANALYSIS_ROOT /
        f"{split}_{task}_{ranking}_top10_matrix.csv",
        index_label="Actual Class"
    )

    selected_report.to_csv(
        CM_ANALYSIS_ROOT /
        f"{split}_{task}_{ranking}_top10_ranking.csv",
        index_label="class_key"
    )

    # --------------------------------------------------------
    # Plot — standard blue confusion matrix with numbers
    # --------------------------------------------------------

    values = compact.to_numpy()

    fig, ax = plt.subplots(
        figsize=(14, 10)
    )

    image = ax.imshow(
        values,
        cmap="Blues",
        interpolation="nearest",
        aspect="auto",
        vmin=0
    )

    fig.colorbar(
        image,
        ax=ax,
        label="Number of Images"
    )

    ax.set_xticks(
        np.arange(len(x_names))
    )

    ax.set_yticks(
        np.arange(len(y_names))
    )

    ax.set_xticklabels(
        x_names,
        rotation=60,
        ha="right",
        fontsize=8
    )

    ax.set_yticklabels(
        y_names,
        fontsize=8
    )

    # Add visible cell borders
    ax.set_xticks(
        np.arange(-0.5, len(x_names), 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, len(y_names), 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linewidth=1.5
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    # Display NUMBER inside EVERY box
    maximum = values.max()

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):

            count = int(values[i, j])

            text_color = (
                "white"
                if maximum > 0 and count > maximum / 2
                else "black"
            )

            ax.text(
                j,
                i,
                str(count),
                ha="center",
                va="center",
                fontsize=10,
                fontweight="bold",
                color=text_color
            )

    ax.set_title(
        f"OFFICIAL {split.upper()} — "
        f"{task.upper()}\n"
        f"TOP {len(selected)} {ranking.upper()} "
        f"CLASSES (BY RECALL)",
        fontsize=13,
        fontweight="bold",
        pad=20
    )

    ax.set_xlabel(
        "Predicted Class",
        fontsize=11
    )

    ax.set_ylabel(
        "Actual Class",
        fontsize=11
    )

    fig.tight_layout()

    output = (
        CM_ANALYSIS_ROOT /
        f"{split}_{task}_{ranking}_top10.png"
    )

    fig.savefig(
        output,
        dpi=250,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(
        f"\n{split.upper()} | "
        f"{task.upper()} | "
        f"{ranking.upper()}"
    )

    display(
        selected_report[
            ["class_name", "support", "TP", "FN", "recall"]
        ].round(3)
    )

    print("Saved:", output)


# ============================================================
# GENERATE ALL BEST / WORST CONFUSION MATRICES
# ============================================================

for split in ["seen", "unseen"]:

    prediction_path = (
        OFFICIAL_ROOT /
        "predictions" /
        f"official_{split}_predictions.csv"
    )

    df = pd.read_csv(prediction_path)

    for task in ["crop", "disease", "composition"]:

        for ranking in ["best", "worst"]:

            plot_best_worst_cm(
                df,
                split,
                task,
                ranking
            )

print("\nALL BEST / WORST CONFUSION MATRICES SAVED.")
print("Folder:", CM_ANALYSIS_ROOT)


In [ ]:

# ============================================================
# 33. STAGE 3 — OFFICIAL MISCLASSIFICATION ANALYSIS
# ============================================================

MISCLASS_ROOT = (
    OFFICIAL_ROOT / 'misclassification'
)

MISCLASS_ROOT.mkdir(parents=True, exist_ok=True)


def short_name(value, max_length=26):

    text = str(value)

    if len(text) > max_length:
        return text[:max_length - 3] + '...'

    return text


def official_error_gallery(
    subset, output, title, max_images=12
):

    if subset.empty:
        print('No samples:', title)
        return

    # Reproducible visual sample
    sample = subset.sample(
        n=min(max_images, len(subset)),
        random_state=SEED
    )

    n = len(sample)
    columns = 3
    rows = int(np.ceil(n / columns))

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(15, rows * 4.5)
    )

    axes = np.asarray(axes).reshape(-1)

    for ax, (_, row) in zip(
        axes,
        sample.iterrows()
    ):

        path = image_lookup[
            str(row['filename'])
        ]

        with Image.open(path) as image:
            ax.imshow(image.convert('RGB'))

        true_crop = short_name(
            row['crop_name']
        )

        true_disease = short_name(
            row['disease_name']
        )

        pred_crop = short_name(
            row['pred_crop_name']
        )

        pred_disease = short_name(
            row['pred_disease_name']
        )

        ax.set_title(
            f'TRUE: {true_crop} / {true_disease}\n'
            f'PRED: {pred_crop} / {pred_disease}\n'
            f'{short_name(row["filename"], 38)}',
            fontsize=8
        )

        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    fig.suptitle(
        title,
        fontsize=13
    )

    fig.tight_layout()

    fig.savefig(
        output,
        dpi=160,
        bbox_inches='tight'
    )

    display(fig)
    plt.close(fig)


error_counts = []

for split in ('seen', 'unseen'):

    df = official_predictions[split].copy()

    # Translate predictions into readable labels
    df['pred_crop_name'] = (
        df['pred_crop_id'].map(crop_name_lookup)
    )

    df['pred_disease_name'] = (
        df['pred_disease_id'].map(disease_name_lookup)
    )

    split_root = MISCLASS_ROOT / split
    split_root.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        split_root / 'all_predictions_with_names.csv',
        index=False
    )

    categories = {
        'crop_wrong':
            df[~df['crop_correct']],
        'disease_wrong':
            df[~df['disease_correct']],
        'crop_wrong_only':
            df[df['error_category'] == 'crop_wrong_only'],
        'disease_wrong_only':
            df[df['error_category'] == 'disease_wrong_only'],
        'both_wrong':
            df[df['error_category'] == 'both_wrong'],
        'both_correct':
            df[df['error_category'] == 'both_correct'],
    }

    print('\n' + '=' * 60)
    print(f'OFFICIAL {split.upper()} MISCLASSIFICATION')
    print('=' * 60)

    for category, subset in categories.items():

        folder = split_root / category
        folder.mkdir(parents=True, exist_ok=True)

        csv_path = (
            folder / f'{category}_all_samples.csv'
        )

        subset.to_csv(
            csv_path,
            index=False
        )

        error_counts.append({
            'Split': split.title(),
            'Category': category,
            'Count': len(subset),
            'Percentage (%)':
                100 * len(subset) / len(df)
        })

        print(
            f'{category}: '
            f'{len(subset)} / {len(df)} images'
        )

        official_error_gallery(
            subset,
            folder / f'{category}_gallery.png',
            f'OFFICIAL {split.upper()} — {category}',
            max_images=GALLERY_IMAGES
        )


error_summary = pd.DataFrame(error_counts)

print('\nOFFICIAL MISCLASSIFICATION SUMMARY')
display(error_summary.round(2))

error_summary.to_csv(
    OFFICIAL_ROOT / 'misclassification_summary.csv',
    index=False
)

print('\nMisclassification analysis completed.')
print('Saved:', MISCLASS_ROOT)


In [ ]:

# ============================================================
# 34. STAGE 3 — FINAL OFFICIAL TEST AUDIT
# ============================================================

required_files = [
    OFFICIAL_ROOT / 'frozen_evaluation_protocol.json',
    OFFICIAL_ROOT / 'official_raw_metrics.json',
    OFFICIAL_ROOT / 'official_test_metrics.csv',
    OFFICIAL_ROOT / 'misclassification_summary.csv',
]

for split in ('seen', 'unseen'):

    required_files.append(
        OFFICIAL_PRED_DIR
        / f'official_{split}_predictions.csv'
    )

    for task in ('crop', 'disease', 'composition'):

        required_files.append(
            CM_ROOT
            / f'{split}_{task}_full_confusion_matrix.csv'
        )

        required_files.append(
            CM_ROOT
            / f'{split}_{task}_confusion_matrix.png'
        )

    required_files.append(
        MISCLASS_ROOT
        / split
        / 'all_predictions_with_names.csv'
    )

missing = [
    str(path)
    for path in required_files
    if not path.exists()
]

assert not missing, (
    'Missing official output files:\n'
    + '\n'.join(missing)
)

seen_hm = official_stats['seen']['composition_top1']
unseen_hm = official_stats['unseen']['composition_top1']

final_hm = harmonic(seen_hm, unseen_hm)

completion = {
    'status': 'COMPLETE',
    'model': 'ResNet50_V2_Higher_Weight_Decay',
    'source_stage': 'Stage2_Full_Data_Fine_Tuning',
    'crop_checkpoint': SELECTED_OFFICIAL_CKPTS['crop'],
    'disease_checkpoint': SELECTED_OFFICIAL_CKPTS['disease'],
    'official_seen_composition_top1': seen_hm,
    'official_unseen_composition_top1': unseen_hm,
    'official_composition_hm': final_hm,
    'evaluation_files_verified': len(required_files),
}

save_json(
    completion,
    OFFICIAL_ROOT / 'OFFICIAL_EVALUATION_COMPLETE.json'
)

print('=' * 65)
print('FINAL OFFICIAL TEST — COMPLETE')
print('=' * 65)

print('Official Seen:', f'{seen_hm:.2%}')
print('Official Unseen:', f'{unseen_hm:.2%}')
print('Composition HM:', f'{final_hm:.2%}')
print('Verified files:', len(required_files))
print('Output:', OFFICIAL_ROOT)
print('\nNo further model selection using official test data.')


## Methodological notes

- The **six individual CNNs** in primary screening are independent ImageNet V2 initializations; checkpoint continuation is used only when extending an existing configuration from epoch 5 to 15 to 30.
- **Checkpoint selection** uses each task's Seen/Unseen **individual Development HM**. **Configuration ranking** uses combined crop–disease **Development Composition HM** over 9,625 pairs.
- Five-epoch results are stored separately before longer training to prevent accidentally ranking a 30-epoch checkpoint against a five-epoch checkpoint.
- Training loss and validation loss plots help identify potential overfitting; early-stage accuracy alone should not be treated as a reliable final ranking.
- The last summary includes mixed training budgets. For a strictly equal-budget comparison, compare experiments at the same phase or continue additional candidates.
- **Stage 2 full-data fine-tuning (8,000 images) and Official Test evaluation are not part of this notebook.** Do not use Official Test scores to repeatedly tune hyperparameters.
- **Compute budget:** the primary workflow trains six models for five epochs, four models for ten further epochs, and two models for fifteen further epochs (100 total CNN epochs). Actual completion time depends on the GPU, I/O and Colab session. The optional phase adds training.

In [ ]:

# ============================================================
# NEW EXPERIMENTS — CELL 1
# Configuration and dataset verification
# ============================================================

from torch.utils.data import WeightedRandomSampler

NEW_EXP_ROOT = (
    RESULTS_ROOT / "new_disease_experiments_5epoch"
)

NEW_EPOCHS = 5

# Match the original 30-epoch LR schedule during
# the FIRST five epochs.
NEW_SCHEDULER_HORIZON = 30

NEW_CROP_NPZ = (
    run_dir("higher_decay", "crop")
    / "best_validation.npz"
)

NEW_ARMS = {
    "original_5ep": {
        "head": "linear",
        "supcon": False,
        "balanced": False,
        "smoothing": 0.0
    },
    "projection_wider": {
        "head": "wider",
        "supcon": False,
        "balanced": False,
        "smoothing": 0.0
    },
    "projection_deeper": {
        "head": "deeper",
        "supcon": False,
        "balanced": False,
        "smoothing": 0.0
    },
    "projection_supcon": {
        "head": "wider",
        "supcon": True,
        "balanced": False,
        "smoothing": 0.0
    },
    "balanced_sampler": {
        "head": "linear",
        "supcon": False,
        "balanced": True,
        "smoothing": 0.0
    },
    "label_smoothing": {
        "head": "linear",
        "supcon": False,
        "balanced": False,
        "smoothing": 0.05
    }
}

# Verify original development splits
assert len(train_df) == 6905
assert len(val_seen_df) == 695
assert len(val_unseen_df) == 400

assert len(crop_ids) == 55
assert len(disease_ids) == 175

assert NEW_CROP_NPZ.exists(), (
    f"Missing original Crop predictions: {NEW_CROP_NPZ}"
)

with np.load(NEW_CROP_NPZ) as fp:
    FIXED_CROP = {
        "seen": fp["seen"].copy(),
        "unseen": fp["unseen"].copy()
    }

for split, table in [
    ("seen", val_seen_df),
    ("unseen", val_unseen_df)
]:
    assert FIXED_CROP[split].shape == (
        len(table), 55
    )

# Check there is no split overlap
assert not (
    set(train_df.filename)
    & set(val_seen_df.filename)
)
assert not (
    set(train_df.filename)
    & set(val_unseen_df.filename)
)
assert not (
    set(val_seen_df.filename)
    & set(val_unseen_df.filename)
)

# Use original Higher Weight Decay hyperparameters
BASE_CFG = config_for("higher_decay", "disease")

assert BASE_CFG["backbone_lr"] == 1e-5
assert BASE_CFG["head_lr"] == 1e-3
assert BASE_CFG["weight_decay"] == 1e-3
assert BASE_CFG["dropout"] == 0.2

NEW_EXP_ROOT.mkdir(
    parents=True, exist_ok=True
)

print("Configuration verified.")
print("Seed:", SEED)
print("Epochs:", NEW_EPOCHS)
print("Batch Size:", BATCH_SIZE)
print("Output:", NEW_EXP_ROOT)
print("Experiments:", list(NEW_ARMS))


In [ ]:

# ============================================================
# NEW EXPERIMENTS — CELL 2
# Models, data loaders, and SupCon
# ============================================================

class DiseaseProjectionResNet(nn.Module):

    def __init__(self, variant="linear"):
        super().__init__()

        self.backbone = models.resnet50(
            weights=ResNet50_Weights.IMAGENET1K_V2
        )

        in_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        if variant == "linear":
            self.projection = None
            self.classifier = nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(in_dim, len(disease_ids))
            )

        elif variant == "wider":
            self.projection = nn.Sequential(
                nn.Linear(in_dim, 512),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(512, 128)
            )
            self.classifier = nn.Linear(
                128, len(disease_ids)
            )

        elif variant == "deeper":
            self.projection = nn.Sequential(
                nn.Linear(in_dim, 512),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(256, 128)
            )
            self.classifier = nn.Linear(
                128, len(disease_ids)
            )

        else:
            raise ValueError(variant)

        self.variant = variant

    def forward(self, x, return_embedding=False):

        features = self.backbone(x)

        raw = (
            self.projection(features)
            if self.projection is not None
            else features
        )

        logits = self.classifier(raw)

        if return_embedding:
            return logits, F.normalize(
                raw.float(), dim=1
            )

        return logits


class DiseaseTwoViews(Dataset):
    """Two independently augmented views for SupCon."""

    def __init__(self, table):
        self.base = PlantDataset(
            table, train_transform
        )

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):

        row = self.base.rows.iloc[index]
        path = image_lookup[str(row.filename)]

        with Image.open(path) as image:
            im = image.convert("RGB")
            view1 = train_transform(im)
            view2 = train_transform(im)

        return (
            view1,
            view2,
            disease_index[int(row.disease_id)]
        )


def seed_worker_new(worker_id):

    worker_seed = (
        torch.initial_seed() % (2 ** 32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def new_train_loader(epoch, arm):

    spec = NEW_ARMS[arm]

    generator = torch.Generator().manual_seed(
        SEED + epoch * 10007
    )

    dataset = (
        DiseaseTwoViews(train_df)
        if spec["supcon"]
        else PlantDataset(train_df, train_transform)
    )

    kwargs = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        worker_init_fn=seed_worker_new,
        generator=generator
    )

    if spec["balanced"]:

        counts = (
            train_df.disease_id
            .astype(int)
            .value_counts()
        )

        weights = torch.as_tensor(
            [
                1.0 / float(counts[int(d)])
                for d in train_df.disease_id
            ],
            dtype=torch.double
        )

        sampler = WeightedRandomSampler(
            weights,
            num_samples=len(train_df),
            replacement=True,
            generator=generator
        )

        return DataLoader(
            dataset,
            sampler=sampler,
            shuffle=False,
            **kwargs
        )

    return DataLoader(
        dataset,
        shuffle=True,
        **kwargs
    )


def disease_supcon(z, labels, tau=0.07):

    z = F.normalize(z.float(), dim=1)
    n = z.shape[0]

    same = labels[:, None].eq(
        labels[None, :]
    )

    eye = torch.eye(
        n, device=z.device, dtype=torch.bool
    )

    positives = same & (~eye)
    valid = positives.any(dim=1)

    if not valid.any():
        return z.sum() * 0.0

    scores = z @ z.T / tau

    scores = scores - scores.max(
        dim=1, keepdim=True
    ).values.detach()

    log_denominator = torch.logsumexp(
        scores.masked_fill(
            eye, float("-inf")
        ),
        dim=1,
        keepdim=True
    )

    log_probabilities = (
        scores - log_denominator
    )

    positive_average = (
        log_probabilities
        .masked_fill(~positives, 0)
        .sum(dim=1)
        / positives.sum(dim=1).clamp_min(1)
    )

    return -positive_average[valid].mean()


def new_optimizer(model):

    head_params = list(
        model.classifier.parameters()
    )

    if model.projection is not None:
        head_params += list(
            model.projection.parameters()
        )

    return torch.optim.AdamW(
        [
            {
                "params": model.backbone.parameters(),
                "lr": BASE_CFG["backbone_lr"]
            },
            {
                "params": head_params,
                "lr": BASE_CFG["head_lr"]
            }
        ],
        weight_decay=BASE_CFG["weight_decay"]
    )


print("Model architectures ready.")
print("SupCon, random sampling and balanced sampling ready.")


In [ ]:

# ============================================================
# NEW EXPERIMENTS — CELL 3
# Development evaluation and plots
# ============================================================

@torch.inference_mode()
def new_disease_evaluate(model, loader):

    model.eval()

    loss_sum = 0.0
    correct = 0
    count = 0
    outputs = []

    for images, _, labels in loader:

        images = images.to(
            DEVICE, non_blocking=True
        )
        labels = labels.to(
            DEVICE, non_blocking=True
        )

        with torch.autocast(
            "cuda", enabled=AMP
        ):
            logits = model(images)

        log_probs = F.log_softmax(
            logits.float(), dim=1
        )

        loss_sum += float(
            F.nll_loss(
                log_probs,
                labels,
                reduction="sum"
            ).item()
        )

        correct += int(
            (log_probs.argmax(1) == labels)
            .sum().item()
        )

        count += len(labels)

        outputs.append(
            log_probs.cpu().numpy()
        )

    return {
        "loss": loss_sum / count,
        "top1": correct / count
    }, np.concatenate(outputs)


def new_metrics(seen_log, unseen_log):

    results = {}

    for split, table, disease_log in [
        ("seen", val_seen_df, seen_log),
        ("unseen", val_unseen_df, unseen_log)
    ]:

        crop_log = FIXED_CROP[split]

        assert disease_log.shape == (
            len(table), len(disease_ids)
        )

        pred_crop = np.asarray(crop_ids)[
            crop_log.argmax(axis=1)
        ]

        pred_disease = np.asarray(disease_ids)[
            disease_log.argmax(axis=1)
        ]

        crop_correct = (
            pred_crop
            == table.crop_id.to_numpy()
        )

        disease_correct = (
            pred_disease
            == table.disease_id.to_numpy()
        )

        results[f"{split}_disease"] = float(
            disease_correct.mean()
        )

        # All-pairs composition Top-1 equals
        # crop Top-1 AND disease Top-1.
        results[f"{split}_composition"] = float(
            (crop_correct & disease_correct).mean()
        )

    results["disease_hm"] = harmonic(
        results["seen_disease"],
        results["unseen_disease"]
    )

    results["composition_hm"] = harmonic(
        results["seen_composition"],
        results["unseen_composition"]
    )

    return results


def new_plot(history, arm, folder):

    data = pd.DataFrame(history)

    fig, axes = plt.subplots(
        1, 3, figsize=(17, 4)
    )

    # Training and validation losses
    for key in [
        "train_loss",
        "seen_loss",
        "unseen_loss"
    ]:
        axes[0].plot(
            data["epoch"],
            data[key],
            marker="o",
            label=key
        )

    # Accuracy
    for key in [
        "train_acc",
        "seen_disease",
        "unseen_disease"
    ]:
        axes[1].plot(
            data["epoch"],
            data[key] * 100,
            marker="o",
            label=key
        )

    # Generalisation HM
    for key in [
        "disease_hm",
        "composition_hm"
    ]:
        axes[2].plot(
            data["epoch"],
            data[key] * 100,
            marker="o",
            label=key
        )

    titles = [
        "Training and Validation Loss",
        "Disease Accuracy (%)",
        "Development Harmonic Mean (%)"
    ]

    for ax, title in zip(axes, titles):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_xticks(
            range(1, NEW_EPOCHS + 1)
        )
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"{arm} — First {NEW_EPOCHS} Epochs"
    )

    fig.tight_layout()

    epoch = int(data.epoch.iloc[-1])

    fig.savefig(
        folder / f"curves_epoch_{epoch:02d}.png",
        dpi=150,
        bbox_inches="tight"
    )

    fig.savefig(
        folder / "curves_latest.png",
        dpi=150,
        bbox_inches="tight"
    )

    display(fig)
    plt.close(fig)


print("Development evaluation and plotting ready.")


In [ ]:

# ============================================================
# NEW EXPERIMENTS — CELL 4
# Training, validation, checkpoints and resume
# ============================================================

def new_train_epoch(
    model, loader, arm, optimizer, scaler, epoch
):

    spec = NEW_ARMS[arm]

    model.train()

    total = 0
    total_loss = 0.0
    correct = 0

    progress = tqdm(
        loader,
        desc=f"{arm} | Epoch {epoch}/{NEW_EPOCHS}",
        leave=False
    )

    for batch in progress:

        if spec["supcon"]:

            view1, view2, y = batch

            images = torch.cat(
                [view1, view2], dim=0
            ).to(DEVICE, non_blocking=True)

            labels = y.to(
                DEVICE, non_blocking=True
            ).repeat(2)

        else:

            images, _, labels = batch

            images = images.to(
                DEVICE, non_blocking=True
            )

            labels = labels.to(
                DEVICE, non_blocking=True
            )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            "cuda", enabled=AMP
        ):

            if spec["supcon"]:

                logits, embeddings = model(
                    images,
                    return_embedding=True
                )

                ce_loss = F.cross_entropy(
                    logits, labels
                )

                contrastive_loss = disease_supcon(
                    embeddings, labels
                )

                loss = (
                    ce_loss
                    + 0.1 * contrastive_loss
                )

            else:

                logits = model(images)

                loss = F.cross_entropy(
                    logits,
                    labels,
                    label_smoothing=spec["smoothing"]
                )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        n = len(labels)

        total += n
        total_loss += float(
            loss.detach().item()
        ) * n

        correct += int(
            (
                logits.detach().argmax(1)
                == labels
            ).sum().item()
        )

    return {
        "train_acc": correct / total,
        "train_loss": total_loss / total
    }


def run_new_arm(arm):

    assert arm in NEW_ARMS

    folder = NEW_EXP_ROOT / arm
    folder.mkdir(
        parents=True, exist_ok=True
    )

    fingerprint = {
        "arm": arm,
        "spec": NEW_ARMS[arm],
        "epochs": NEW_EPOCHS,
        "base_config": BASE_CFG,
        "train_rows": len(train_df),
        "seen_rows": len(val_seen_df),
        "unseen_rows": len(val_unseen_df),
        "fixed_crop_predictions": str(NEW_CROP_NPZ),
        "seed": SEED,
        "scheduler_T_max": NEW_SCHEDULER_HORIZON
    }

    config_path = folder / "config.json"

    if config_path.exists():

        saved_config = json.loads(
            config_path.read_text()
        )

        assert saved_config == fingerprint, (
            f"Configuration changed for {arm}"
        )

    else:

        assert not any(folder.iterdir()), (
            f"Unrecognized files: {folder}"
        )

        save_json(
            fingerprint,
            config_path
        )

    resume_path = folder / "resume_state.pt"
    history_path = folder / "history.csv"

    # Same initial seed across experiments
    seed_all(SEED)

    model = DiseaseProjectionResNet(
        NEW_ARMS[arm]["head"]
    ).to(DEVICE)

    optimizer = new_optimizer(model)

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=NEW_SCHEDULER_HORIZON
        )
    )

    scaler = torch.amp.GradScaler(
        "cuda", enabled=AMP
    )

    history = []
    first_epoch = 1

    # Resume interrupted experiments
    if resume_path.exists():

        state = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False
        )

        assert state["config"] == fingerprint

        model.load_state_dict(
            state["model"]
        )

        optimizer.load_state_dict(
            state["optimizer"]
        )

        scheduler.load_state_dict(
            state["scheduler"]
        )

        scaler.load_state_dict(
            state["scaler"]
        )

        restore_rng(
            state["rng"]
        )

        history = state["history"]

        first_epoch = int(
            state["epoch"]
        ) + 1

        print(
            f"Resuming {arm} at Epoch {first_epoch}"
        )

    try:

        for epoch in range(
            first_epoch, NEW_EPOCHS + 1
        ):

            loader = new_train_loader(
                epoch, arm
            )

            train_stats = new_train_epoch(
                model,
                loader,
                arm,
                optimizer,
                scaler,
                epoch
            )

            seen_stats, seen_log = (
                new_disease_evaluate(
                    model,
                    seen_loader
                )
            )

            unseen_stats, unseen_log = (
                new_disease_evaluate(
                    model,
                    unseen_loader
                )
            )

            scores = new_metrics(
                seen_log, unseen_log
            )

            row = {
                "epoch": epoch,
                **train_stats,
                "seen_loss": seen_stats["loss"],
                "unseen_loss": unseen_stats["loss"],
                **scores
            }

            history.append(row)

            # Save validation predictions
            np.savez_compressed(
                folder /
                f"predictions_epoch_{epoch:02d}.npz",
                seen=seen_log,
                unseen=unseen_log
            )

            # Save EVERY epoch checkpoint
            checkpoint = {
                "arm": arm,
                "epoch": epoch,
                "state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v
                    in model.state_dict().items()
                },
                "head": NEW_ARMS[arm]["head"],
                "config": fingerprint,
                "crop_ids": crop_ids,
                "disease_ids": disease_ids,
                "validation_scores": scores,
                "train_accuracy": (
                    train_stats["train_acc"]
                )
            }

            atomic_torch_save(
                checkpoint,
                folder / f"epoch_{epoch:02d}.pt"
            )

            scheduler.step()

            # Save full training state for resume
            atomic_torch_save(
                {
                    "config": fingerprint,
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optimizer": (
                        optimizer.state_dict()
                    ),
                    "scheduler": (
                        scheduler.state_dict()
                    ),
                    "scaler": scaler.state_dict(),
                    "history": copy.deepcopy(history),
                    "rng": capture_rng()
                },
                resume_path
            )

            pd.DataFrame(history).to_csv(
                history_path,
                index=False
            )

            # Show graph after EVERY epoch
            new_plot(
                history, arm, folder
            )

            print(
                f"\n{arm} | Epoch {epoch:02d}"
            )

            print(
                "Training Accuracy:",
                f"{train_stats['train_acc']:.2%}"
            )

            print(
                "Disease Seen:",
                f"{scores['seen_disease']:.2%}"
            )

            print(
                "Disease Unseen:",
                f"{scores['unseen_disease']:.2%}"
            )

            print(
                "Disease HM:",
                f"{scores['disease_hm']:.2%}"
            )

            print(
                "Composition HM:",
                f"{scores['composition_hm']:.2%}"
            )

            del loader

        if len(history) == NEW_EPOCHS:

            save_json(
                {
                    "status": "COMPLETE",
                    "epochs": NEW_EPOCHS
                },
                folder / "COMPLETE.json"
            )

    finally:

        del model, optimizer, scheduler, scaler

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(history)


print("Training function ready.")
print("All epochs will be saved individually.")


In [ ]:

# E0 — ORIGINAL HIGHER WEIGHT DECAY CONTROL
# Train from ImageNet-1K V2 for 5 epochs

baseline_history = run_new_arm("original_5ep")

display(
    baseline_history[
        [
            "epoch", "train_acc",
            "seen_disease", "unseen_disease",
            "disease_hm", "composition_hm"
        ]
    ].round(4)
)


In [ ]:

# E1 — PROJECTION WIDER
# ResNet50: 2048 -> 512 -> 128 -> 175

projection_wider_history = run_new_arm(
    "projection_wider"
)

display(projection_wider_history.round(4))


In [ ]:

# E2 — PROJECTION DEEPER
# ResNet50: 2048 -> 512 -> 256 -> 128 -> 175

projection_deeper_history = run_new_arm(
    "projection_deeper"
)

display(projection_deeper_history.round(4))


In [ ]:

# E3 — PROJECTION + SUPERVISED CONTRASTIVE LOSS
# Uses two augmented views of each image
# Loss = CrossEntropy + 0.1 * SupCon

projection_supcon_history = run_new_arm(
    "projection_supcon"
)

display(projection_supcon_history.round(4))


In [ ]:

# E4 — CLASS-BALANCED SAMPLING
# WeightedRandomSampler, inverse disease frequency

balanced_sampler_history = run_new_arm(
    "balanced_sampler"
)

display(balanced_sampler_history.round(4))


In [ ]:

# E5 — LABEL SMOOTHING
# CrossEntropy label_smoothing = 0.05

label_smoothing_history = run_new_arm(
    "label_smoothing"
)

display(label_smoothing_history.round(4))


In [ ]:

# ============================================================
# NEW EXPERIMENTS — CELL 11
# FAIR 5-EPOCH COMPARISON
# ============================================================

rows = []

for arm in NEW_ARMS:

    folder = NEW_EXP_ROOT / arm
    history_path = folder / "history.csv"

    if not history_path.exists():
        print("Not trained:", arm)
        continue

    history = pd.read_csv(history_path)

    history = history[
        history["epoch"] <= NEW_EPOCHS
    ].copy()

    assert history["epoch"].is_unique

    # Select by DEVELOPMENT Disease HM,
    # not official test performance.
    best = history.sort_values(
        ["disease_hm", "epoch"],
        ascending=[False, True]
    ).iloc[0]

    final = history.sort_values(
        "epoch"
    ).iloc[-1]

    exact_five = int(final["epoch"]) == 5

    rows.append({
        "Experiment": arm,
        "Epochs": len(history),
        "Best Epoch": int(best["epoch"]),
        "Disease Seen (%)":
            best["seen_disease"] * 100,
        "Disease Unseen (%)":
            best["unseen_disease"] * 100,
        "Disease HM (%)":
            best["disease_hm"] * 100,
        "Composition Seen (%)":
            best["seen_composition"] * 100,
        "Composition Unseen (%)":
            best["unseen_composition"] * 100,
        "Composition HM (%)":
            best["composition_hm"] * 100,
        "Exact Epoch 5 Disease HM (%)":
            final["disease_hm"] * 100
            if exact_five else np.nan,
        "Exact Epoch 5 Composition HM (%)":
            final["composition_hm"] * 100
            if exact_five else np.nan
    })


comparison = pd.DataFrame(rows)

if comparison.empty:
    print("No experiment results yet.")

else:

    control = comparison[
        comparison["Experiment"] == "original_5ep"
    ]

    assert len(control) == 1, (
        "Run Original Control first."
    )

    original_comp = control.iloc[0][
        "Composition HM (%)"
    ]

    original_disease = control.iloc[0][
        "Disease HM (%)"
    ]

    comparison["Disease HM Delta (pp)"] = (
        comparison["Disease HM (%)"]
        - original_disease
    )

    comparison["Composition HM Delta (pp)"] = (
        comparison["Composition HM (%)"]
        - original_comp
    )

    comparison = comparison.sort_values(
        "Composition HM (%)",
        ascending=False
    )

    print("=" * 65)
    print("BEST DEVELOPMENT CHECKPOINT WITHIN 5 EPOCHS")
    print("=" * 65)

    display(
        comparison.round(2)
    )

    comparison.to_csv(
        NEW_EXP_ROOT / "comparison_best_within_5.csv",
        index=False
    )

    exact = comparison[
        [
            "Experiment",
            "Epochs",
            "Exact Epoch 5 Disease HM (%)",
            "Exact Epoch 5 Composition HM (%)"
        ]
    ].copy()

    original_exact = exact.loc[
        exact["Experiment"] == "original_5ep",
        "Exact Epoch 5 Composition HM (%)"
    ].iloc[0]

    exact["Composition HM Delta (pp)"] = (
        exact["Exact Epoch 5 Composition HM (%)"]
        - original_exact
    )

    print("\n" + "=" * 65)
    print("EXACT EPOCH 5 — SAME TRAINING BUDGET")
    print("=" * 65)

    display(
        exact.round(2)
    )

    exact.to_csv(
        NEW_EXP_ROOT / "comparison_exact_epoch5.csv",
        index=False
    )

    print("\nOriginal Control Composition HM:",
          f"{original_comp:.2f}%")

    print("\nSaved comparison CSVs to:")
    print(NEW_EXP_ROOT)


In [ ]:

# ============================================================
# CELL 11 — UNIFIED 5-EPOCH DEVELOPMENT COMPARISON
# Original + 5 New Experiments + 2 Previous Experiments
# No training, no official test
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. All experiment locations
# ------------------------------------------------------------

ALL_EXPERIMENTS = {
    "Original 5ep": {
        "folder": NEW_EXP_ROOT / "original_5ep",
        "pattern": "predictions_epoch_{epoch:02d}.npz",
    },
    "Projection Wider": {
        "folder": NEW_EXP_ROOT / "projection_wider",
        "pattern": "predictions_epoch_{epoch:02d}.npz",
    },
    "Projection Deeper": {
        "folder": NEW_EXP_ROOT / "projection_deeper",
        "pattern": "predictions_epoch_{epoch:02d}.npz",
    },
    "Projection + SupCon": {
        "folder": NEW_EXP_ROOT / "projection_supcon",
        "pattern": "predictions_epoch_{epoch:02d}.npz",
    },
    "Balanced Sampler": {
        "folder": NEW_EXP_ROOT / "balanced_sampler",
        "pattern": "predictions_epoch_{epoch:02d}.npz",
    },
    "Label Smoothing": {
        "folder": NEW_EXP_ROOT / "label_smoothing",
        "pattern": "predictions_epoch_{epoch:02d}.npz",
    },
    "Resolution 320": {
        "folder": disease_exp_dir("disease_resolution_320"),
        "pattern": "validation_epoch_{epoch:02d}.npz",
    },
    "Weighted CE": {
        "folder": disease_exp_dir("disease_weighted_ce"),
        "pattern": "validation_epoch_{epoch:02d}.npz",
    },
}

SCREEN_EPOCHS = 5
CONTROL_NAME = "Original 5ep"

# ------------------------------------------------------------
# 2. Load same fixed Crop predictions
# ------------------------------------------------------------

FIXED_CROP_FILE = (
    run_dir("higher_decay", "crop")
    / "best_validation.npz"
)

assert FIXED_CROP_FILE.exists(), FIXED_CROP_FILE

with np.load(FIXED_CROP_FILE) as data:
    fixed_crop_logs = {
        "seen": data["seen"].copy(),
        "unseen": data["unseen"].copy(),
    }

crop_array = np.asarray(crop_ids)
disease_array = np.asarray(disease_ids)

for split, table in [
    ("seen", val_seen_df),
    ("unseen", val_unseen_df),
]:
    assert fixed_crop_logs[split].shape == (
        len(table), len(crop_ids)
    )

# ------------------------------------------------------------
# 3. Recalculate metrics from saved NPZ predictions
# ------------------------------------------------------------

def unified_evaluate(prediction_file):

    results = {}

    with np.load(prediction_file) as data:

        for split, table in [
            ("seen", val_seen_df),
            ("unseen", val_unseen_df),
        ]:

            crop_log = fixed_crop_logs[split]
            disease_log = data[split]

            assert disease_log.shape == (
                len(table), len(disease_ids)
            ), f"Shape mismatch: {prediction_file}"

            crop_pred = crop_array[
                crop_log.argmax(axis=1)
            ]

            disease_pred = disease_array[
                disease_log.argmax(axis=1)
            ]

            true_crop = table[
                "crop_id"
            ].to_numpy(dtype=int)

            true_disease = table[
                "disease_id"
            ].to_numpy(dtype=int)

            crop_correct = (
                crop_pred == true_crop
            )

            disease_correct = (
                disease_pred == true_disease
            )

            composition_correct = (
                crop_correct & disease_correct
            )

            # Equivalent to Top-1 argmax over all
            # 55 * 175 = 9,625 candidate pairs.
            results[split] = {
                "crop": float(crop_correct.mean()),
                "disease": float(disease_correct.mean()),
                "composition": float(
                    composition_correct.mean()
                ),
            }

    results["disease_hm"] = harmonic(
        results["seen"]["disease"],
        results["unseen"]["disease"],
    )

    results["composition_hm"] = harmonic(
        results["seen"]["composition"],
        results["unseen"]["composition"],
    )

    return results


# ------------------------------------------------------------
# 4. Evaluate Epochs 1–5 for ALL experiments
# ------------------------------------------------------------

epoch_rows = []

for experiment, spec in ALL_EXPERIMENTS.items():

    folder = spec["folder"]

    assert folder.exists(), (
        f"Missing experiment folder: {folder}"
    )

    print(f"Checking: {experiment}")

    for epoch in range(1, SCREEN_EPOCHS + 1):

        prediction_file = (
            folder
            / spec["pattern"].format(epoch=epoch)
        )

        assert prediction_file.exists(), (
            f"Missing prediction: {prediction_file}"
        )

        metrics = unified_evaluate(
            prediction_file
        )

        epoch_rows.append({
            "Experiment": experiment,
            "Epoch": epoch,
            "Disease Seen (%)":
                metrics["seen"]["disease"] * 100,
            "Disease Unseen (%)":
                metrics["unseen"]["disease"] * 100,
            "Disease HM (%)":
                metrics["disease_hm"] * 100,
            "Composition Seen (%)":
                metrics["seen"]["composition"] * 100,
            "Composition Unseen (%)":
                metrics["unseen"]["composition"] * 100,
            "Composition HM (%)":
                metrics["composition_hm"] * 100,
        })

all_epoch_results = pd.DataFrame(epoch_rows)

assert len(all_epoch_results) == (
    len(ALL_EXPERIMENTS) * SCREEN_EPOCHS
)

# ------------------------------------------------------------
# 5. Select checkpoint by highest Disease HM
#    within Epochs 1–5; earliest epoch breaks ties
# ------------------------------------------------------------

best_rows = []

for experiment in ALL_EXPERIMENTS:

    subset = all_epoch_results[
        all_epoch_results["Experiment"] == experiment
    ]

    best = subset.sort_values(
        ["Disease HM (%)", "Epoch"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]

    best_rows.append(best)

best_within_5 = pd.DataFrame(best_rows)

# ------------------------------------------------------------
# 6. Exact Epoch 5
# ------------------------------------------------------------

exact_epoch_5 = all_epoch_results[
    all_epoch_results["Epoch"] == 5
].copy()

# ------------------------------------------------------------
# 7. Difference from original 5-epoch control
# ------------------------------------------------------------

def add_baseline_deltas(table):

    result = table.copy()

    control = result.loc[
        result["Experiment"] == CONTROL_NAME
    ]

    assert len(control) == 1

    baseline = control.iloc[0]

    result["Disease HM Delta (pp)"] = (
        result["Disease HM (%)"]
        - baseline["Disease HM (%)"]
    )

    result["Composition HM Delta (pp)"] = (
        result["Composition HM (%)"]
        - baseline["Composition HM (%)"]
    )

    return result.sort_values(
        "Composition HM (%)",
        ascending=False
    ).reset_index(drop=True)


best_within_5 = add_baseline_deltas(
    best_within_5
)

exact_epoch_5 = add_baseline_deltas(
    exact_epoch_5
)

# ------------------------------------------------------------
# 8. Display
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("UNIFIED DEVELOPMENT COMPARISON — 8 EXPERIMENTS")
print("=" * 75)

print("Fixed Crop: Higher WD Stage 1 Best")
print("Disease Selection: Highest Development Disease HM")
print("Epoch Budget: 5")
print("Seed: 42 (different experiment batches)")
print("Candidate Pairs: 9,625")
print("Evaluation: DEVELOPMENT ONLY — NO OFFICIAL TEST")

print("\nA. BEST CHECKPOINT WITHIN EPOCHS 1–5")
display(best_within_5.round(2))

print("\nB. EXACT EPOCH 5")
display(exact_epoch_5.round(2))

# ------------------------------------------------------------
# 9. Save without overwriting earlier comparisons
# ------------------------------------------------------------

UNIFIED_DIR = (
    RESULTS_ROOT
    / "unified_disease_screening_5epoch"
)

UNIFIED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

all_epoch_results.to_csv(
    UNIFIED_DIR / "all_epoch_metrics.csv",
    index=False
)

best_within_5.to_csv(
    UNIFIED_DIR / "best_disease_hm_within_5.csv",
    index=False
)

exact_epoch_5.to_csv(
    UNIFIED_DIR / "exact_epoch_5.csv",
    index=False
)

print("\nComparison saved to:", UNIFIED_DIR)


In [ ]:

# ============================================================
# STAGE: EXTEND SELECTED EXPERIMENTS TO 15 EPOCHS
# Resume from Epoch 5 — DO NOT RESTART
# ============================================================

import copy
import json
import gc
import shutil
import pandas as pd
import torch
from pathlib import Path

EXTEND_TO_EPOCH = 15

EXTEND_ARMS = [
    "projection_wider",
    "projection_supcon",
]


def extend_new_arm(arm, target_epoch=15):

    assert arm in EXTEND_ARMS

    folder = NEW_EXP_ROOT / arm

    config_path = folder / "config.json"
    resume_path = folder / "resume_state.pt"
    history_path = folder / "history.csv"

    assert config_path.exists(), config_path
    assert resume_path.exists(), resume_path
    assert history_path.exists(), history_path

    saved_config = json.loads(
        config_path.read_text(encoding="utf-8")
    )

    # The original 5-epoch experiment must remain unchanged
    assert saved_config["arm"] == arm
    assert saved_config["epochs"] == 5
    assert saved_config["scheduler_T_max"] == 30
    assert saved_config["seed"] == SEED
    assert saved_config["spec"] == NEW_ARMS[arm]
    assert saved_config["base_config"] == BASE_CFG
    assert saved_config["fixed_crop_predictions"] == str(
        NEW_CROP_NPZ
    )

    # Load complete training state, not just model weights
    state = torch.load(
        resume_path,
        map_location="cpu",
        weights_only=False
    )

    assert state["config"] == saved_config

    completed_epoch = int(state["epoch"])

    assert 5 <= completed_epoch <= target_epoch, (
        f"Unexpected saved epoch: {completed_epoch}"
    )

    assert len(state["history"]) == completed_epoch
    assert all(
        int(row["epoch"]) == i
        for i, row in enumerate(
            state["history"], start=1
        )
    )

    # Confirm already saved checkpoints are present
    for epoch in range(1, completed_epoch + 1):

        assert (
            folder / f"epoch_{epoch:02d}.pt"
        ).exists()

        assert (
            folder / f"predictions_epoch_{epoch:02d}.npz"
        ).exists()

    print("\n" + "=" * 65)
    print("RESUME:", arm)
    print("Completed Epoch:", completed_epoch)
    print("Target Epoch:", target_epoch)
    print("Scheduler Horizon:", saved_config["scheduler_T_max"])
    print("=" * 65)

    # Preserve the original Epoch 5 resume checkpoint
    backup = folder / "resume_state_epoch05.pt"

    if completed_epoch == 5 and not backup.exists():
        shutil.copy2(resume_path, backup)
        print("Original Epoch 5 resume state backed up.")

    # Reconstruct original architecture
    seed_all(SEED)

    model = DiseaseProjectionResNet(
        NEW_ARMS[arm]["head"]
    ).to(DEVICE)

    optimizer = new_optimizer(model)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NEW_SCHEDULER_HORIZON
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP
    )

    # Restore weights, optimizer, scheduler and AMP
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])

    # Restore exact stored random-number-generator states
    restore_rng(state["rng"])

    history = copy.deepcopy(state["history"])

    start_epoch = completed_epoch + 1

    print("Continuing from Epoch:", start_epoch)

    try:

        for epoch in range(
            start_epoch,
            target_epoch + 1
        ):

            # Keep NEW_EPOCHS unchanged globally (=5).
            # Temporarily use 15 for graph ticks and progress.
            old_new_epochs = globals()["NEW_EPOCHS"]
            globals()["NEW_EPOCHS"] = target_epoch

            try:

                loader = new_train_loader(
                    epoch, arm
                )

                train_stats = new_train_epoch(
                    model,
                    loader,
                    arm,
                    optimizer,
                    scaler,
                    epoch
                )

                seen_stats, seen_log = new_disease_evaluate(
                    model, seen_loader
                )

                unseen_stats, unseen_log = new_disease_evaluate(
                    model, unseen_loader
                )

                scores = new_metrics(
                    seen_log, unseen_log
                )

                row = {
                    "epoch": epoch,
                    **train_stats,
                    "seen_loss": seen_stats["loss"],
                    "unseen_loss": unseen_stats["loss"],
                    **scores
                }

                history.append(row)

                # -----------------------------------------
                # Save validation predictions every epoch
                # -----------------------------------------

                prediction_path = (
                    folder
                    / f"predictions_epoch_{epoch:02d}.npz"
                )

                np.savez_compressed(
                    prediction_path,
                    seen=seen_log,
                    unseen=unseen_log
                )

                # -----------------------------------------
                # Save full model checkpoint every epoch
                # -----------------------------------------

                checkpoint = {
                    "arm": arm,
                    "epoch": epoch,
                    "state_dict": {
                        key: value.detach().cpu().clone()
                        for key, value
                        in model.state_dict().items()
                    },
                    "head": NEW_ARMS[arm]["head"],
                    "config": saved_config,
                    "crop_ids": crop_ids,
                    "disease_ids": disease_ids,
                    "validation_scores": scores,
                    "train_accuracy": train_stats["train_acc"]
                }

                epoch_path = (
                    folder / f"epoch_{epoch:02d}.pt"
                )

                atomic_torch_save(
                    checkpoint,
                    epoch_path
                )

                # Continue original scheduler state
                scheduler.step()

                # -----------------------------------------
                # Save resume state (optimizer + RNG)
                # -----------------------------------------

                atomic_torch_save(
                    {
                        "config": saved_config,
                        "epoch": epoch,
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(),
                        "scaler": scaler.state_dict(),
                        "history": copy.deepcopy(history),
                        "rng": capture_rng()
                    },
                    resume_path
                )

                pd.DataFrame(history).to_csv(
                    history_path,
                    index=False
                )

                # Update graphs after every epoch
                new_plot(
                    history,
                    arm,
                    folder
                )

                print(
                    f"\n{arm} | Epoch {epoch:02d}/{target_epoch}"
                )

                print(
                    "Train Accuracy:",
                    f"{train_stats['train_acc']:.2%}"
                )

                print(
                    "Disease Seen:",
                    f"{scores['seen_disease']:.2%}"
                )

                print(
                    "Disease Unseen:",
                    f"{scores['unseen_disease']:.2%}"
                )

                print(
                    "Disease HM:",
                    f"{scores['disease_hm']:.2%}"
                )

                print(
                    "Composition Seen:",
                    f"{scores['seen_composition']:.2%}"
                )

                print(
                    "Composition Unseen:",
                    f"{scores['unseen_composition']:.2%}"
                )

                print(
                    "Composition HM:",
                    f"{scores['composition_hm']:.2%}"
                )

                del loader

            finally:
                globals()["NEW_EPOCHS"] = old_new_epochs

        # ---------------------------------------------
        # Verify all 15 epochs
        # ---------------------------------------------

        assert len(history) == target_epoch

        for epoch in range(1, target_epoch + 1):

            assert (
                folder / f"epoch_{epoch:02d}.pt"
            ).exists()

            assert (
                folder / f"predictions_epoch_{epoch:02d}.npz"
            ).exists()

        # Select by highest individual Disease HM
        history_df = pd.DataFrame(history)

        best_row = history_df.sort_values(
            ["disease_hm", "epoch"],
            ascending=[False, True]
        ).iloc[0]

        best_epoch = int(best_row["epoch"])

        # Copy selected checkpoint
        shutil.copy2(
            folder / f"epoch_{best_epoch:02d}.pt",
            folder / "best_disease_hm_15ep.pt"
        )

        # Save selected checkpoint metadata
        save_json(
            {
                "status": "COMPLETE",
                "arm": arm,
                "total_epochs": target_epoch,
                "selection_metric":
                    "highest_development_disease_hm",
                "best_epoch": best_epoch,
                "best_disease_hm": float(
                    best_row["disease_hm"]
                ),
                "corresponding_composition_hm": float(
                    best_row["composition_hm"]
                )
            },
            folder / "EXTENDED_COMPLETE_15.json"
        )

        print("\n" + "=" * 65)
        print("15-EPOCH TRAINING COMPLETED:", arm)
        print("Best Disease HM Epoch:", best_epoch)
        print(
            "Best Disease HM:",
            f'{best_row["disease_hm"]:.2%}'
        )
        print(
            "Corresponding Composition HM:",
            f'{best_row["composition_hm"]:.2%}'
        )
        print("=" * 65)

        return history_df

    finally:

        del model, optimizer, scheduler, scaler
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("15-Epoch Resume Function Ready.")
print("Original five-epoch checkpoints will be retained.")


In [ ]:

# ============================================================
# PROJECTION WIDER — RESUME EPOCH 6 TO 15
# ============================================================

projection_wider_15_history = extend_new_arm(
    "projection_wider",
    target_epoch=15
)

display(
    projection_wider_15_history[
        [
            "epoch",
            "train_acc",
            "seen_disease",
            "unseen_disease",
            "disease_hm",
            "seen_composition",
            "unseen_composition",
            "composition_hm"
        ]
    ].round(4)
)


In [ ]:

# ============================================================
# PROJECTION + SUPCON — RESUME EPOCH 6 TO 15
# ============================================================

projection_supcon_15_history = extend_new_arm(
    "projection_supcon",
    target_epoch=15
)

display(
    projection_supcon_15_history[
        [
            "epoch",
            "train_acc",
            "seen_disease",
            "unseen_disease",
            "disease_hm",
            "seen_composition",
            "unseen_composition",
            "composition_hm"
        ]
    ].round(4)
)


In [ ]:

# ============================================================
# COMPARE PROJECTION WIDER VS SUPCON — FIRST 15 EPOCHS
# Read existing 20-epoch history without modifying it
# ============================================================

import pandas as pd

rows = []

for arm in ["projection_wider", "projection_supcon"]:

    folder = NEW_EXP_ROOT / arm

    assert (
        folder / "EXTENDED_COMPLETE_15.json"
    ).exists(), f"{arm} 15-epoch training not completed"

    history = pd.read_csv(folder / "history.csv")

    # Select ONLY the first 15 epochs
    history_15 = history[
        history["epoch"].between(1, 15)
    ].copy()

    assert len(history_15) == 15
    assert history_15["epoch"].tolist() == list(range(1, 16))

    # Select checkpoint using highest Disease HM
    best = history_15.sort_values(
        ["disease_hm", "epoch"],
        ascending=[False, True]
    ).iloc[0]

    epoch15 = history_15.loc[
        history_15["epoch"] == 15
    ].iloc[0]

    rows.append({
        "Experiment": arm,
        "Selected Epoch": int(best["epoch"]),

        "Disease Seen (%)": best["seen_disease"] * 100,
        "Disease Unseen (%)": best["unseen_disease"] * 100,
        "Disease HM (%)": best["disease_hm"] * 100,

        "Composition Seen (%)": best["seen_composition"] * 100,
        "Composition Unseen (%)": best["unseen_composition"] * 100,
        "Composition HM (%)": best["composition_hm"] * 100,

        "Epoch 15 Disease HM (%)": epoch15["disease_hm"] * 100,
        "Epoch 15 Composition HM (%)": epoch15["composition_hm"] * 100,
    })

comparison_15 = pd.DataFrame(rows).sort_values(
    "Composition HM (%)",
    ascending=False
)

print("=" * 65)
print("15-EPOCH DEVELOPMENT COMPARISON")
print("Selection: Highest Individual Disease HM")
print("Crop: Fixed Higher WD Stage 1 Best Crop")
print("=" * 65)

display(comparison_15.round(2))

# Save comparison without changing training history
comparison_15.to_csv(
    NEW_EXP_ROOT / "comparison_15epoch.csv",
    index=False
)

print("\nHistorical Original Higher WD Reference:")
print("Disease best epoch: 21")
print("Composition HM: 11.93%")
print("(30-epoch training budget; not a matched 15-epoch control)")


In [ ]:

# ============================================================
# CHECK CONVERGENCE / OVERFITTING — EPOCHS 1 TO 15
# Projection Wider vs Projection + SupCon
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CHECK_ARMS = ["projection_wider", "projection_supcon"]

summary_rows = []

for arm in CHECK_ARMS:

    path = NEW_EXP_ROOT / arm / "history.csv"
    assert path.exists(), f"Missing: {path}"

    df = pd.read_csv(path).sort_values("epoch")
    df = df[df["epoch"] <= 15].copy()

    assert len(df) == 15

    # Best checkpoint based on individual Disease HM
    best = df.sort_values(
        ["disease_hm", "epoch"],
        ascending=[False, True]
    ).iloc[0]

    # Last 5 epochs: 11-15
    recent = df.tail(5)
    early_recent = df.iloc[-10:-5]

    disease_change = (
        recent["disease_hm"].mean()
        - early_recent["disease_hm"].mean()
    ) * 100

    composition_change = (
        recent["composition_hm"].mean()
        - early_recent["composition_hm"].mean()
    ) * 100

    disease_slope = np.polyfit(
        recent["epoch"],
        recent["disease_hm"] * 100,
        1
    )[0]

    composition_slope = np.polyfit(
        recent["epoch"],
        recent["composition_hm"] * 100,
        1
    )[0]

    latest = df.iloc[-1]

    summary_rows.append({
        "Experiment": arm,
        "Best Disease HM Epoch": int(best["epoch"]),
        "Latest Disease HM (%)":
            latest["disease_hm"] * 100,
        "Latest Composition HM (%)":
            latest["composition_hm"] * 100,
        "Disease HM Mean Change (pp)":
            disease_change,
        "Composition HM Mean Change (pp)":
            composition_change,
        "Disease HM Recent Slope (pp/epoch)":
            disease_slope,
        "Composition HM Recent Slope (pp/epoch)":
            composition_slope,
        "Best at Last Epoch?":
            int(best["epoch"]) == 15
    })

    # --------------------------------------------------------
    # Display recent epoch metrics
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print(arm.upper())
    print("=" * 65)

    recent_columns = [
        "epoch",
        "train_loss",
        "train_acc",
        "seen_loss",
        "unseen_loss",
        "seen_disease",
        "unseen_disease",
        "disease_hm",
        "seen_composition",
        "unseen_composition",
        "composition_hm"
    ]

    display(df.tail(5)[recent_columns].round(4))

    # --------------------------------------------------------
    # Plot curves
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        2, 2, figsize=(14, 9)
    )

    axes[0, 0].plot(
        df["epoch"], df["train_loss"],
        marker="o", label="Training Loss"
    )
    axes[0, 0].plot(
        df["epoch"], df["seen_loss"],
        marker="o", label="Seen Validation Loss"
    )
    axes[0, 0].plot(
        df["epoch"], df["unseen_loss"],
        marker="o", label="Unseen Validation Loss"
    )
    axes[0, 0].set_title("Loss Curves")

    axes[0, 1].plot(
        df["epoch"], df["train_acc"] * 100,
        marker="o", label="Training Accuracy"
    )
    axes[0, 1].plot(
        df["epoch"], df["seen_disease"] * 100,
        marker="o", label="Disease Seen"
    )
    axes[0, 1].plot(
        df["epoch"], df["unseen_disease"] * 100,
        marker="o", label="Disease Unseen"
    )
    axes[0, 1].set_title("Accuracy (%)")

    axes[1, 0].plot(
        df["epoch"], df["disease_hm"] * 100,
        marker="o", label="Disease HM"
    )
    axes[1, 0].axvline(
        int(best["epoch"]),
        linestyle="--",
        label="Best Disease HM Epoch"
    )
    axes[1, 0].set_title("Disease HM (%)")

    axes[1, 1].plot(
        df["epoch"], df["composition_hm"] * 100,
        marker="o", label="Composition HM"
    )
    axes[1, 1].axvline(
        int(best["epoch"]),
        linestyle="--",
        label="Selected Disease Epoch"
    )
    axes[1, 1].set_title("Composition HM (%)")

    for ax in axes.flat:
        ax.set_xlabel("Epoch")
        ax.set_xticks(range(1, 16))
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"{arm} — Convergence Analysis",
        fontsize=14
    )
    fig.tight_layout()

    fig.savefig(
        NEW_EXP_ROOT / arm
        / "convergence_analysis_15ep.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

convergence_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 75)
print("CONVERGENCE ANALYSIS SUMMARY")
print("=" * 75)

display(convergence_summary.round(3))

convergence_summary.to_csv(
    NEW_EXP_ROOT / "convergence_summary_15ep.csv",
    index=False
)


In [ ]:

# ============================================================
# RESUME PROJECTION EXPERIMENTS TO EPOCH 20
# ============================================================

import shutil
import copy

def extend_new_arm(arm, target_epoch=20):

    folder = NEW_EXP_ROOT / arm
    resume_path = folder / "resume_state.pt"

    assert arm in ["projection_wider", "projection_supcon"]
    assert resume_path.exists(), resume_path

    config = json.loads(
        (folder / "config.json").read_text()
    )

    state = torch.load(
        resume_path, map_location="cpu",
        weights_only=False
    )

    assert state["config"] == config
    assert config["scheduler_T_max"] == 30

    start_epoch = int(state["epoch"]) + 1
    assert 16 <= start_epoch <= target_epoch + 1

    # Preserve the existing 15-epoch resume state
    backup = folder / "resume_state_epoch15.pt"
    if start_epoch == 16 and not backup.exists():
        shutil.copy2(resume_path, backup)

    # Reconstruct original model and optimizer
    seed_all(SEED)

    model = DiseaseProjectionResNet(
        NEW_ARMS[arm]["head"]
    ).to(DEVICE)

    optimizer = new_optimizer(model)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=30
    )

    scaler = torch.amp.GradScaler(
        "cuda", enabled=AMP
    )

    # Restore ALL training states
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    restore_rng(state["rng"])

    history = copy.deepcopy(state["history"])

    assert len(history) == start_epoch - 1

    print(f"{arm}: Epoch {start_epoch} → {target_epoch}")

    global NEW_EPOCHS
    original_epochs = NEW_EPOCHS
    NEW_EPOCHS = target_epoch

    try:
        for epoch in range(start_epoch, target_epoch + 1):

            loader = new_train_loader(epoch, arm)

            train = new_train_epoch(
                model, loader, arm,
                optimizer, scaler, epoch
            )

            seen, seen_log = new_disease_evaluate(
                model, seen_loader
            )

            unseen, unseen_log = new_disease_evaluate(
                model, unseen_loader
            )

            scores = new_metrics(seen_log, unseen_log)

            history.append({
                "epoch": epoch,
                **train,
                "seen_loss": seen["loss"],
                "unseen_loss": unseen["loss"],
                **scores
            })

            # Save validation predictions
            np.savez_compressed(
                folder / f"predictions_epoch_{epoch:02d}.npz",
                seen=seen_log,
                unseen=unseen_log
            )

            # Save EVERY epoch checkpoint
            atomic_torch_save({
                "arm": arm,
                "epoch": epoch,
                "state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                "head": NEW_ARMS[arm]["head"],
                "config": config,
                "crop_ids": crop_ids,
                "disease_ids": disease_ids,
                "validation_scores": scores,
                "train_accuracy": train["train_acc"]
            }, folder / f"epoch_{epoch:02d}.pt")

            scheduler.step()

            # Save full resume state
            atomic_torch_save({
                "config": config,
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "history": copy.deepcopy(history),
                "rng": capture_rng()
            }, resume_path)

            pd.DataFrame(history).to_csv(
                folder / "history.csv", index=False
            )

            new_plot(history, arm, folder)

            print(
                f"Epoch {epoch}: "
                f"Disease HM={scores['disease_hm']:.2%}, "
                f"Composition HM={scores['composition_hm']:.2%}"
            )

            del loader

        # Select by highest individual Disease HM
        df = pd.DataFrame(history)

        best = df.sort_values(
            ["disease_hm", "epoch"],
            ascending=[False, True]
        ).iloc[0]

        best_epoch = int(best["epoch"])

        shutil.copy2(
            folder / f"epoch_{best_epoch:02d}.pt",
            folder / f"best_disease_hm_{target_epoch}ep.pt"
        )

        save_json({
            "status": "COMPLETE",
            "best_epoch": best_epoch,
            "disease_hm": float(best["disease_hm"]),
            "composition_hm": float(best["composition_hm"])
        }, folder / f"EXTENDED_COMPLETE_{target_epoch}.json")

        print(f"\n{arm} completed!")
        print("Best Disease Epoch:", best_epoch)
        print("Disease HM:", f"{best['disease_hm']:.2%}")
        print("Composition HM:", f"{best['composition_hm']:.2%}")

        return df

    finally:
        NEW_EPOCHS = original_epochs
        del model, optimizer, scheduler, scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:

# ============================================================
# RUN BOTH EXPERIMENTS: EPOCH 16–20
# ============================================================

wider_20 = extend_new_arm(
    "projection_wider", target_epoch=20
)

supcon_20 = extend_new_arm(
    "projection_supcon", target_epoch=20
)


In [ ]:

# ============================================================
# COMPARE 20-EPOCH RESULTS
# ============================================================

rows = []

for arm in ["projection_wider", "projection_supcon"]:

    folder = NEW_EXP_ROOT / arm
    df = pd.read_csv(folder / "history.csv")

    assert len(df) == 20

    best = df.sort_values(
        ["disease_hm", "epoch"],
        ascending=[False, True]
    ).iloc[0]

    rows.append({
        "Experiment": arm,
        "Best Epoch": int(best["epoch"]),
        "Disease Seen (%)": best["seen_disease"] * 100,
        "Disease Unseen (%)": best["unseen_disease"] * 100,
        "Disease HM (%)": best["disease_hm"] * 100,
        "Composition Seen (%)": best["seen_composition"] * 100,
        "Composition Unseen (%)": best["unseen_composition"] * 100,
        "Composition HM (%)": best["composition_hm"] * 100
    })

comparison_20 = pd.DataFrame(rows).sort_values(
    "Composition HM (%)", ascending=False
)

display(comparison_20.round(2))

comparison_20.to_csv(
    NEW_EXP_ROOT / "comparison_20epoch.csv",
    index=False
)


In [ ]:

# ============================================================
# IMPROVED BASELINE 1 — STAGE 2 CONFIGURATION
# Fixed Original Crop + SupCon Disease
# ============================================================

import copy, json, gc, shutil, random, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm.auto import tqdm
from PIL import Image

IMPROVED_ROOT = (
    RESULTS_ROOT / "stage2_improved_supcon_disease"
)

DISEASE_STAGE2_DIR = IMPROVED_ROOT / "disease"
IMPROVED_OFFICIAL_DIR = IMPROVED_ROOT / "official_test"

SUPCON_SOURCE = (
    NEW_EXP_ROOT / "projection_supcon" / "epoch_15.pt"
)

FIXED_CROP_STAGE2 = (
    STAGE2_ROOT / "crop" / "best_train_acc.pt"
)

FT_EPOCHS = 10
FT_BACKBONE_LR = 1e-6
FT_HEAD_LR = 1e-4
FT_WEIGHT_DECAY = 1e-3

assert SUPCON_SOURCE.exists(), SUPCON_SOURCE
assert FIXED_CROP_STAGE2.exists(), FIXED_CROP_STAGE2
assert len(full_train_df) == 8000

# Verify Stage 1 selected checkpoint
source = torch.load(
    SUPCON_SOURCE,
    map_location="cpu",
    weights_only=False
)

assert source["arm"] == "projection_supcon"
assert source["head"] == "wider"
assert source["epoch"] == 15
assert list(source["crop_ids"]) == list(crop_ids)
assert list(source["disease_ids"]) == list(disease_ids)

# Verify highest Stage 1 Disease HM
supcon_history = pd.read_csv(
    NEW_EXP_ROOT / "projection_supcon" / "history.csv"
)

selected_stage1 = supcon_history.sort_values(
    ["disease_hm", "epoch"],
    ascending=[False, True]
).iloc[0]

assert int(selected_stage1["epoch"]) == 15

# Verify existing Crop Stage 2
crop_ckpt = torch.load(
    FIXED_CROP_STAGE2,
    map_location="cpu",
    weights_only=False
)

assert crop_ckpt["task"] == "crop"
assert crop_ckpt["stage"] == 2
assert list(crop_ckpt["crop_ids"]) == list(crop_ids)
assert list(crop_ckpt["disease_ids"]) == list(disease_ids)

crop_epoch = int(crop_ckpt["epoch"])

del source, crop_ckpt

# Freeze experiment configuration
FT_CONFIG = {
    "model": "ResNet50_Projection_Wider",
    "source": str(SUPCON_SOURCE),
    "source_epoch": 15,
    "source_arm": "projection_supcon",
    "training_images": 8000,
    "additional_epochs": FT_EPOCHS,
    "backbone_lr": FT_BACKBONE_LR,
    "head_lr": FT_HEAD_LR,
    "weight_decay": FT_WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "loss": "CrossEntropy_only",
    "scheduler": "CosineAnnealingLR_Tmax10",
    "checkpoint_selection": "highest_training_top1",
    "official_test_for_selection": False
}

DISEASE_STAGE2_DIR.mkdir(
    parents=True, exist_ok=True
)

config_file = DISEASE_STAGE2_DIR / "config.json"

if config_file.exists():
    assert json.loads(config_file.read_text()) == FT_CONFIG
else:
    assert not any(DISEASE_STAGE2_DIR.iterdir())
    save_json(FT_CONFIG, config_file)

print("PREFLIGHT PASSED")
print("Source Disease Epoch:", 15)
print("Fixed Crop Stage 2 Epoch:", crop_epoch)
print("Training Images:", len(full_train_df))
print("Additional Epochs:", FT_EPOCHS)
print("Disease Stage 2 Loss: CE-only")
print("Output:", DISEASE_STAGE2_DIR)


In [ ]:

# ============================================================
# IMPROVED STAGE 2 — MODEL + TRAINING FUNCTIONS
# ============================================================

class ImprovedDiseaseCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = models.resnet50(weights=None)

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.projection = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128)
        )

        self.classifier = nn.Linear(
            128, len(disease_ids)
        )

    def forward(self, x):
        features = self.backbone(x)
        projected = self.projection(features)
        return self.classifier(projected)


def improved_worker_seed(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    random.seed(seed)
    np.random.seed(seed)


def improved_train_loader(epoch):

    generator = torch.Generator().manual_seed(
        SEED + epoch
    )

    return DataLoader(
        PlantDataset(full_train_df, train_transform),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=False,
        worker_init_fn=improved_worker_seed,
        generator=generator
    )


def plot_improved_history(history):

    df = pd.DataFrame(history)
    epoch = int(df["epoch"].iloc[-1])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(
        df["epoch"], df["train_loss"], marker="o"
    )
    axes[0].set_title("Stage 2 Training Loss")
    axes[0].set_ylabel("Cross-Entropy Loss")

    axes[1].plot(
        df["epoch"],
        df["train_acc"] * 100,
        marker="o"
    )
    axes[1].set_title("Stage 2 Training Accuracy")
    axes[1].set_ylabel("Top-1 Accuracy (%)")
    axes[1].set_ylim(0, 100)

    for ax in axes:
        ax.set_xlabel("Stage 2 Epoch")
        ax.set_xticks(range(1, FT_EPOCHS + 1))
        ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        DISEASE_STAGE2_DIR
        / f"training_curves_epoch_{epoch:02d}.png",
        dpi=160, bbox_inches="tight"
    )

    fig.savefig(
        DISEASE_STAGE2_DIR / "training_curves.png",
        dpi=160, bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


def train_improved_stage2():

    folder = DISEASE_STAGE2_DIR
    resume_path = folder / "resume_state.pt"

    seed_all(SEED)

    model = ImprovedDiseaseCNN().to(DEVICE)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.backbone.parameters(),
                "lr": FT_BACKBONE_LR
            },
            {
                "params": list(model.projection.parameters())
                          + list(model.classifier.parameters()),
                "lr": FT_HEAD_LR
            }
        ],
        weight_decay=FT_WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=FT_EPOCHS
    )

    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    history = []
    best_acc = -1.0
    best_epoch = 0
    start_epoch = 1

    if resume_path.exists():

        state = torch.load(
            resume_path,
            map_location="cpu",
            weights_only=False
        )

        assert state["config"] == FT_CONFIG

        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])

        history = state["history"]
        best_acc = float(state["best_acc"])
        best_epoch = int(state["best_epoch"])
        start_epoch = int(state["epoch"]) + 1

        restore_rng(state["rng"])

        print("Resuming Stage 2 Epoch:", start_epoch)

    else:

        ckpt = torch.load(
            SUPCON_SOURCE,
            map_location="cpu",
            weights_only=False
        )

        # Exact compatibility with Stage 1 SupCon
        model.load_state_dict(
            ckpt["state_dict"], strict=True
        )

        print("Loaded SupCon Stage 1 Epoch 15.")

        del ckpt

    try:
        for epoch in range(
            start_epoch, FT_EPOCHS + 1
        ):

            model.train()

            total = 0
            correct = 0
            loss_sum = 0.0

            loader = improved_train_loader(epoch)

            for images, _, labels in tqdm(
                loader,
                desc=f"SupCon Disease Stage 2 {epoch}/10"
            ):

                images = images.to(
                    DEVICE, non_blocking=True
                )

                labels = labels.to(
                    DEVICE, non_blocking=True
                )

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    "cuda", enabled=AMP
                ):
                    logits = model(images)
                    loss = F.cross_entropy(logits, labels)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                n = len(labels)
                total += n
                loss_sum += float(loss.item()) * n

                correct += int(
                    (logits.detach().argmax(1) == labels)
                    .sum().item()
                )

            scheduler.step()

            train_acc = correct / total
            train_loss = loss_sum / total

            history.append({
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc
            })

            checkpoint = {
                "stage": 2,
                "task": "disease",
                "architecture": "resnet50_projection_wider",
                "source_stage1_epoch": 15,
                "epoch": epoch,
                "train_acc": train_acc,
                "train_loss": train_loss,
                "config": FT_CONFIG,
                "crop_ids": crop_ids,
                "disease_ids": disease_ids,
                "state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
            }

            epoch_path = (
                folder / f"epoch_{epoch:02d}.pt"
            )

            # Save EVERY epoch
            atomic_torch_save(checkpoint, epoch_path)

            if train_acc > best_acc + 1e-12:
                best_acc = train_acc
                best_epoch = epoch

                shutil.copy2(
                    epoch_path,
                    folder / "best_train_acc.pt"
                )

            # Save full resume state
            atomic_torch_save(
                {
                    "config": FT_CONFIG,
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(),
                    "history": copy.deepcopy(history),
                    "best_acc": best_acc,
                    "best_epoch": best_epoch,
                    "rng": capture_rng()
                },
                resume_path
            )

            pd.DataFrame(history).to_csv(
                folder / "history.csv",
                index=False
            )

            plot_improved_history(history)

            print(
                f"Epoch {epoch}: "
                f"Loss={train_loss:.4f} | "
                f"Train Acc={train_acc:.2%} | "
                f"Best Epoch={best_epoch}"
            )

            del loader

        # Final verification
        assert len(history) == FT_EPOCHS

        for epoch in range(1, FT_EPOCHS + 1):
            assert (
                folder / f"epoch_{epoch:02d}.pt"
            ).exists()

        save_json(
            {
                "status": "COMPLETE",
                "epochs": FT_EPOCHS,
                "best_train_acc_epoch": best_epoch,
                "best_training_accuracy": best_acc
            },
            folder / "COMPLETE.json"
        )

        print("\nSTAGE 2 COMPLETED")
        print("Best Training Epoch:", best_epoch)
        print("Best Training Accuracy:", f"{best_acc:.2%}")

    finally:
        del model, optimizer, scheduler, scaler
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("Improved Stage 2 training functions ready.")


In [ ]:

# ============================================================
# RUN IMPROVED DISEASE FULL-DATA FINE-TUNING
# ============================================================

train_improved_stage2()

history = pd.read_csv(
    DISEASE_STAGE2_DIR / "history.csv"
)

display(history.round(4))

best_row = history.sort_values(
    ["train_acc", "epoch"],
    ascending=[False, True]
).iloc[0]

print("\nSelected Disease Stage 2 Checkpoint")
print("Epoch:", int(best_row["epoch"]))
print("Training Accuracy:", f'{best_row["train_acc"]:.2%}')
print(
    "Checkpoint:",
    DISEASE_STAGE2_DIR / "best_train_acc.pt"
)


In [ ]:

# ============================================================
# IMPROVED BASELINE 1 — OFFICIAL TEST SETUP
# Original Stage 2 Crop + SupCon Stage 2 Disease
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import gc

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm

IMP_TEST_ROOT = (
    RESULTS_ROOT
    / "stage2_improved_supcon_disease"
    / "official_test"
)

IMP_CROP_PATH = (
    STAGE2_ROOT / "crop" / "best_train_acc.pt"
)

IMP_DISEASE_PATH = (
    RESULTS_ROOT
    / "stage2_improved_supcon_disease"
    / "disease" / "best_train_acc.pt"
)

IMP_TEST_ROOT.mkdir(parents=True, exist_ok=True)

assert IMP_CROP_PATH.exists(), IMP_CROP_PATH
assert IMP_DISEASE_PATH.exists(), IMP_DISEASE_PATH

# Load fixed Original Stage 2 Crop CNN
crop_ckpt = torch.load(
    IMP_CROP_PATH, map_location="cpu", weights_only=False
)

imp_crop = models.resnet50(weights=None)
imp_crop.fc = nn.Sequential(
    nn.Dropout(crop_ckpt["config"]["dropout"]),
    nn.Linear(imp_crop.fc.in_features, len(crop_ids))
)

imp_crop.load_state_dict(crop_ckpt["state_dict"], strict=True)
imp_crop = imp_crop.to(DEVICE).eval()

# Load improved SupCon Stage 2 Disease CNN
disease_ckpt = torch.load(
    IMP_DISEASE_PATH, map_location="cpu", weights_only=False
)

imp_disease = ImprovedDiseaseCNN()
imp_disease.load_state_dict(
    disease_ckpt["state_dict"], strict=True
)
imp_disease = imp_disease.to(DEVICE).eval()

assert crop_ckpt["task"] == "crop"
assert disease_ckpt["task"] == "disease"
assert disease_ckpt["stage"] == 2

print("Models loaded successfully!")
print("Crop Stage 2 Epoch:", crop_ckpt["epoch"])
print("Disease Stage 2 Epoch:", disease_ckpt["epoch"])
print("Output Folder:", IMP_TEST_ROOT)


In [ ]:

# ============================================================
# IMPROVED OFFICIAL TEST — INFERENCE
# ============================================================

TEST_COLUMNS = [
    "filename", "crop_id", "disease_id",
    "composition_id", "crop_name",
    "disease_name", "composition_name"
]

def imp_read_test(filename):
    path = find_case_insensitive(DRIVE_PROJECT, filename)
    first = path.open(encoding="utf-8-sig").readline()
    header = 0 if first.lower().startswith("filename,") else None

    df = pd.read_csv(
        path, names=TEST_COLUMNS,
        header=header, encoding="utf-8-sig"
    )

    for col in ["crop_id", "disease_id", "composition_id"]:
        df[col] = df[col].astype(int)

    return df.reset_index(drop=True)


class ImpTestDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = str(self.df.iloc[idx]["filename"])

        with Image.open(image_lookup[filename]) as img:
            return validation_transform(img.convert("RGB"))


@torch.inference_mode()
def imp_evaluate(split, df):

    loader = DataLoader(
        ImpTestDataset(df),
        batch_size=32,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    crop_outputs, disease_outputs = [], []


    for images in tqdm(loader, desc=f"Official {split}"):

        images = images.to(DEVICE, non_blocking=True)

        # Match Original Official Test inference precision
        with torch.autocast("cuda", enabled=AMP):
            crop_logits = imp_crop(images)
            disease_logits = imp_disease(images)

        crop_outputs.append(
            F.log_softmax(crop_logits.float(), dim=1)
            .cpu().numpy()
        )

        disease_outputs.append(
            F.log_softmax(disease_logits.float(), dim=1)
            .cpu().numpy()
        )


    c = np.concatenate(crop_outputs)
    d = np.concatenate(disease_outputs)

    # All 55 x 175 = 9625 candidate compositions
    composition = (
        c[:, :, None] + d[:, None, :]
    ).reshape(len(df), -1)

    true_c = df["crop_id"].map(
        {int(v): i for i, v in enumerate(crop_ids)}
    ).to_numpy()

    true_d = df["disease_id"].map(
        {int(v): i for i, v in enumerate(disease_ids)}
    ).to_numpy()

    assert not pd.isna(true_c).any()
    assert not pd.isna(true_d).any()

    true_c = true_c.astype(int)
    true_d = true_d.astype(int)
    true_pair = true_c * len(disease_ids) + true_d

    def scores(log_probs, truth):
        top1 = log_probs.argmax(axis=1)
        top5 = np.argpartition(
            log_probs, -5, axis=1
        )[:, -5:]

        return (
            float(np.mean(top1 == truth)),
            float(np.mean(
                (top5 == truth[:, None]).any(axis=1)
            ))
        )

    results = {}

    for name, logits, truth in [
        ("crop", c, true_c),
        ("disease", d, true_d),
        ("composition", composition, true_pair)
    ]:
        results[name + "_top1"], results[name + "_top5"] = (
            scores(logits, truth)
        )

    pred_c = c.argmax(axis=1)
    pred_d = d.argmax(axis=1)
    pred_pair = composition.argmax(axis=1)

    df = df.copy()
    df["pred_crop_id"] = np.asarray(crop_ids)[pred_c]
    df["pred_disease_id"] = np.asarray(disease_ids)[pred_d]
    df["pred_composition_crop_id"] = np.asarray(crop_ids)[
        pred_pair // len(disease_ids)
    ]
    df["pred_composition_disease_id"] = np.asarray(disease_ids)[
        pred_pair % len(disease_ids)
    ]

    df["crop_correct"] = pred_c == true_c
    df["disease_correct"] = pred_d == true_d
    df["composition_correct"] = pred_pair == true_pair

    assert np.array_equal(
        df["composition_correct"].to_numpy(),
        (
            df["crop_correct"] &
            df["disease_correct"]
        ).to_numpy()
    )

    df.to_csv(
        IMP_TEST_ROOT / f"{split}_predictions.csv",
        index=False
    )

    return results


imp_test_data = {
    "seen": imp_read_test("DPD_Small_test_seen.csv"),
    "unseen": imp_read_test("DPD_Small_test_unseen.csv")
}

assert len(imp_test_data["seen"]) == 2000
assert len(imp_test_data["unseen"]) == 1448

# Ensure official test images were not used for training
train_names = set(full_train_df["filename"].astype(str))
test_names = set(
    pd.concat(imp_test_data.values())["filename"].astype(str)
)

assert not train_names.intersection(test_names)
assert test_names.issubset(set(image_lookup))

imp_results = {}

for split, df in imp_test_data.items():
    imp_results[split] = imp_evaluate(split, df)

with open(IMP_TEST_ROOT / "metrics.json", "w") as f:
    json.dump(imp_results, f, indent=2)

print("\nIMPROVED OFFICIAL TEST COMPLETED")


In [ ]:

# ============================================================
# IMPROVED OFFICIAL RESULTS
# ============================================================

with open(IMP_TEST_ROOT / "metrics.json") as f:
    results = json.load(f)

def imp_hm(a, b):
    return 2 * a * b / (a + b) if a + b > 0 else 0

rows = []

for task in ["crop", "disease", "composition"]:
    for k in [1, 5]:

        metric = f"{task}_top{k}"

        seen = results["seen"][metric]
        unseen = results["unseen"][metric]

        rows.append({
            "Task": task.title(),
            "Top-K": k,
            "Seen (%)": seen * 100,
            "Unseen (%)": unseen * 100,
            "HM (%)": imp_hm(seen, unseen) * 100
        })

imp_table = pd.DataFrame(rows)

print("IMPROVED RESNET-50 OFFICIAL TEST RESULTS")
display(imp_table.round(2))

imp_table.to_csv(
    IMP_TEST_ROOT / "official_metrics.csv",
    index=False
)

# Original baseline reference
print("\nORIGINAL BASELINE — OFFICIAL TOP-1")
print("Crop: Seen 87.90%, Unseen 67.27%")
print("Disease: Seen 64.25%, Unseen 6.98%")
print("Composition: Seen 60.90%, Unseen 3.04%, HM 5.79%")

print("\nIMPROVED COMPOSITION TOP-1")
row = imp_table[
    (imp_table["Task"] == "Composition") &
    (imp_table["Top-K"] == 1)
].iloc[0]

display(pd.DataFrame([{
    "Model": "Original ResNet-50",
    "Seen (%)": 60.90,
    "Unseen (%)": 3.04,
    "HM (%)": 5.79
}, {
    "Model": "Improved SupCon Disease",
    "Seen (%)": row["Seen (%)"],
    "Unseen (%)": row["Unseen (%)"],
    "HM (%)": row["HM (%)"]
}]).round(2))

print("\nResults saved to:", IMP_TEST_ROOT)


In [ ]:
# ============================================================
# ORIGINAL HIGHER WD vs IMPROVED SUPCON — OFFICIAL TEST
# Read saved results only (NO retraining / NO inference)
# ============================================================

import json
import pandas as pd
from pathlib import Path

def hm(a, b):
    return 2 * a * b / (a + b) if a + b > 0 else 0.0

# Original Higher Weight Decay Official Test
original_file = (
    STAGE2_ROOT
    / "stage3_official_test"
    / "official_raw_metrics.json"
)

# Improved SupCon Official Test
improved_file = (
    RESULTS_ROOT
    / "stage2_improved_supcon_disease"
    / "official_test"
    / "metrics.json"
)

print("Original file:", original_file)
print("Improved file:", improved_file)

assert original_file.exists(), "Original Official Test metrics not found."
assert improved_file.exists(), "Improved Official Test metrics not found."

with open(original_file) as f:
    original = json.load(f)

with open(improved_file) as f:
    improved = json.load(f)

rows = []

for model_name, results in [
    ("Original Higher WD", original),
    ("Improved SupCon Disease", improved)
]:
    for task in ["crop", "disease", "composition"]:
        for k in [1, 5]:

            key = f"{task}_top{k}"

            seen = results["seen"][key]
            unseen = results["unseen"][key]

            rows.append({
                "Model": model_name,
                "Task": task.title(),
                "Top-K": k,
                "Seen (%)": seen * 100,
                "Unseen (%)": unseen * 100,
                "HM (%)": hm(seen, unseen) * 100
            })

comparison = pd.DataFrame(rows)

print("\nORIGINAL HIGHER WEIGHT DECAY — OFFICIAL TEST")
display(
    comparison[
        comparison["Model"] == "Original Higher WD"
    ].drop(columns="Model").round(2)
)

print("\nIMPROVED SUPCON DISEASE — OFFICIAL TEST")
display(
    comparison[
        comparison["Model"] == "Improved SupCon Disease"
    ].drop(columns="Model").round(2)
)

print("\nSIDE-BY-SIDE TOP-1 COMPARISON")
display(
    comparison[
        comparison["Top-K"] == 1
    ].round(2)
)

# Save comparison
comparison.to_csv(
    RESULTS_ROOT / "original_vs_supcon_official_comparison.csv",
    index=False
)

In [ ]:

# ============================================================
# STAGE 1 DEVELOPMENT VALIDATION COMPARISON
# Original Higher WD vs Improved Projection + SupCon
# SAME Stage 1 Crop CNN for BOTH models
# NO retraining / NO inference / NO official test
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

# Stage 1 best prediction files
CROP_FILE = (
    run_dir("higher_decay", "crop")
    / "best_validation.npz"
)

ORIGINAL_DISEASE_FILE = (
    run_dir("higher_decay", "disease")
    / "best_validation.npz"
)

SUPCON_DISEASE_FILE = (
    NEW_EXP_ROOT / "projection_supcon"
    / "predictions_epoch_15.npz"
)

for path in [
    CROP_FILE,
    ORIGINAL_DISEASE_FILE,
    SUPCON_DISEASE_FILE
]:
    assert path.exists(), f"Missing file: {path}"

# Confirm SupCon selected best epoch = 15
supcon_history = pd.read_csv(
    NEW_EXP_ROOT / "projection_supcon" / "history.csv"
)

best_supcon = supcon_history.sort_values(
    ["disease_hm", "epoch"],
    ascending=[False, True]
).iloc[0]

assert int(best_supcon["epoch"]) == 15

# Read saved predictions
def load_predictions(path):
    with np.load(path) as data:
        return {
            "seen": data["seen"].copy(),
            "unseen": data["unseen"].copy()
        }

crop_pred = load_predictions(CROP_FILE)
original_pred = load_predictions(ORIGINAL_DISEASE_FILE)
supcon_pred = load_predictions(SUPCON_DISEASE_FILE)

def harmonic_mean(a, b):
    return 2 * a * b / (a + b) if a + b else 0.0

def calculate_validation(crop_logs, disease_logs):

    result = {}

    for split, df in [
        ("seen", val_seen_df),
        ("unseen", val_unseen_df)
    ]:

        c = crop_logs[split]
        d = disease_logs[split]

        assert c.shape == (len(df), len(crop_ids))
        assert d.shape == (len(df), len(disease_ids))

        true_crop = df["crop_id"].map(crop_index).to_numpy()
        true_disease = df["disease_id"].map(disease_index).to_numpy()

        assert not pd.isna(true_crop).any()
        assert not pd.isna(true_disease).any()

        crop_correct = c.argmax(axis=1) == true_crop
        disease_correct = d.argmax(axis=1) == true_disease

        result[split] = {
            "Crop": crop_correct.mean(),
            "Disease": disease_correct.mean(),
            "Composition": (
                crop_correct & disease_correct
            ).mean()
        }

    return result

experiments = {
    "Original Higher WD": original_pred,
    "Improved SupCon": supcon_pred
}

rows = []

for model_name, disease_predictions in experiments.items():

    result = calculate_validation(
        crop_pred,
        disease_predictions
    )

    for task in ["Crop", "Disease", "Composition"]:

        seen = result["seen"][task]
        unseen = result["unseen"][task]

        rows.append({
            "Model": model_name,
            "Task": task,
            "Seen Accuracy (%)": seen * 100,
            "Unseen Accuracy (%)": unseen * 100,
            "HM (%)": harmonic_mean(seen, unseen) * 100
        })

comparison = pd.DataFrame(rows)

print("=" * 65)
print("STAGE 1 — DEVELOPMENT VALIDATION COMPARISON")
print("BEFORE FULL-DATA FINE-TUNING")
print("=" * 65)

display(comparison.round(2))

# Side-by-side comparison
side_by_side = comparison.pivot(
    index="Task",
    columns="Model",
    values=[
        "Seen Accuracy (%)",
        "Unseen Accuracy (%)",
        "HM (%)"
    ]
)

print("\nSIDE-BY-SIDE COMPARISON")
display(side_by_side.round(2))

# Verify identical Crop accuracy
crop_rows = comparison[comparison["Task"] == "Crop"]

assert crop_rows["Seen Accuracy (%)"].nunique() == 1
assert crop_rows["Unseen Accuracy (%)"].nunique() == 1

print("\nVERIFICATION")
print("Same Stage 1 Crop CNN: YES")
print("Original Disease: Higher WD Epoch 21")
print("Improved Disease: SupCon Epoch 15")
print("Crop Accuracy Identical: YES")
print("Official Test Used: NO")

# Save comparison
output_file = (
    RESULTS_ROOT
    / "stage1_original_vs_supcon_validation.csv"
)

comparison.to_csv(output_file, index=False)

print("\nSaved to:", output_file)


In [ ]:

# ============================================================
# STAGE 1 OFFICIAL TEST — ORIGINAL vs SUPCON
# BEFORE FULL-DATA FINE-TUNING
# DIAGNOSTIC EVALUATION ONLY
# ============================================================

import json
import gc
import hashlib
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torchvision import models
from IPython.display import display

# ------------------------------------------------------------
# 1. Output directory and source checkpoints
# ------------------------------------------------------------

NEW_EXP_ROOT = (
    RESULTS_ROOT / "new_disease_experiments_5epoch"
)

STAGE1_TEST_ROOT = (
    RESULTS_ROOT / "stage1_official_diagnostic"
)

STAGE1_TEST_ROOT.mkdir(
    parents=True, exist_ok=True
)

crop_path = (
    run_dir("higher_decay", "crop") / "best.pt"
)

original_path = (
    run_dir("higher_decay", "disease") / "best.pt"
)

supcon_path = (
    NEW_EXP_ROOT
    / "projection_supcon"
    / "epoch_15.pt"
)

paths = [crop_path, original_path, supcon_path]

for path in paths:
    assert path.exists(), f"Checkpoint missing: {path}"

# ------------------------------------------------------------
# 2. Load and verify Stage 1 checkpoints
# ------------------------------------------------------------

crop_ckpt, original_ckpt, supcon_ckpt = [
    torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )
    for path in paths
]

assert crop_ckpt["task"] == "crop"
assert original_ckpt["task"] == "disease"

assert crop_ckpt["epoch"] == 27
assert original_ckpt["epoch"] == 21

assert supcon_ckpt["epoch"] == 15
assert supcon_ckpt["arm"] == "projection_supcon"
assert supcon_ckpt["head"] == "wider"

for ckpt in [crop_ckpt, original_ckpt, supcon_ckpt]:
    assert list(ckpt["crop_ids"]) == list(crop_ids)
    assert list(ckpt["disease_ids"]) == list(disease_ids)

# ------------------------------------------------------------
# 3. Reconstruct Original Stage 1 architecture
# ------------------------------------------------------------

def load_original_stage1(ckpt, task):

    assert ckpt["task"] == task

    model = models.resnet50(weights=None)

    n_classes = (
        len(crop_ids)
        if task == "crop"
        else len(disease_ids)
    )

    # Correct Stage 1 checkpoint configuration
    dropout = ckpt["config"]["dropout"]

    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(
            model.fc.in_features,
            n_classes
        )
    )

    model.load_state_dict(
        ckpt["state_dict"],
        strict=True
    )

    return model.to(DEVICE).eval()


# SAME Crop CNN for both experiments
crop_model = load_original_stage1(
    crop_ckpt, "crop"
)

original_disease = load_original_stage1(
    original_ckpt, "disease"
)

# ------------------------------------------------------------
# 4. Load SupCon Stage 1 architecture
# ------------------------------------------------------------

supcon_disease = ImprovedDiseaseCNN().to(DEVICE)

supcon_disease.load_state_dict(
    supcon_ckpt["state_dict"],
    strict=True
)

supcon_disease.eval()

print("=" * 60)
print("STAGE 1 MODEL LOADING SUCCESSFUL")
print("=" * 60)

print("Crop CNN: Original Higher WD Epoch 27")
print("Original Disease CNN: Epoch 21")
print("SupCon Disease CNN: Epoch 15")
print("Full-Data Fine-Tuning: NOT USED")

# ------------------------------------------------------------
# 5. Verify Official Test setup
# ------------------------------------------------------------

assert len(official_dfs["seen"]) == 2000
assert len(official_dfs["unseen"]) == 1448

assert crop_model.training is False
assert original_disease.training is False
assert supcon_disease.training is False

# Ensure same source Crop checkpoint for both
print("\nShared Crop Checkpoint:", crop_path)
print("AMP:", AMP)
print("Official Batch Size:", OFFICIAL_BATCH_SIZE)

# ------------------------------------------------------------
# 6. Record evaluation protocol
# ------------------------------------------------------------

def checkpoint_hash(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


protocol = {
    "purpose": "post_hoc_stage1_diagnostic",
    "crop_checkpoint": str(crop_path),
    "original_disease_checkpoint": str(original_path),
    "supcon_disease_checkpoint": str(supcon_path),

    "checkpoint_hashes": {
        "crop": checkpoint_hash(crop_path),
        "original_disease": checkpoint_hash(original_path),
        "supcon_disease": checkpoint_hash(supcon_path)
    },

    "amp": bool(AMP),
    "preprocessing": "Resize256_CenterCrop224_ImageNetNormalize",
    "checkpoint_selection": "Stage1_Development_Disease_HM",
    "official_test_used_for_selection": False
}

protocol_file = STAGE1_TEST_ROOT / "protocol.json"

if protocol_file.exists():

    saved_protocol = json.loads(
        protocol_file.read_text()
    )

    assert saved_protocol == protocol, (
        "Existing protocol differs. "
        "Do not overwrite previous evaluation."
    )

else:

    protocol_file.write_text(
        json.dumps(protocol, indent=2)
    )

# ------------------------------------------------------------
# 7. Run Official Test — SAME evaluation function
# ------------------------------------------------------------

stage1_results = {}

try:

    for name, disease_model in [
        ("Original_Higher_WD", original_disease),
        ("Improved_SupCon", supcon_disease)
    ]:

        stage1_results[name] = {}

        print("\n" + "=" * 60)
        print("EXPERIMENT:", name)
        print("=" * 60)

        for split in ["seen", "unseen"]:

            metrics, predictions = evaluate_official_split(
                split,
                crop_model,
                disease_model
            )

            stage1_results[name][split] = metrics

            output_path = (
                STAGE1_TEST_ROOT
                / f"{name}_{split}_predictions.csv"
            )

            predictions.to_csv(
                output_path,
                index=False
            )

            print(f"\n{split.upper()} RESULTS")

            print(
                "Crop Top-1:",
                f"{metrics['crop_top1']:.2%}"
            )

            print(
                "Disease Top-1:",
                f"{metrics['disease_top1']:.2%}"
            )

            print(
                "Composition Top-1:",
                f"{metrics['composition_top1']:.2%}"
            )

finally:

    del crop_model
    del original_disease
    del supcon_disease

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 8. Save final metrics
# ------------------------------------------------------------

with open(
    STAGE1_TEST_ROOT / "metrics.json",
    "w"
) as f:

    json.dump(
        stage1_results,
        f,
        indent=2
    )

print("\n" + "=" * 60)
print("STAGE 1 OFFICIAL DIAGNOSTIC COMPLETED")
print("=" * 60)

print("Results saved to:", STAGE1_TEST_ROOT)


In [ ]:

# ============================================================
# STAGE 1 OFFICIAL TEST — COMPARISON TABLE
# ============================================================

with open(STAGE1_TEST_ROOT / "metrics.json") as f:
    results = json.load(f)

def stage1_hm(a, b):
    return 2 * a * b / (a + b) if a + b else 0

rows = []

for model_name, data in results.items():
    for task in ["crop", "disease", "composition"]:
        for k in [1, 5]:

            metric = f"{task}_top{k}"
            seen = data["seen"][metric]
            unseen = data["unseen"][metric]

            rows.append({
                "Model": model_name,
                "Task": task.title(),
                "Top-K": k,
                "Seen (%)": seen * 100,
                "Unseen (%)": unseen * 100,
                "HM (%)": stage1_hm(seen, unseen) * 100
            })

df = pd.DataFrame(rows)

print("STAGE 1 — OFFICIAL TEST (BEFORE FULL-DATA FINE-TUNING)")
display(df.round(2))

print("\nTOP-1 COMPARISON")
display(df[df["Top-K"] == 1].round(2))

# Verify Crop predictions exactly match between experiments
for split in ["seen", "unseen"]:

    a = pd.read_csv(
        STAGE1_TEST_ROOT /
        f"Original_Higher_WD_{split}_predictions.csv"
    )

    b = pd.read_csv(
        STAGE1_TEST_ROOT /
        f"Improved_SupCon_{split}_predictions.csv"
    )

    check = a[["filename", "pred_crop_id"]].merge(
        b[["filename", "pred_crop_id"]],
        on="filename",
        suffixes=("_original", "_supcon"),
        validate="one_to_one"
    )

    assert len(check) == len(a) == len(b)
    assert (
        check["pred_crop_id_original"]
        == check["pred_crop_id_supcon"]
    ).all()

    print(f"{split.upper()}: Crop predictions identical — YES")

df.to_csv(
    STAGE1_TEST_ROOT / "stage1_official_comparison.csv",
    index=False
)

print("\nComparison saved successfully.")


In [ ]:
import pandas as pd

for task in ["crop", "disease"]:
    path = STAGE2_ROOT / task / "history.csv"

    print(f"\n{task.upper()} — STAGE 2 TRAINING")
    if path.exists():
        display(pd.read_csv(path).round(4))
    else:
        print("History not found:", path)

In [ ]:
from pathlib import Path

stage2_dir = (
    RESULTS_ROOT / "stage2_from_higher_decay_best_full8000"
)

for task in ["crop", "disease"]:
    folder = stage2_dir / task

    print(f"\n{task.upper()} Checkpoints")

    for epoch in [3, 5, 10]:
        path = folder / f"epoch_{epoch:02d}.pt"
        print(f"Epoch {epoch}:", path.exists())

In [ ]:

# ============================================================
# STAGE 2 EARLY CHECKPOINT DIAGNOSTIC
# Compare Epoch 3, 5 and 10 — Original Higher WD
# NO TRAINING / POST-HOC ANALYSIS ONLY
# ============================================================

import torch
import torch.nn as nn
import pandas as pd
import json
import gc

from torchvision import models
from pathlib import Path
from IPython.display import display

STAGE2_DIR = (
    RESULTS_ROOT / "stage2_from_higher_decay_best_full8000"
)

OUTPUT_DIR = (
    RESULTS_ROOT / "stage2_epoch_3_5_10_diagnostic"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS_TO_TEST = [3, 5, 10]

# Reuse the original official evaluation function
assert "evaluate_official_split" in globals()
assert "official_dfs" in globals()

def load_stage2_checkpoint(task, epoch):

    path = (
        STAGE2_DIR / task / f"epoch_{epoch:02d}.pt"
    )

    assert path.exists(), f"Missing: {path}"

    ckpt = torch.load(
        path, map_location="cpu", weights_only=False
    )

    assert ckpt["stage"] == 2
    assert ckpt["task"] == task
    assert int(ckpt["epoch"]) == epoch
    assert list(ckpt["crop_ids"]) == list(crop_ids)
    assert list(ckpt["disease_ids"]) == list(disease_ids)

    model = models.resnet50(weights=None)

    n_classes = (
        len(crop_ids)
        if task == "crop"
        else len(disease_ids)
    )

    model.fc = nn.Sequential(
        nn.Dropout(ckpt["config"]["dropout"]),
        nn.Linear(model.fc.in_features, n_classes)
    )

    model.load_state_dict(
        ckpt["state_dict"], strict=True
    )

    return model.to(DEVICE).eval()


def harmonic_mean(a, b):
    return 2 * a * b / (a + b) if a + b else 0.0


rows = []
all_results = {}

for epoch in EPOCHS_TO_TEST:

    print("\n" + "=" * 60)
    print(f"STAGE 2 — EPOCH {epoch}")
    print("=" * 60)

    crop_model = load_stage2_checkpoint("crop", epoch)
    disease_model = load_stage2_checkpoint("disease", epoch)

    all_results[str(epoch)] = {}

    try:
        for split in ["seen", "unseen"]:

            metrics, predictions = evaluate_official_split(
                split, crop_model, disease_model
            )

            all_results[str(epoch)][split] = metrics

            predictions.to_csv(
                OUTPUT_DIR /
                f"epoch_{epoch:02d}_{split}_predictions.csv",
                index=False
            )

    finally:
        del crop_model, disease_model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for task in ["crop", "disease", "composition"]:

        for k in [1, 5]:

            key = f"{task}_top{k}"

            seen = all_results[str(epoch)]["seen"][key]
            unseen = all_results[str(epoch)]["unseen"][key]

            rows.append({
                "Epoch": epoch,
                "Task": task.title(),
                "Top-K": k,
                "Seen (%)": seen * 100,
                "Unseen (%)": unseen * 100,
                "HM (%)": harmonic_mean(seen, unseen) * 100
            })

comparison = pd.DataFrame(rows)

print("\n" + "=" * 60)
print("STAGE 2 — EPOCH 3 vs 5 vs 10")
print("OFFICIAL TEST DIAGNOSTIC")
print("=" * 60)

display(comparison.round(2))

print("\nTOP-1 COMPARISON")
display(
    comparison[
        comparison["Top-K"] == 1
    ].round(2)
)

comparison.to_csv(
    OUTPUT_DIR / "epoch_3_5_10_comparison.csv",
    index=False
)

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\nDiagnostic results saved to:", OUTPUT_DIR)


In [ ]:

# ============================================================
# STAGE 2 — ALL 10 EPOCH OFFICIAL TEST DIAGNOSTIC
# Original Higher WD Crop + Disease
# ============================================================

import torch
import pandas as pd
import json
import gc
from pathlib import Path
from IPython.display import display

STAGE2_DIR = (
    RESULTS_ROOT / "stage2_from_higher_decay_best_full8000"
)

OUTPUT_DIR = (
    RESULTS_ROOT / "stage2_all_10_epochs_diagnostic"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert "evaluate_official_split" in globals()
assert "official_dfs" in globals()
assert "load_stage2_checkpoint" in globals()

def hm(a, b):
    return 2 * a * b / (a + b) if a + b else 0

results = {}
rows = []

for epoch in range(1, 11):

    print(f"\n{'='*55}")
    print(f"TESTING STAGE 2 EPOCH {epoch}/10")
    print(f"{'='*55}")

    crop = load_stage2_checkpoint("crop", epoch)
    disease = load_stage2_checkpoint("disease", epoch)

    results[str(epoch)] = {}

    try:
        for split in ["seen", "unseen"]:

            metrics, predictions = evaluate_official_split(
                split, crop, disease
            )

            results[str(epoch)][split] = metrics

            predictions.to_csv(
                OUTPUT_DIR /
                f"epoch_{epoch:02d}_{split}_predictions.csv",
                index=False
            )

    finally:
        del crop, disease
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for task in ["crop", "disease", "composition"]:

        s = results[str(epoch)]["seen"][f"{task}_top1"]
        u = results[str(epoch)]["unseen"][f"{task}_top1"]

        rows.append({
            "Epoch": epoch,
            "Task": task.title(),
            "Seen (%)": s * 100,
            "Unseen (%)": u * 100,
            "HM (%)": hm(s, u) * 100
        })

    # Save progress after each epoch
    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / "all_10_epochs_results.csv",
        index=False
    )

    with open(OUTPUT_DIR / "metrics.json", "w") as f:
        json.dump(results, f, indent=2)

# ============================================================
# DISPLAY RESULTS
# ============================================================

df = pd.DataFrame(rows)

print("\nALL 10 EPOCHS — TOP-1 RESULTS")
display(df.round(2))

# Composition only
composition = df[
    df["Task"] == "Composition"
].sort_values("Epoch")

print("\nCOMPOSITION COMPARISON")
display(composition.round(2))

# Best observed epoch for each task (descriptive only)
print("\nHIGHEST OBSERVED OFFICIAL HM — DIAGNOSTIC ONLY")

best_rows = []

for task in ["Crop", "Disease", "Composition"]:

    task_df = df[df["Task"] == task]

    best = task_df.sort_values(
        ["HM (%)", "Epoch"],
        ascending=[False, True]
    ).iloc[0]

    best_rows.append({
        "Task": task,
        "Highest Observed Epoch": int(best["Epoch"]),
        "Seen (%)": best["Seen (%)"],
        "Unseen (%)": best["Unseen (%)"],
        "HM (%)": best["HM (%)"]
    })

best_table = pd.DataFrame(best_rows)
display(best_table.round(2))

print("\nNOTE: These are post-hoc diagnostic rankings.")
print("Do not use official test scores for checkpoint selection.")
print("\nSaved to:", OUTPUT_DIR)


In [ ]:

# ============================================================
# IMPROVED SUPCON — ALL 10 STAGE 2 EPOCHS
# Fixed Original Crop + SupCon Disease Epoch 1–10
# POST-HOC OFFICIAL DIAGNOSTIC / NO TRAINING
# ============================================================

import torch
import torch.nn as nn
import pandas as pd
import json
import gc

from pathlib import Path
from torchvision import models
from IPython.display import display

ROOT = Path(
    "/content/drive/MyDrive/AML_B1_ResNet50_V2_Full_Tuning"
)

CROP_PATH = (
    ROOT / "stage2_from_higher_decay_best_full8000"
    / "crop" / "best_train_acc.pt"
)

SUPCON_DIR = (
    ROOT / "stage2_improved_supcon_disease" / "disease"
)

OUTPUT_DIR = (
    ROOT / "supcon_stage2_all_10_epochs_diagnostic"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CROP_PATH.exists()
assert "ImprovedDiseaseCNN" in globals()
assert "evaluate_official_split" in globals()
assert "official_dfs" in globals()

for epoch in range(1, 11):
    assert (
        SUPCON_DIR / f"epoch_{epoch:02d}.pt"
    ).exists(), f"Missing SupCon Epoch {epoch}"

# ------------------------------------------------------------
# Load fixed Original Stage 2 Crop CNN
# ------------------------------------------------------------

crop_ckpt = torch.load(
    CROP_PATH, map_location="cpu", weights_only=False
)

crop = models.resnet50(weights=None)

crop.fc = nn.Sequential(
    nn.Dropout(crop_ckpt["config"]["dropout"]),
    nn.Linear(crop.fc.in_features, len(crop_ids))
)

crop.load_state_dict(
    crop_ckpt["state_dict"], strict=True
)

crop = crop.to(DEVICE).eval()

assert crop_ckpt["task"] == "crop"
assert crop_ckpt["stage"] == 2

print("Fixed Crop Stage 2 Epoch:", crop_ckpt["epoch"])
print("AMP:", AMP)
print("Batch Size:", OFFICIAL_BATCH_SIZE)

# ------------------------------------------------------------
# Evaluation helpers
# ------------------------------------------------------------

def supcon_hm(a, b):
    return 2 * a * b / (a + b) if a + b > 0 else 0.0


def load_supcon_stage2(epoch):

    path = SUPCON_DIR / f"epoch_{epoch:02d}.pt"

    ckpt = torch.load(
        path, map_location="cpu", weights_only=False
    )

    assert ckpt["task"] == "disease"
    assert ckpt["stage"] == 2
    assert int(ckpt["epoch"]) == epoch

    assert list(ckpt["crop_ids"]) == list(crop_ids)
    assert list(ckpt["disease_ids"]) == list(disease_ids)

    model = ImprovedDiseaseCNN()

    model.load_state_dict(
        ckpt["state_dict"], strict=True
    )

    return model.to(DEVICE).eval()


# ------------------------------------------------------------
# Evaluate SupCon Disease Epoch 1–10
# ------------------------------------------------------------

results = {}
rows = []

try:
    for epoch in range(1, 11):

        print(f"\n{'=' * 55}")
        print(f"SUPCON DISEASE — STAGE 2 EPOCH {epoch}/10")
        print(f"{'=' * 55}")

        disease = load_supcon_stage2(epoch)
        results[str(epoch)] = {}

        try:
            for split in ["seen", "unseen"]:

                metrics, predictions = evaluate_official_split(
                    split, crop, disease
                )

                results[str(epoch)][split] = metrics

                predictions.to_csv(
                    OUTPUT_DIR /
                    f"epoch_{epoch:02d}_{split}_predictions.csv",
                    index=False
                )

        finally:
            del disease
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        for task in ["crop", "disease", "composition"]:

            for k in [1, 5]:

                key = f"{task}_top{k}"

                seen = results[str(epoch)]["seen"][key]
                unseen = results[str(epoch)]["unseen"][key]

                rows.append({
                    "Disease Epoch": epoch,
                    "Crop Epoch": int(crop_ckpt["epoch"]),
                    "Task": task.title(),
                    "Top-K": k,
                    "Seen (%)": seen * 100,
                    "Unseen (%)": unseen * 100,
                    "HM (%)": supcon_hm(seen, unseen) * 100
                })

        # Save progress after every epoch
        pd.DataFrame(rows).to_csv(
            OUTPUT_DIR / "all_10_epochs_results.csv",
            index=False
        )

        with open(OUTPUT_DIR / "metrics.json", "w") as f:
            json.dump(results, f, indent=2)

finally:
    del crop
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# Display all results
# ------------------------------------------------------------

comparison = pd.DataFrame(rows)

print("\n" + "=" * 60)
print("IMPROVED SUPCON — ALL 10 EPOCHS")
print("=" * 60)

display(comparison.round(2))

top1 = comparison[
    comparison["Top-K"] == 1
].copy()

print("\nTOP-1 RESULTS")
display(top1.round(2))

print("\nCOMPOSITION COMPARISON")
composition = top1[
    top1["Task"] == "Composition"
].copy()

display(composition.round(2))

# ------------------------------------------------------------
# Highest observed Official HM (diagnostic only)
# ------------------------------------------------------------

best_rows = []

for task in ["Crop", "Disease", "Composition"]:

    task_df = top1[top1["Task"] == task]

    best = task_df.sort_values(
        ["HM (%)", "Disease Epoch"],
        ascending=[False, True]
    ).iloc[0]

    best_rows.append({
        "Task": task,
        "Disease Epoch": int(best["Disease Epoch"]),
        "Seen (%)": best["Seen (%)"],
        "Unseen (%)": best["Unseen (%)"],
        "HM (%)": best["HM (%)"]
    })

print("\nHIGHEST OBSERVED OFFICIAL HM — DIAGNOSTIC ONLY")
display(pd.DataFrame(best_rows).round(2))

print("\nFixed Crop CNN for all experiments.")
print("Crop Epoch:", crop_ckpt["epoch"])
print("Results saved to:", OUTPUT_DIR)
print("No model weights were updated.")


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

root = Path("/content/drive/MyDrive/AML_B1_ResNet50_V2_Full_Tuning")

paths = {
    "Original Crop": root / "stage2_from_higher_decay_best_full8000/crop/history.csv",
    "Original Disease": root / "stage2_from_higher_decay_best_full8000/disease/history.csv",
    "Improved SupCon Disease": root / "stage2_improved_supcon_disease/disease/history.csv"
}

histories = {}

for name, path in paths.items():
    if not path.exists():
        print("Missing:", path)
        continue

    df = pd.read_csv(path)
    histories[name] = df

    print(f"\n{name}")
    print(df.to_string(index=False))

# Plot automatically using history column names
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, df in histories.items():
    cols = list(df.columns)

    epoch_col = next(
        (c for c in cols if c.lower() == "epoch"), None
    )

    acc_col = next(
        (c for c in cols
         if "acc" in c.lower()
         and "val" not in c.lower()
         and "test" not in c.lower()),
        None
    )

    loss_col = next(
        (c for c in cols
         if "loss" in c.lower()
         and "val" not in c.lower()
         and "test" not in c.lower()),
        None
    )

    epochs = (
        df[epoch_col] if epoch_col
        else range(1, len(df) + 1)
    )

    if acc_col:
        acc = pd.to_numeric(df[acc_col])
        if acc.dropna().max() <= 1.0:
            acc = acc * 100

        axes[0].plot(
            epochs, acc, marker="o", label=name
        )

    if loss_col:
        axes[1].plot(
            epochs,
            pd.to_numeric(df[loss_col]),
            marker="o",
            label=name
        )

axes[0].set_title("Stage 2 Training Accuracy")
axes[0].set_ylabel("Accuracy (%)")

axes[1].set_title("Stage 2 Training Loss")
axes[1].set_ylabel("Loss")

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

for task in ["crop", "disease"]:

    path = run_dir("higher_decay", task) / "history.csv"
    df = pd.read_csv(path)

    print(f"\n{'='*60}")
    print(f"STAGE 1 — {task.upper()}")
    print(f"{'='*60}")

    print("\nAvailable columns:")
    print(df.columns.tolist())

    print("\nLast 10 epochs:")
    display(df.tail(10))

    epoch_col = next(
        (c for c in df.columns if c.lower() == "epoch"),
        None
    )

    epochs = df[epoch_col] if epoch_col else range(1, len(df)+1)

    train_cols = [
        c for c in df.columns
        if "train" in c.lower()
        and ("loss" in c.lower() or "acc" in c.lower())
        and "best" not in c.lower()
    ]

    val_cols = [
        c for c in df.columns
        if any(s in c.lower() for s in
               ["val", "seen", "unseen", "harmonic", "hm"])
        and "best" not in c.lower()
        and pd.api.types.is_numeric_dtype(df[c])
    ]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for c in train_cols:
        if "loss" in c.lower():
            axes[0].plot(
                epochs, df[c],
                marker=".", label=c
            )

    for c in val_cols:
        axes[1].plot(
            epochs, df[c],
            marker=".", label=c
        )

    axes[0].set_title(f"{task.upper()} — Training Loss")
    axes[1].set_title(f"{task.upper()} — Validation Metrics")

    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        if ax.lines:
            ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

for task in ["crop", "disease"]:
    folder = run_dir("higher_decay", task)
    df = pd.read_csv(folder / "history.csv")

    print(f"\n{'='*60}")
    print(f"{task.upper()} — EARLY EPOCH ANALYSIS")
    print(f"{'='*60}")

    display(
        df.loc[df["epoch"].between(1, 15), [
            "epoch",
            "train_loss",
            "seen_loss",
            "unseen_loss",
            "seen_acc",
            "unseen_acc",
            "individual_h"
        ]].round(4)
    )

    print("\nAvailable checkpoint files:")
    for p in sorted(folder.glob("*.pt")):
        print(p.name)